In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_solutions.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_evaluation_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/sample_submission.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_training_challenges.json
/kaggle/input/datasets/pharaohtut/arc-arg3/arc-agi_test_challenges.json
/kaggle/input/datasets/pharaohtut/test-challenges/arc-agi_test_challenges.json
/kaggle/input/datasets/pharaohtut/arc-agi2-analytics-submission-20260414-145000z-zip/dataset_solver_behavior.json
/kaggle/input/datasets/pharaohtut/arc-agi2-analytics-submission-20260414-145000z-zip/DATASET_README.txt
/kaggle/input/datasets/pharaohtut/arc-agi2-analytics-submission-20260414-145000z-zip/dataset_background_sensitivity.json
/kaggle/input/datasets/pharaohtut/arc-agi2-analytics-submission-20260414-145000z-zip/dataset_manifest.json
/kaggle/input/datasets/pharaoht

In [2]:
# CELL 01
# Global deterministic guarantees and environment invariants

import sys
import math
from typing import Any, Dict, Tuple, Optional, List

# Absolute determinism flags
DETERMINISTIC_MODE = True
OFFLINE_ONLY = True

# ARC-AGI-2 invariant: exactly two attempts
MAX_ATTEMPTS = 2

# Hard refusal token (canonical)
REFUSAL_GRID = None  # will later be replaced by a canonical no-op grid

assert DETERMINISTIC_MODE is True
assert OFFLINE_ONLY is True

In [3]:
# CELL 02
# Canonical registry, execution receipts, and state containers

class ExecutionReceipt:
    def __init__(self, logic_id: str, status: str, reason: str):
        self.logic_id = logic_id
        self.status = status      # "executed", "refused", "terminated"
        self.reason = reason      # deterministic explanation

    def as_dict(self) -> Dict[str, str]:
        return {
            "logic_id": self.logic_id,
            "status": self.status,
            "reason": self.reason,
        }


class Registry:
    """
    Immutable registry snapshot.
    All writes must produce a new registry.
    """
    def __init__(self, data: Dict[str, Any]):
        self._data = dict(data)

    def read(self, key: str) -> Any:
        return self._data.get(key)

    def write(self, key: str, value: Any) -> "Registry":
        new_data = dict(self._data)
        new_data[key] = value
        return Registry(new_data)

    def snapshot(self) -> Dict[str, Any]:
        return dict(self._data)

In [4]:
# CELL 03
# AO3 Failure Vector and deterministic termination logic

AO3_COMPLEXITY_TAX = 1e-4
AO3_THRESHOLD = 1.0  # canonical safety bound

class AO3Result:
    def __init__(self, allowed: bool, J: float, reason: str):
        self.allowed = allowed
        self.J = J
        self.reason = reason


def compute_ao3(DS: float, DK: float) -> AO3Result:
    """
    Deterministic, pre-execution AO3 evaluation.
    """
    J = DS + (DK * AO3_COMPLEXITY_TAX)

    if J > AO3_THRESHOLD:
        return AO3Result(
            allowed=False,
            J=J,
            reason="AO3 threshold exceeded"
        )

    return AO3Result(
        allowed=True,
        J=J,
        reason="AO3 within bounds"
    )

In [5]:
# CELL 03a
# V16 capability levels and expansion budgets

class CapabilityLevel:
    """
    Discrete, ordered capability levels.
    """
    BASELINE = 0        # V15 behavior (default)
    MICRO_COMPOSE = 1   # Allow 2-step actuation
    MULTI_ACT = 2       # Allow >2 actuation steps (future)
    EXPRESSIVE = 3      # Reserved for V17+

# Hard caps per attempt (deterministic)
CAPABILITY_BUDGET = {
    CapabilityLevel.BASELINE: 0,
    CapabilityLevel.MICRO_COMPOSE: 1,
    CapabilityLevel.MULTI_ACT: 3,
}

# V16 starts here
V16_MAX_CAPABILITY = CapabilityLevel.MICRO_COMPOSE

In [6]:
# CELL 03b
# V16 expansion state (immutable)

class ExpansionState:
    """
    Tracks per-attempt expansion usage.
    """
    def __init__(self, capability: int, used_budget: int):
        self.capability = capability
        self.used_budget = used_budget

    def can_expand(self) -> bool:
        return self.used_budget < CAPABILITY_BUDGET[self.capability]

    def consume(self) -> "ExpansionState":
        return ExpansionState(
            capability=self.capability,
            used_budget=self.used_budget + 1
        )

    def snapshot(self) -> dict:
        return {
            "capability": self.capability,
            "used_budget": self.used_budget,
            "remaining_budget": (
                CAPABILITY_BUDGET[self.capability] - self.used_budget
            ),
        }

In [7]:
# CELL 03c
# V16.2 empirical tuning parameters (static, observable)

class EmpiricalTuningConfig:
    """
    Static configuration for empirical observation.
    Changing these values constitutes a new experiment,
    not a runtime adaptation.
    """

    # AO3 thresholds (observable, not adaptive)
    AO3_DETECTION_THRESHOLD = AO3_THRESHOLD
    AO3_ACTUATION_THRESHOLD = AO3_THRESHOLD

    # Capability budgets (mirrors CAPABILITY_BUDGET)
    CAPABILITY_BUDGETS = dict(CAPABILITY_BUDGET)

    # Enable empirical receipts
    ENABLE_EMPIRICAL_RECEIPTS = True

In [8]:
# CELL 03c.bootstrap
# E-Section Infrastructure Bootstrap (Analysis-Only, Non-Solver)

import json
import numpy as np
from collections import defaultdict, deque
from types import SimpleNamespace

print("[BOOTSTRAP] Initializing E-section infrastructure")

# ---------------------------------------------------------------------
# ARC DATASET LOADER (MINIMAL, OFFLINE)
# ---------------------------------------------------------------------
class _ARCContainer:
    pass

ARC = _ARCContainer()

def _load_json(path):
    with open(path, "r") as f:
        return json.load(f)

ARC.train_challenges = _load_json("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json")
ARC.eval_challenges  = _load_json("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
ARC.test_challenges  = _load_json("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")

# ---------------------------------------------------------------------
# TASK PAIR PARSER
# ---------------------------------------------------------------------
def parse_task_pairs(task):
    train_pairs = []
    test_inputs = []

    for ex in task.get("train", []):
        train_pairs.append((
            np.array(ex["input"], dtype=int),
            np.array(ex["output"], dtype=int),
        ))

    for ex in task.get("test", []):
        test_inputs.append(np.array(ex["input"], dtype=int))

    return train_pairs, test_inputs

# ---------------------------------------------------------------------
# STRUCTURAL DIAGNOSTICS
# ---------------------------------------------------------------------
COMPONENT_COUNT_THRESHOLD = 4

def _count_components(grid):
    bg = np.bincount(grid.flatten()).argmax()
    h, w = grid.shape
    visited = np.zeros_like(grid, dtype=bool)
    count = 0

    for i in range(h):
        for j in range(w):
            if visited[i, j] or grid[i, j] == bg:
                continue
            count += 1
            q = deque([(i, j)])
            visited[i, j] = True
            while q:
                x, y = q.popleft()
                for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nx, ny = x+dx, y+dy
                    if 0 <= nx < h and 0 <= ny < w:
                        if not visited[nx, ny] and grid[nx, ny] != bg:
                            visited[nx, ny] = True
                            q.append((nx, ny))
    return count

def diagnose_task_structure(grid):
    bg = np.bincount(grid.flatten()).argmax()
    color_density = len(set(grid.flatten()) - {bg}) / grid.size
    return {
        "component_count": _count_components(grid),
        "color_density": color_density,
    }

# ---------------------------------------------------------------------
# REGISTRY (ANALYSIS-ONLY VIEW)
# ---------------------------------------------------------------------
REGISTRY = {
    "GEOMETRIC": SimpleNamespace(family="GEOMETRIC"),
    "REPETITION": SimpleNamespace(family="REPETITION"),
    "SYMMETRY": SimpleNamespace(family="SYMMETRY"),
    "OBJECT": SimpleNamespace(family="OBJECT"),
    "MORPHOLOGY": SimpleNamespace(family="MORPHOLOGY"),
    "COLOR": SimpleNamespace(family="COLOR"),
}

# ---------------------------------------------------------------------
# PER-TASK DIAGNOSTICS (PLACEHOLDER, SAFE)
# ---------------------------------------------------------------------
per_task_diagnostics = {}

for split in (ARC.train_challenges, ARC.eval_challenges):
    for task_id, task in split.items():
        train_pairs, _ = parse_task_pairs(task)
        if not train_pairs:
            continue
        per_task_diagnostics[task_id] = {
            "test_cases": [{
                "same_shape": True,
                "uniform_1": False,
                "uniform_2": False,
                "disagreement_ratio": 0.0,
                "entropy_delta": 0.0,
            }]
        }

# ---------------------------------------------------------------------
# TASK CONFIDENCE (NEUTRAL BASELINE)
# ---------------------------------------------------------------------
task_confidence = {tid: 0.5 for tid in per_task_diagnostics.keys()}

print("[BOOTSTRAP] ✅ E-section infrastructure ready")

[BOOTSTRAP] Initializing E-section infrastructure
[BOOTSTRAP] ✅ E-section infrastructure ready


In [9]:
# CELL 03d
# Structural Grid Features — Pure Analysis Layer (Deterministic, Read-Only)

from collections import deque, Counter
from typing import Dict, Any, List, Tuple

# -----------------------------------------------------------------------------
# Basic grid helpers
# -----------------------------------------------------------------------------

def grid_shape(grid: List[List[int]]) -> Tuple[int, int]:
    return len(grid), len(grid[0]) if grid else (0, 0)


def infer_background_color(grid: List[List[int]]) -> int:
    """
    Deterministically infer background as the most frequent color.
    """
    flat = [c for row in grid for c in row]
    if not flat:
        return 0
    return Counter(flat).most_common(1)[0][0]


def grid_colors(grid: List[List[int]]) -> List[int]:
    """
    Sorted list of colors present in the grid.
    """
    return sorted(set(c for row in grid for c in row))


# -----------------------------------------------------------------------------
# Connected component analysis (4-connected, background-excluding)
# -----------------------------------------------------------------------------

def connected_components(
    grid: List[List[int]],
    background: int
) -> List[Dict[str, Any]]:
    """
    Extract 4-connected foreground components.
    Returns a list of component dicts with stable ordering.
    """
    h, w = grid_shape(grid)
    visited = [[False] * w for _ in range(h)]
    components = []

    for r in range(h):
        for c in range(w):
            if visited[r][c]:
                continue
            if grid[r][c] == background:
                continue

            color = grid[r][c]
            q = deque([(r, c)])
            visited[r][c] = True
            cells = []

            while q:
                cr, cc = q.popleft()
                cells.append((cr, cc))
                for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w:
                        if not visited[nr][nc] and grid[nr][nc] == color:
                            visited[nr][nc] = True
                            q.append((nr, nc))

            rows = [p[0] for p in cells]
            cols = [p[1] for p in cells]
            component = {
                "color": color,
                "cells": tuple(cells),          # immutable
                "area": len(cells),
                "bbox": (
                    min(rows), max(rows),
                    min(cols), max(cols),
                ),
            }
            components.append(component)

    # Stable ordering: top-left scan order, then color
    components.sort(key=lambda c: (c["bbox"][0], c["bbox"][2], c["color"]))
    return components


# -----------------------------------------------------------------------------
# Symmetry detection (global, exact)
# -----------------------------------------------------------------------------

def is_vertical_symmetric(grid: List[List[int]]) -> bool:
    h, w = grid_shape(grid)
    for r in range(h):
        for c in range(w):
            if grid[r][c] != grid[r][w - 1 - c]:
                return False
    return True


def is_horizontal_symmetric(grid: List[List[int]]) -> bool:
    h, w = grid_shape(grid)
    for r in range(h):
        for c in range(w):
            if grid[r][c] != grid[h - 1 - r][c]:
                return False
    return True


def is_rot180_symmetric(grid: List[List[int]]) -> bool:
    h, w = grid_shape(grid)
    for r in range(h):
        for c in range(w):
            if grid[r][c] != grid[h - 1 - r][w - 1 - c]:
                return False
    return True


# -----------------------------------------------------------------------------
# Master feature extractor (single entry point)
# -----------------------------------------------------------------------------

def grid_features(grid: List[List[int]]) -> Dict[str, Any]:
    """
    Canonical structural feature bundle for a grid.
    Pure, deterministic, side-effect free.
    """
    h, w = grid_shape(grid)
    bg = infer_background_color(grid)
    comps = connected_components(grid, background=bg)

    return {
        "shape": (h, w),
        "background": bg,
        "colors": grid_colors(grid),
        "num_colors": len(grid_colors(grid)),
        "num_components": len(comps),
        "components": comps,
        "symmetry": {
            "vertical": is_vertical_symmetric(grid),
            "horizontal": is_horizontal_symmetric(grid),
            "rot180": is_rot180_symmetric(grid),
        },
    }

In [10]:
# CELL 03e
# Hypothesis Scoring & Ranking — Structural, Deterministic, Non-Actuating

from typing import List

# -----------------------------------------------------------------------------
# Hypothesis scoring (lower is better)
# -----------------------------------------------------------------------------

def score_hypothesis(hypothesis) -> float:
    """
    Assigns a structural cost to a solver hypothesis.
    Lower score = structurally simpler / more plausible.
    This function MUST be deterministic and side-effect free.
    """

    grid = hypothesis.grid
    feats = grid_features(grid)

    score = 0.0

    # ------------------------------------------------------------------
    # 1. Component complexity
    # ------------------------------------------------------------------
    # Prefer fewer components
    score += feats["num_components"] * 5.0

    # ------------------------------------------------------------------
    # 2. Spatial compactness
    # ------------------------------------------------------------------
    # Penalize large bounding boxes
    for comp in feats["components"]:
        r0, r1, c0, c1 = comp["bbox"]
        height = (r1 - r0 + 1)
        width = (c1 - c0 + 1)
        score += (height * width) * 0.1

    # ------------------------------------------------------------------
    # 3. Color complexity
    # ------------------------------------------------------------------
    # Penalize introduction of many colors
    score += feats["num_colors"] * 2.0

    # ------------------------------------------------------------------
    # 4. Symmetry preservation bonus
    # ------------------------------------------------------------------
    # Reward grids that maintain exact symmetry
    if feats["symmetry"]["vertical"]:
        score -= 1.0
    if feats["symmetry"]["horizontal"]:
        score -= 1.0
    if feats["symmetry"]["rot180"]:
        score -= 0.5

    # ------------------------------------------------------------------
    # 5. Shape sanity
    # ------------------------------------------------------------------
    h, w = feats["shape"]
    if h == 0 or w == 0:
        # Degenerate grids are extremely bad
        score += 1000.0

    return round(score, 6)


# -----------------------------------------------------------------------------
# Hypothesis ranking
# -----------------------------------------------------------------------------

def rank_hypotheses(hypotheses: List) -> List:
    """
    Returns hypotheses sorted by ascending structural score.
    Stable ordering for equal scores is preserved.
    """
    return sorted(
        hypotheses,
        key=lambda h: score_hypothesis(h)
    )

In [11]:
# CELL 03f
# Hypothesis Expansion & Pruning — Bounded Structural Search Control

from typing import List, Callable

# -----------------------------------------------------------------------------
# Hypothesis expansion
# -----------------------------------------------------------------------------

def expand_hypothesis(
    hypothesis,
    operators: List[Callable],
) -> List:
    """
    Applies each operator once to the hypothesis grid.
    Returns new SolverHypothesis instances.
    """
    new_hypotheses = []

    for op in operators:
        try:
            new_grid = op(hypothesis.grid)
        except Exception:
            # Operator failed deterministically; skip
            continue

        if new_grid is None:
            continue

        new_hypotheses.append(
            SolverHypothesis(
                grid=new_grid,
                history=hypothesis.history + (op.__name__,)
            )
        )

    return new_hypotheses


# -----------------------------------------------------------------------------
# Expand + prune (single step)
# -----------------------------------------------------------------------------

def expand_and_prune(
    hypotheses: List,
    operators: List[Callable],
    max_hypotheses: int = 16,
) -> List:
    """
    Performs ONE bounded expand → rank → prune step.
    Deterministic. No recursion. No side effects.
    """

    expanded: List = []

    for hyp in hypotheses:
        expanded.extend(expand_hypothesis(hyp, operators))

    if not expanded:
        return hypotheses  # no expansion possible

    # Rank structurally
    ranked = rank_hypotheses(expanded)

    # Prune deterministically
    return ranked[:max_hypotheses]

In [12]:
# CELL 03g
# Bounded Structural Search Loop — Solver Intelligence Activation

from typing import List

# -----------------------------------------------------------------------------
# Operator set (solver-internal only)
# -----------------------------------------------------------------------------

def _solver_operators():
    """
    Returns the ordered set of pure, deterministic operators
    available to the symbolic solver.

    NOTE:
    - These are NOT governed transforms
    - They operate only on grids
    - They do not write to registries
    """
    ops = []

    # Use operators that already exist in the solver cell
    if "rotate90" in globals():
        ops.append(rotate90)
    if "flip_h" in globals():
        ops.append(flip_h)

    return ops


# -----------------------------------------------------------------------------
# Structural search (multi-step, bounded)
# -----------------------------------------------------------------------------

def run_structural_search(
    input_grid: List[List[int]],
    max_steps: int = 2,
    max_hypotheses: int = 16,
):
    """
    Performs a bounded symbolic structural search.

    Characteristics:
    - Deterministic
    - Offline
    - No recursion
    - No learning
    - No execution coupling
    - Uses expand_and_prune + scoring

    Returns:
    - ranked SolverHypothesis list
    """

    # Initial hypothesis
    initial = SolverHypothesis(
        grid=normalize_grid(input_grid),
        history=(),
    )

    hypotheses = [initial]
    operators = _solver_operators()

    for step in range(max_steps):
        hypotheses = expand_and_prune(
            hypotheses=hypotheses,
            operators=operators,
            max_hypotheses=max_hypotheses,
        )

        if not hypotheses:
            break

    # Final ranking (stable)
    return rank_hypotheses(hypotheses)


# -----------------------------------------------------------------------------
# Solver-facing entry (does NOT affect executor)
# -----------------------------------------------------------------------------

def run_search(
    input_grid: List[List[int]],
    max_steps: int = 2,
    max_hypotheses: int = 16,
):
    """
    Canonical solver search entry point.

    This function:
    - Activates multi-step structural search
    - Returns SolverResult-compatible output
    - Does NOT decide execution or submission

    Safe for diagnostics and offline analysis.
    """

    hypotheses = run_structural_search(
        input_grid=input_grid,
        max_steps=max_steps,
        max_hypotheses=max_hypotheses,
    )

    return {
        "hypotheses": hypotheses,
        "top_candidates": hypotheses[:2],
    }

In [13]:
# CELL 40a
# Symbolic Solver Engine — Minimal, Deterministic, Traceable

from dataclasses import dataclass
from typing import List, Tuple, Dict, Any

# -----------------------------------------------------------------------------
# Core data structures
# -----------------------------------------------------------------------------

Grid = Tuple[Tuple[int, ...], ...]

@dataclass(frozen=True)
class SolverHypothesis:
    grid: Grid
    history: Tuple[str, ...]

@dataclass
class SolverResult:
    candidates: List[Grid]
    trace: Dict[str, Any]

# -----------------------------------------------------------------------------
# Grid utilities
# -----------------------------------------------------------------------------

def normalize_grid(grid: List[List[int]]) -> Grid:
    return tuple(tuple(row) for row in grid)

def rotate90(grid: Grid) -> Grid:
    return tuple(zip(*grid[::-1]))

def flip_h(grid: Grid) -> Grid:
    return tuple(tuple(reversed(r)) for r in grid)

# -----------------------------------------------------------------------------
# Solver engine
# -----------------------------------------------------------------------------

class ARCSolver:
    """
    Deterministic symbolic ARC solver.
    No intent access. No learning. No randomness.
    """

    def __init__(self, max_depth: int = 2):
        self.max_depth = max_depth

    def solve(self, input_grid: List[List[int]]) -> SolverResult:
        base = normalize_grid(input_grid)

        frontier = [SolverHypothesis(base, ())]
        seen = {base}

        trace = {
            "expanded": 0,
            "unique_states": 1,
            "histories": [],
        }

        for depth in range(self.max_depth):
            next_frontier = []
            for hyp in frontier:
                for name, op in [
                    ("rotate90", rotate90),
                    ("flip_h", flip_h),
                ]:
                    new_grid = op(hyp.grid)
                    if new_grid in seen:
                        continue
                    seen.add(new_grid)
                    trace["expanded"] += 1
                    next_frontier.append(
                        SolverHypothesis(
                            new_grid,
                            hyp.history + (name,)
                        )
                    )
            frontier = next_frontier
            if not frontier:
                break

        trace["unique_states"] = len(seen)
        trace["histories"] = [h.history for h in frontier]

        # ARC allows up to 2 outputs
        outputs = [h.grid for h in frontier[:2]]

        return SolverResult(
            candidates=outputs,
            trace=trace,
        )

# -----------------------------------------------------------------------------
# Standalone solver entry (diagnostic-friendly)
# -----------------------------------------------------------------------------

def run_search(input_grid: List[List[int]], max_depth: int = 2) -> SolverResult:
    solver = ARCSolver(max_depth=max_depth)
    return solver.solve(input_grid)

In [14]:
# CELL 03h
# Solver Trace Diagnostics — Observational, Non-Invasive

"""
This cell adds observational tracing to the symbolic solver.

Invariants:
- Does NOT change solver behavior
- Does NOT affect execution or submission
- Deterministic, offline, append-only
- Safe to leave enabled permanently
"""

from typing import Dict, Any
import copy

# -----------------------------------------------------------------------------
# Global trace store (append-only)
# -----------------------------------------------------------------------------

if "traces" not in globals():
    traces = []  # list of trace dicts


# -----------------------------------------------------------------------------
# Trace capture helper
# -----------------------------------------------------------------------------

def _record_solver_trace(
    input_grid,
    hypotheses,
    max_steps,
    max_hypotheses,
):
    """
    Records a single solver search trace.
    """

    # Defensive feature extraction
    try:
        input_feats = grid_features(input_grid)
    except Exception:
        input_feats = None

    trace = {
        "input_shape": (
            len(input_grid),
            len(input_grid[0]) if input_grid else 0,
        ),
        "input_features": input_feats,
        "search_config": {
            "max_steps": max_steps,
            "max_hypotheses": max_hypotheses,
        },
        "num_final_hypotheses": len(hypotheses),
        "hypotheses": [
            {
                "score": score_hypothesis(h),
                "history": h.history,
                "features": grid_features(h.grid),
            }
            for h in hypotheses[:5]  # cap for safety
        ],
    }

    traces.append(trace)


# -----------------------------------------------------------------------------
# Traced solver entry (wrapper)
# -----------------------------------------------------------------------------

def run_search_traced(
    input_grid,
    max_steps: int = 2,
    max_hypotheses: int = 16,
):
    """
    Wrapper around run_search that records solver traces.

    This function is SAFE for diagnostics and analysis.
    It does NOT replace run_search automatically.
    """

    result = run_search(
        input_grid=input_grid,
        max_steps=max_steps,
        max_hypotheses=max_hypotheses,
    )

    hypotheses = result.get("hypotheses", [])

    _record_solver_trace(
        input_grid=input_grid,
        hypotheses=hypotheses,
        max_steps=max_steps,
        max_hypotheses=max_hypotheses,
    )

    return result


print(
    "[INIT] ✅ Solver trace diagnostics enabled "
    "(use run_search_traced for observational analysis)"
)

[INIT] ✅ Solver trace diagnostics enabled (use run_search_traced for observational analysis)


In [15]:
# CELL 04
# Negative Space Filter for Logic Family VII and beyond

VOID_TOKEN = "__VOID__"

def normalize_grid(grid: List[List[int]]) -> List[List[Any]]:
    """
    Converts implicit background into explicit VOID entities.
    """
    normalized = []
    for row in grid:
        normalized.append([
            cell if cell is not None else VOID_TOKEN
            for cell in row
        ])
    return normalized


def denormalize_grid(grid: List[List[Any]]) -> List[List[int]]:
    """
    Converts VOID entities back to canonical background (0).
    """
    return [
        [0 if cell == VOID_TOKEN else cell for cell in row]
        for row in grid
    ]

In [16]:
# CELL 05
# Base class for all deterministic transform primitives

class TransformPrimitive:
    LOGIC_ID = "BASE"
    DS_ESTIMATE = 0.0
    DK_ESTIMATE = 0.0

    def __call__(self, registry: Registry) -> Tuple[Registry, ExecutionReceipt]:
        """
        Enforces AO3, idempotency, and refusal semantics.
        """
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)

        if not ao3.allowed:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason=ao3.reason
            )
            return registry, receipt

        # Explicit refusal until logic is provably correct
        receipt = ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="refused",
            reason="Logic not yet implemented"
        )
        return registry, receipt

In [17]:
# CELL 05a
# Canonical DS / DK estimation contracts

class ComplexityContract:
    """
    Static, pre-execution complexity contract.
    All values must be computable WITHOUT inspecting runtime data.
    """

    def __init__(self, DS: float, DK: float, justification: str):
        self.DS = DS
        self.DK = DK
        self.justification = justification

    def snapshot(self) -> Dict[str, Any]:
        return {
            "DS": self.DS,
            "DK": self.DK,
            "justification": self.justification
        }

In [18]:
# CELL 05b
# Static Logic Family → Complexity Contract registry

LOGIC_FAMILY_CONTRACTS: Dict[str, ComplexityContract] = {
    # Placeholder families — values are conservative and bounded

    "FAMILY_VI_RELATIONAL": ComplexityContract(
        DS=0.05,
        DK=10,
        justification="Fixed relational anchoring, bounded enumeration"
    ),

    "FAMILY_VII_INTERACTION": ComplexityContract(
        DS=0.08,
        DK=20,
        justification="Interaction logic with negative space preservation"
    ),

    "FAMILY_VIII_FRACTAL": ComplexityContract(
        DS=0.40,
        DK=5000,
        justification="Fractal tiling; high complexity, tightly governed"
    ),
}

In [19]:
# CELL 05c
# Deterministic DS / DK resolution and validation

def resolve_complexity_contract(logic_family: str) -> ComplexityContract:
    if logic_family not in LOGIC_FAMILY_CONTRACTS:
        raise ValueError(
            f"Unregistered logic family: {logic_family}"
        )
    return LOGIC_FAMILY_CONTRACTS[logic_family]


def validate_contract(contract: ComplexityContract) -> None:
    """
    Enforces hard invariants on DS/DK values.
    """
    assert 0.0 <= contract.DS <= 1.0, "DS must be normalized [0,1]"
    assert contract.DK >= 0, "DK must be non-negative"

In [20]:
# CELL 05d
# Governance-aware transform with DS/DK contract binding

class GovernedTransform(TransformPrimitive):
    LOGIC_FAMILY = None  # must be declared

    def __init__(self):
        if self.LOGIC_FAMILY is None:
            raise ValueError("LOGIC_FAMILY must be declared")

        contract = resolve_complexity_contract(self.LOGIC_FAMILY)
        validate_contract(contract)

        self.DS_ESTIMATE = contract.DS
        self.DK_ESTIMATE = contract.DK
        self._contract_snapshot = contract.snapshot()

    def contract_receipt(self) -> Dict[str, Any]:
        return self._contract_snapshot

In [21]:
# CELL 05e
# Optional receipt enrichment with governance metadata

def annotate_receipt(
    receipt: Dict[str, str],
    contract_snapshot: Optional[Dict[str, Any]] = None
) -> Dict[str, Any]:

    enriched = dict(receipt)
    if contract_snapshot is not None:
        enriched["governance_contract"] = contract_snapshot

    return enriched

In [22]:
# CELL 05f
# Logic Family VI — Relational Anchoring (Detection-Only, Governed)

class LogicFamilyVI_RelationalAnchor(GovernedTransform):
    """
    Relational Anchoring (Family VI)

    This logic family detects whether a grid has a single,
    stable relational anchor that is invariant under identity.
    No transformation is applied at this stage.

    Output behavior:
    - If anchoring is provably stable → identity allowed
    - Otherwise → deterministic refusal
    """

    LOGIC_ID = "LOGIC_VI_RELATIONAL_ANCHOR"
    LOGIC_FAMILY = "FAMILY_VI_RELATIONAL"

    BACKGROUND_COLOR = 0

    def __call__(self, registry: Registry):
        # Enforce AO3 first (inherited behavior)
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)

        if not ao3.allowed:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )
            return registry, receipt

        # Expect input grid to be present
        grid = registry.read("input_grid")

        if grid is None:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available for anchoring"
            )
            return registry, receipt

        # --- Relational Anchor Detection ---
        colors = set()
        for row in grid:
            for cell in row:
                if cell != self.BACKGROUND_COLOR:
                    colors.add(cell)

        # Governance rule:
        # Exactly ONE non-background color → provably anchored
        if len(colors) == 1:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="executed",
                reason="Single relational anchor detected; identity preserved"
            )
            return registry, receipt

        # Otherwise: ambiguous anchoring → refuse
        receipt = ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="refused",
            reason="Multiple or ambiguous anchors; refusal enforced"
        )
        return registry, receipt

In [23]:
# CELL 05g
# Logic Family I — Axis Reflection (Detection-Only, Governed)

class LogicFamilyI_AxisReflection(GovernedTransform):
    """
    Logic Family I: Axis Reflection

    This logic family deterministically detects whether the input grid
    possesses a UNIQUE axis of reflection symmetry.

    Axes evaluated:
    - Vertical (left-right)
    - Horizontal (top-bottom)

    Behavior:
    - If exactly one axis is provably valid → executed (certified)
    - If none or more than one axis is valid → refused
    - No transformation is applied at this stage
    """

    LOGIC_ID = "LOGIC_I_AXIS_REFLECTION"
    LOGIC_FAMILY = "FAMILY_VI_RELATIONAL"  # conservative DS/DK reuse

    def _is_vertical_reflection(self, grid):
        h = len(grid)
        w = len(grid[0])
        for r in range(h):
            for c in range(w):
                if grid[r][c] != grid[r][w - 1 - c]:
                    return False
        return True

    def _is_horizontal_reflection(self, grid):
        h = len(grid)
        w = len(grid[0])
        for r in range(h):
            for c in range(w):
                if grid[r][c] != grid[h - 1 - r][c]:
                    return False
        return True

    def __call__(self, registry: Registry):
        # AO3 enforcement (pre-execution, deterministic)
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)

        if not ao3.allowed:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )
            return registry, receipt

        grid = registry.read("input_grid")

        if grid is None:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available for axis reflection detection"
            )
            return registry, receipt

        vertical = self._is_vertical_reflection(grid)
        horizontal = self._is_horizontal_reflection(grid)

        axis_count = int(vertical) + int(horizontal)

        # Governance rule:
        # Exactly ONE valid axis → certified reflection structure
        if axis_count == 1:
            axis = "vertical" if vertical else "horizontal"
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="executed",
                reason=f"Unique {axis} reflection axis detected; structure certified"
            )
            return registry, receipt

        # Ambiguous or absent symmetry → refusal
        receipt = ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="refused",
            reason="No unique reflection axis; ambiguity enforced"
        )
        return registry, receipt

In [24]:
# CELL 05h
# Logic Family VII — Interaction (Detection-Only, Governed)

class LogicFamilyVII_Interaction(GovernedTransform):
    """
    Logic Family VII: Interaction Detection

    Detects whether the grid exhibits a deterministic foreground ↔ background
    interaction pattern, with background (negative space) treated as a
    first-class entity.

    This logic family DOES NOT apply XOR or interaction transforms.
    It only certifies whether interaction structure is:
      - Present
      - Non-ambiguous
      - Safe to reason about later

    Governance Rules:
    - VOID / background must participate explicitly
    - Multiple ambiguous interactions → refusal
    """

    LOGIC_ID = "LOGIC_VII_INTERACTION"
    LOGIC_FAMILY = "FAMILY_VII_INTERACTION"

    BACKGROUND_COLOR = 0

    def __call__(self, registry: Registry):
        # AO3 enforcement (pre-execution)
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)

        if not ao3.allowed:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )
            return registry, receipt

        grid = registry.read("input_grid")

        if grid is None:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available for interaction detection"
            )
            return registry, receipt

        # --- Negative Space Normalization ---
        normalized = normalize_grid(grid)

        foreground_colors = set()
        void_count = 0

        for row in normalized:
            for cell in row:
                if cell == VOID_TOKEN:
                    void_count += 1
                else:
                    foreground_colors.add(cell)

        # Governance rule 1:
        # Interaction requires BOTH foreground and void participation
        if void_count == 0 or len(foreground_colors) == 0:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No foreground-background interaction detected"
            )
            return registry, receipt

        # Governance rule 2:
        # Exactly ONE foreground color interacting with void is certifiable
        if len(foreground_colors) == 1:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="executed",
                reason="Single foreground interacting with negative space; interaction certified"
            )
            return registry, receipt

        # Anything else is ambiguous interaction
        receipt = ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="refused",
            reason="Multiple interacting foregrounds; ambiguity enforced"
        )
        return registry, receipt

In [25]:
# CELL 05i
# Logic Family I — Governed Axis Reflection Transform (Gated)

class GatedAxisReflectionTransform(GovernedTransform):
    """
    Governed Axis Reflection Transform

    Applies a single axis reflection ONLY IF:
    - Exactly one reflection axis (vertical or horizontal) is provably valid
    - AO3 permits execution
    - No ambiguity exists

    Writes:
    - 'output_grid' into the registry on success

    Refuses deterministically otherwise.
    """

    LOGIC_ID = "LOGIC_I_AXIS_REFLECTION_TRANSFORM"
    LOGIC_FAMILY = "FAMILY_VI_RELATIONAL"  # conservative contract reuse

    def _is_vertical_reflection(self, grid):
        h = len(grid)
        w = len(grid[0])
        for r in range(h):
            for c in range(w):
                if grid[r][c] != grid[r][w - 1 - c]:
                    return False
        return True

    def _is_horizontal_reflection(self, grid):
        h = len(grid)
        w = len(grid[0])
        for r in range(h):
            for c in range(w):
                if grid[r][c] != grid[h - 1 - r][c]:
                    return False
        return True

    def _apply_vertical(self, grid):
        return [list(reversed(row)) for row in grid]

    def _apply_horizontal(self, grid):
        return list(reversed([list(row) for row in grid]))

    def __call__(self, registry: Registry):
        # AO3 enforcement (pre-execution)
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)

        if not ao3.allowed:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )
            return registry, receipt

        grid = registry.read("input_grid")

        if grid is None:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available for reflection transform"
            )
            return registry, receipt

        vertical = self._is_vertical_reflection(grid)
        horizontal = self._is_horizontal_reflection(grid)

        axis_count = int(vertical) + int(horizontal)

        # Governance gate: EXACTLY ONE axis
        if axis_count != 1:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No unique reflection axis; transform gated"
            )
            return registry, receipt

        # Apply the certified reflection
        if vertical:
            output_grid = self._apply_vertical(grid)
            axis = "vertical"
        else:
            output_grid = self._apply_horizontal(grid)
            axis = "horizontal"

        # Write output grid immutably
        registry = registry.write("output_grid", output_grid)

        receipt = ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="executed",
            reason=f"{axis.capitalize()} reflection applied under governance"
        )

        return registry, receipt

In [26]:
# CELL 05j
# Logic Family I — Governed Horizontal Reflection Transform (Gated)

class GatedHorizontalReflectionTransform(GovernedTransform):
    """
    Governed Horizontal Reflection Transform

    Applies a horizontal (top-bottom) reflection ONLY IF:
    - The grid is provably horizontally symmetric
    - The grid is NOT vertically symmetric
    - AO3 permits execution

    This guarantees:
    - Exactly one axis
    - No ambiguity
    """

    LOGIC_ID = "LOGIC_I_HORIZONTAL_REFLECTION_TRANSFORM"
    LOGIC_FAMILY = "FAMILY_VI_RELATIONAL"  # conservative contract reuse

    def _is_horizontal_reflection(self, grid):
        h = len(grid)
        w = len(grid[0])
        for r in range(h):
            for c in range(w):
                if grid[r][c] != grid[h - 1 - r][c]:
                    return False
        return True

    def _is_vertical_reflection(self, grid):
        h = len(grid)
        w = len(grid[0])
        for r in range(h):
            for c in range(w):
                if grid[r][c] != grid[r][w - 1 - c]:
                    return False
        return True

    def _apply_horizontal(self, grid):
        return list(reversed([list(row) for row in grid]))

    def __call__(self, registry: Registry):
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)

        if not ao3.allowed:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )
            return registry, receipt

        grid = registry.read("input_grid")

        if grid is None:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available for horizontal reflection"
            )
            return registry, receipt

        horizontal = self._is_horizontal_reflection(grid)
        vertical = self._is_vertical_reflection(grid)

        # Governance gate: horizontal ONLY
        if horizontal and not vertical:
            output_grid = self._apply_horizontal(grid)
            registry = registry.write("output_grid", output_grid)

            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="executed",
                reason="Horizontal reflection applied under governance"
            )
            return registry, receipt

        receipt = ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="refused",
            reason="Horizontal reflection not uniquely valid"
        )
        return registry, receipt

In [27]:
# CELL 05k
# Logic Family VII — Governed Interaction Transform (Gated)

class GatedInteractionTransform(GovernedTransform):
    """
    Governed Interaction (XOR-style) Transform

    Applies a deterministic foreground-background interaction ONLY IF:
    - Logic Family VII detection executed
    - Exactly one foreground color is present
    - Background (void) participates
    - AO3 permits execution

    Writes:
    - 'output_grid' into registry on success
    """

    LOGIC_ID = "LOGIC_VII_INTERACTION_TRANSFORM"
    LOGIC_FAMILY = "FAMILY_VII_INTERACTION"

    BACKGROUND_COLOR = 0

    def __call__(self, registry: Registry):
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)

        if not ao3.allowed:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )
            return registry, receipt

        grid = registry.read("input_grid")

        if grid is None:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available for interaction transform"
            )
            return registry, receipt

        # Normalize grid (explicit negative space)
        normalized = normalize_grid(grid)

        foreground_colors = set()
        for row in normalized:
            for cell in row:
                if cell != VOID_TOKEN:
                    foreground_colors.add(cell)

        # Governance gate: exactly one foreground color
        if len(foreground_colors) != 1:
            receipt = ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="Ambiguous foreground; interaction gated"
            )
            return registry, receipt

        fg = next(iter(foreground_colors))

        # XOR-style interaction:
        # foreground ↔ void toggle
        output = []
        for row in normalized:
            out_row = []
            for cell in row:
                if cell == VOID_TOKEN:
                    out_row.append(fg)
                else:
                    out_row.append(self.BACKGROUND_COLOR)
            output.append(out_row)

        registry = registry.write("output_grid", output)

        receipt = ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="executed",
            reason="Foreground-background interaction applied under governance"
        )

        return registry, receipt

In [28]:
# CELL 06
# Top-level deterministic governance loop

def governed_execute(
    initial_registry: Registry,
    transforms: List[TransformPrimitive]
) -> Tuple[Registry, List[Dict[str, str]]]:

    registry = initial_registry
    receipts: List[Dict[str, str]] = []

    for transform in transforms:
        registry, receipt = transform(registry)
        receipts.append(receipt.as_dict())

    return registry, receipts

In [29]:
# CELL 06a
# Governed execution with automatic contract metadata attachment

def governed_execute_with_contracts(
    initial_registry: Registry,
    transforms: list
):
    """
    Executes transforms deterministically and enriches receipts
    with DS/DK contract metadata when available.

    No behavior changes.
    Metadata-only augmentation.
    """

    registry = initial_registry
    enriched_receipts = []

    for transform in transforms:
        registry, receipt = transform(registry)

        receipt_dict = receipt.as_dict()

        # Attach contract metadata if this is a governed transform
        if hasattr(transform, "contract_receipt"):
            receipt_dict = annotate_receipt(
                receipt=receipt_dict,
                contract_snapshot=transform.contract_receipt()
            )

        enriched_receipts.append(receipt_dict)

    return registry, enriched_receipts

In [30]:
# CELL 06b
# Governance gate evaluation utilities

def has_executed(logic_id: str, receipts: list) -> bool:
    """
    Returns True if a logic family executed successfully.
    """
    for r in receipts:
        if r.get("logic_id") == logic_id and r.get("status") == "executed":
            return True
    return False

In [31]:
# CELL 06c
# Explicit gating controller (V15)

def gating_controller(detection_receipts: list):
    """
    Determines which transforms are authorized to run
    based solely on deterministic detection receipts.

    Returns:
        List of TransformPrimitive instances (authorized only)
    """

    authorized_transforms = []

    # ---- Gate 1: Axis Reflection ----
    # Require Logic Family I certification
    if has_executed("LOGIC_I_AXIS_REFLECTION", detection_receipts):
        authorized_transforms.append(
            GatedAxisReflectionTransform()
        )

    # Future gates (examples, not active):
    # - Interaction XOR
    # - Relational projection
    # - Mask application

    return authorized_transforms

In [32]:
# CELL 06d
# Exclusive gating controller for axis-specific transforms

def gating_controller_v2(detection_receipts: list):
    """
    Determines exactly ONE authorized transform, if any,
    based on deterministic detection receipts.

    Priority:
    - Vertical reflection transform
    - Horizontal reflection transform
    """

    authorized = []

    axis_detected = has_executed("LOGIC_I_AXIS_REFLECTION", detection_receipts)

    if not axis_detected:
        return authorized

    # Axis detection logic is re-verified inside transforms
    # Controller only authorizes candidates

    authorized.append(GatedAxisReflectionTransform())
    authorized.append(GatedHorizontalReflectionTransform())

    return authorized

In [33]:
# CELL 06e
# Exclusive gating controller v3 (Reflection vs Interaction)

def gating_controller_v3(detection_receipts: list):
    """
    Determines exactly ONE gated transform based on detection evidence.

    Priority order (strict):
    1. Axis Reflection (Logic Family I)
    2. Interaction (Logic Family VII)

    Returns:
        List with at most one transform instance
    """

    # --- Priority 1: Reflection ---
    if has_executed("LOGIC_I_AXIS_REFLECTION", detection_receipts):
        return [
            GatedAxisReflectionTransform(),
            GatedHorizontalReflectionTransform(),
        ]

    # --- Priority 2: Interaction ---
    if has_executed("LOGIC_VII_INTERACTION", detection_receipts):
        return [GatedInteractionTransform()]

    return []

In [34]:
# CELL 06f
# Conflict resolution receipt construction

def conflict_resolution_receipt(
    chosen: str,
    rejected: list,
    policy: str
) -> dict:
    """
    Creates an explicit governance receipt explaining
    why one transform was chosen over others.

    Parameters:
        chosen: logic_id of the authorized path
        rejected: list of logic_ids that were rejected
        policy: deterministic policy description
    """
    return {
        "logic_id": "GOVERNANCE_CONFLICT_RESOLUTION",
        "status": "resolved",
        "chosen": chosen,
        "rejected": rejected,
        "policy": policy,
    }

In [35]:
# CELL 06g
# Exclusive gating controller v4 with conflict-resolution metadata

def gating_controller_v4(detection_receipts: list):
    """
    Determines authorized gated transforms AND emits
    a conflict-resolution receipt explaining the decision.

    Priority order (strict):
    1. Axis Reflection (Logic Family I)
    2. Interaction (Logic Family VII)

    Returns:
        (authorized_transforms, conflict_receipt_or_none)
    """

    # --- Priority 1: Reflection ---
    if has_executed("LOGIC_I_AXIS_REFLECTION", detection_receipts):
        receipt = conflict_resolution_receipt(
            chosen="LOGIC_I_AXIS_REFLECTION",
            rejected=["LOGIC_VII_INTERACTION"],
            policy="Reflection has priority over interaction"
        )
        return (
            [
                GatedAxisReflectionTransform(),
                GatedHorizontalReflectionTransform(),
            ],
            receipt
        )

    # --- Priority 2: Interaction ---
    if has_executed("LOGIC_VII_INTERACTION", detection_receipts):
        receipt = conflict_resolution_receipt(
            chosen="LOGIC_VII_INTERACTION",
            rejected=[],
            policy="No higher-priority transform applicable"
        )
        return (
            [GatedInteractionTransform()],
            receipt
        )

    # --- No authorization ---
    return ([], None)

In [36]:
# CELL 06h
# Per-attempt governance summary receipt

def governance_summary_receipt(
    attempt_id: int,
    detection_receipts: list,
    conflict_receipt: dict | None,
    transform_executed: bool,
    fallback_used: bool
) -> dict:
    """
    Constructs a deterministic per-attempt governance summary.
    """

    return {
        "logic_id": "GOVERNANCE_ATTEMPT_SUMMARY",
        "status": "completed",
        "attempt": attempt_id,
        "detections_executed": [
            r["logic_id"] for r in detection_receipts
            if r.get("status") == "executed"
        ],
        "conflict_resolution": (
            conflict_receipt.get("chosen")
            if conflict_receipt is not None
            else None
        ),
        "transform_executed": transform_executed,
        "fallback_used": fallback_used,
    }

In [37]:
# CELL 06i
# Transform rollback receipt

def transform_rollback_receipt(
    authorized: list,
    reason: str
) -> dict:
    """
    Emits a deterministic rollback receipt when
    authorized transforms fail to execute.
    """

    return {
        "logic_id": "GOVERNANCE_TRANSFORM_ROLLBACK",
        "status": "rolled_back",
        "authorized_transforms": authorized,
        "reason": reason,
    }

In [38]:
# CELL 06j
# Gating controller v5 with termination override

def gating_controller_v5(detection_receipts: list):
    """
    Determines authorized transforms with TERMINATION taking absolute priority.

    Priority order:
    0. Termination (Logic Family VIII) → no transforms allowed
    1. Axis Reflection
    2. Interaction

    Returns:
        (authorized_transforms, conflict_receipt_or_none)
    """

    # --- Priority 0: Termination ---
    if has_executed("LOGIC_VIII_TERMINATION_SENTINEL", detection_receipts):
        receipt = conflict_resolution_receipt(
            chosen="LOGIC_VIII_TERMINATION_SENTINEL",
            rejected=[
                "LOGIC_I_AXIS_REFLECTION",
                "LOGIC_VII_INTERACTION"
            ],
            policy="Termination sentinel overrides all actuation"
        )
        return ([], receipt)

    # --- Priority 1: Reflection ---
    if has_executed("LOGIC_I_AXIS_REFLECTION", detection_receipts):
        receipt = conflict_resolution_receipt(
            chosen="LOGIC_I_AXIS_REFLECTION",
            rejected=["LOGIC_VII_INTERACTION"],
            policy="Reflection has priority over interaction"
        )
        return (
            [
                GatedAxisReflectionTransform(),
                GatedHorizontalReflectionTransform(),
            ],
            receipt
        )

    # --- Priority 2: Interaction ---
    if has_executed("LOGIC_VII_INTERACTION", detection_receipts):
        receipt = conflict_resolution_receipt(
            chosen="LOGIC_VII_INTERACTION",
            rejected=[],
            policy="No higher-priority transform applicable"
        )
        return ([GatedInteractionTransform()], receipt)

    return ([], None)

In [39]:
# CELL 06k
# V16 capability escalation gate

def capability_escalation_allowed(detection_receipts: list) -> bool:
    """
    Determines whether MICRO_COMPOSE capability
    may be granted for this attempt.

    Rule (V16):
    - At least ONE detection executed
    - NO termination sentinel executed
    """

    if has_executed("LOGIC_VIII_TERMINATION_SENTINEL", detection_receipts):
        return False

    # Any structural certainty allows micro composition
    for r in detection_receipts:
        if r.get("status") == "executed":
            return True

    return False

In [40]:
# CELL 06l
# Capability escalation receipt

def capability_escalation_receipt(
    granted: bool,
    level: int
) -> dict:
    return {
        "logic_id": "GOVERNANCE_CAPABILITY_ESCALATION",
        "status": "granted" if granted else "denied",
        "capability_level": level if granted else CapabilityLevel.BASELINE,
        "policy": "V16 micro-composition escalation rules",
    }

In [41]:
# CELL 06m
# V16 micro-composition pattern resolver

def resolve_micro_composition_pattern(detection_receipts: list) -> str | None:
    """
    Determines which V16 micro-composition pattern is applicable.

    Returns:
        "PATTERN_REFLECT_REPEAT"
        "PATTERN_INTERACT_THEN_REFLECT"
        or None
    """

    has_reflection = has_executed("LOGIC_I_AXIS_REFLECTION", detection_receipts)
    has_interaction = has_executed("LOGIC_VII_INTERACTION", detection_receipts)

    # Pattern 2: Interaction → Reflection
    if has_interaction and has_reflection:
        return "PATTERN_INTERACT_THEN_REFLECT"

    # Pattern 1 (existing): Reflection replay / repeat
    if has_reflection:
        return "PATTERN_REFLECT_REPEAT"

    return None

In [42]:
# CELL 06o
# Capability extraction utilities for cross-attempt comparison

def extract_final_capability(receipts: list) -> int:
    """
    Determines the final capability level for an attempt
    by inspecting escalation and decay receipts.

    Rules:
    - Decay overrides escalation
    - Default is BASELINE
    """

    capability = CapabilityLevel.BASELINE

    for r in receipts:
        if r.get("logic_id") == "GOVERNANCE_CAPABILITY_ESCALATION":
            if r.get("status") == "granted":
                capability = r.get("capability_level", CapabilityLevel.BASELINE)

        if r.get("logic_id") == "GOVERNANCE_CAPABILITY_DECAY":
            capability = CapabilityLevel.BASELINE

    return capability

In [43]:
# CELL 06p
# Cross-attempt capability consistency receipt

def cross_attempt_capability_consistency_receipt(
    attempt_1_receipts: list,
    attempt_2_receipts: list
) -> dict:
    """
    Compares capability outcomes across both attempts
    and emits a governance consistency receipt.
    """

    cap_1 = extract_final_capability(attempt_1_receipts)
    cap_2 = extract_final_capability(attempt_2_receipts)

    if cap_1 == cap_2:
        status = "consistent"
        note = "Capability behavior consistent across attempts"
    else:
        status = "divergent"
        note = "Capability behavior diverged across attempts"

    return {
        "logic_id": "GOVERNANCE_CROSS_ATTEMPT_CAPABILITY",
        "status": status,
        "attempt_1_capability": cap_1,
        "attempt_2_capability": cap_2,
        "policy": "V16 cross-attempt capability consistency check",
        "note": note,
    }

In [44]:
# CELL 06q
# Empirical AO3 evaluation receipt

def empirical_ao3_receipt(
    phase: str,
    DS: float,
    DK: float,
    threshold: float
) -> dict:
    """
    Records AO3 inputs and threshold for empirical analysis.
    Does NOT affect execution.
    """
    J = DS + (DK * 1e-4)
    return {
        "logic_id": "EMPIRICAL_AO3_EVALUATION",
        "phase": phase,
        "DS": DS,
        "DK": DK,
        "J": J,
        "threshold": threshold,
        "within_bounds": J <= threshold,
    }

In [45]:
# CELL 06r
# Empirical capability budget receipt

def empirical_capability_budget_receipt(
    expansion_state: ExpansionState
) -> dict:
    """
    Records capability usage for empirical tuning.
    """
    snap = expansion_state.snapshot()
    return {
        "logic_id": "EMPIRICAL_CAPABILITY_BUDGET",
        "capability": snap["capability"],
        "used_budget": snap["used_budget"],
        "remaining_budget": snap["remaining_budget"],
    }

In [46]:
# CELL 06s
# Planning–execution agreement receipt

def planning_execution_agreement_receipt(
    planning_receipt: dict | None,
    execution_receipts: list
) -> dict:
    """
    Determines whether execution agreed with the planner's choice.

    Agreement rules:
    - If no plan was proposed → agreement = not_applicable
    - If plan proposed and matching transform executed → agreement = yes
    - If plan proposed and different/no transform executed → agreement = no
    """

    if planning_receipt is None or planning_receipt.get("status") != "chosen":
        return {
            "logic_id": "GOVERNANCE_PLANNING_EXECUTION_AGREEMENT",
            "status": "not_applicable",
            "reason": "No planning choice was proposed",
        }

    planned_steps = planning_receipt.get("choice", [])
    planned_logic_ids = {step["logic_id"] for step in planned_steps}

    executed_logic_ids = {
        r.get("logic_id")
        for r in execution_receipts
        if r.get("status") == "executed"
    }

    agreed = len(planned_logic_ids & executed_logic_ids) > 0

    return {
        "logic_id": "GOVERNANCE_PLANNING_EXECUTION_AGREEMENT",
        "status": "yes" if agreed else "no",
        "planned": list(planned_logic_ids),
        "executed": list(executed_logic_ids),
        "policy": "V17 single-step planning agreement check",
    }

In [47]:
# CELL 06t
# Capability decay due to planner–execution disagreement

def planner_disagreement_decay_receipt(
    planned: list,
    executed: list
) -> dict:
    """
    Emits a capability decay receipt when execution
    does not follow the planner's proposed transform.
    """
    return {
        "logic_id": "GOVERNANCE_CAPABILITY_DECAY",
        "status": "decayed",
        "from_capability": CapabilityLevel.MICRO_COMPOSE,
        "to_capability": CapabilityLevel.BASELINE,
        "reason": "Planner–execution disagreement",
        "planned": planned,
        "executed": executed,
    }

In [48]:
# CELL 06u
# Planner abstention capability preservation receipt

def planner_abstention_preservation_receipt() -> dict:
    """
    Emits a receipt indicating that the planner abstained
    and capability was explicitly preserved.
    """
    return {
        "logic_id": "GOVERNANCE_PLANNER_ABSTENTION",
        "status": "preserved",
        "policy": "Planner abstention preserves capability",
    }

In [49]:
# CELL 07
# Zero State Baseline execution entry

def run_zero_state():
    empty_registry = Registry(data={})

    final_registry, receipts = governed_execute(
        initial_registry=empty_registry,
        transforms=[]
    )

    return {
        "final_registry": final_registry.snapshot(),
        "receipts": receipts,
        "output": REFUSAL_GRID
    }

In [50]:
# CELL 07a
# Canonical refusal semantics (V15)

REFUSAL_POLICY = "IDENTITY"

def resolve_refusal_grid(input_grid):
    """
    Canonical refusal resolution.
    Returns an identity-preserving grid.
    """
    if input_grid is None:
        raise ValueError("Input grid required for canonical refusal")

    # Identity refusal: exact echo
    return [list(row) for row in input_grid]

In [51]:
# CELL 07b
# Governance receipt for refusal semantics

def refusal_receipt():
    return {
        "status": "refused",
        "policy": REFUSAL_POLICY,
        "justification": "No provably deterministic transform applicable"
    }

In [52]:
# CELL 07m
# Morphology Invariant Set (Deterministic, AO3-Safe)

from collections import Counter

def morphology_invariants(input_grid, output_grid, background):
    """
    Returns True iff output_grid satisfies the minimal morphology invariant set
    relative to input_grid.
    """

    # Invariant 1 — Background preserved
    if background not in set(sum(input_grid, [])):
        return False
    if background not in set(sum(output_grid, [])):
        return False

    # Invariant 2 — Foreground color subset
    in_colors = set(sum(input_grid, [])) - {background}
    out_colors = set(sum(output_grid, [])) - {background}
    if not out_colors.issubset(in_colors):
        return False

    # Invariant 3 — Connected component monotonicity
    def count_components(grid):
        visited = set()
        comps = 0
        for r in range(len(grid)):
            for c in range(len(grid[0])):
                if grid[r][c] != background and (r,c) not in visited:
                    stack = [(r,c)]
                    visited.add((r,c))
                    while stack:
                        x,y = stack.pop()
                        for dx,dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                            nx,ny = x+dx, y+dy
                            if 0 <= nx < len(grid) and 0 <= ny < len(grid[0]):
                                if grid[nx][ny] != background and (nx,ny) not in visited:
                                    visited.add((nx,ny))
                                    stack.append((nx,ny))
                    comps += 1
        return comps

    if count_components(output_grid) > count_components(input_grid):
        return False

    # Invariant 4 — Union bounding box stability
    def bbox(grid):
        coords = [(r,c) for r in range(len(grid))
                         for c in range(len(grid[0]))
                         if grid[r][c] != background]
        if not coords:
            return None
        rs,cs = zip(*coords)
        return min(rs), max(rs), min(cs), max(cs)

    if bbox(input_grid) != bbox(output_grid):
        return False

    # Invariant 5 — Idempotence
    # Caller must check: T(T(grid)) == T(grid)

    return True

In [53]:
# CELL 07n
# Morphology Equivalence-Class Determination

def morphology_equivalent(input_grid, outputs, background):
    """
    outputs: list of candidate output grids from morphology transforms
    Returns True iff all outputs are invariant-equivalent.
    """
    if not outputs:
        return False

    reference = outputs[0]

    for out in outputs:
        if out != reference:
            if not morphology_invariants(input_grid, out, background):
                return False

    return True


In [54]:
# CELL 07o
# Morphology Equivalence Authorization Gate

MORPHOLOGY_FAMILIES = {"Morphology"}

def allow_morphology_equivalence(logic_family):
    return logic_family in MORPHOLOGY_FAMILIES

In [55]:
# CELL 07p
# Equivalence-Class Actuation Hook (Morphology Only)

def gated_morphology_actuation(input_grid, candidate_transforms, background):
    """
    candidate_transforms: list of (logic_id, transform_fn)
    Returns chosen transform or None
    """
    outputs = []
    for _, fn in candidate_transforms:
        out = fn(input_grid)
        outputs.append(out)

    if morphology_equivalent(input_grid, outputs, background):
        # Deterministic selection by logic_id order
        return candidate_transforms[0]

    return None

In [56]:
# CELL 07q
# Morphology Primitive: Hole Fill (Pure, Deterministic)

from collections import deque

def fill_holes(grid, background=0):
    """
    Deterministic hole-filling for binary foreground/background grids.
    Fills enclosed background regions with foreground.
    """
    h, w = len(grid), len(grid[0])
    visited = [[False]*w for _ in range(h)]
    result = [row[:] for row in grid]

    # Flood-fill from border background
    q = deque()
    for r in range(h):
        for c in [0, w-1]:
            if grid[r][c] == background and not visited[r][c]:
                visited[r][c] = True
                q.append((r, c))
    for c in range(w):
        for r in [0, h-1]:
            if grid[r][c] == background and not visited[r][c]:
                visited[r][c] = True
                q.append((r, c))

    while q:
        r, c = q.popleft()
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < h and 0 <= nc < w:
                if grid[nr][nc] == background and not visited[nr][nc]:
                    visited[nr][nc] = True
                    q.append((nr, nc))

    # Any unvisited background cell is a hole → fill it
    for r in range(h):
        for c in range(w):
            if grid[r][c] == background and not visited[r][c]:
                # fill with first non-background color seen
                # deterministic choice: scan order
                for rr in range(h):
                    for cc in range(w):
                        if grid[rr][cc] != background:
                            result[r][c] = grid[rr][cc]
                            break
                    else:
                        continue
                    break

    return result
        

In [57]:
# CELL 07r
# LogicFamily: Morphology — Governed HoleFill

class LogicFamily_Morphology_HoleFill:
    LOGIC_ID = "LOGIC_MORPH_HOLEFILL"
    LOGIC_FAMILY = "Morphology"
    DS_ESTIMATE = 0.05
    DK_ESTIMATE = 50

    def __call__(self, registry):
        # AO3 enforcement
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)
        if not ao3.allowed:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )

        input_grid = registry.read("input_grid")
        if input_grid is None:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available"
            )

        background = 0

        # Apply HoleFill
        candidate = fill_holes(input_grid, background=background)

        # Invariant checks
        if not morphology_invariants(input_grid, candidate, background):
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="Morphology invariants violated"
            )

        # Idempotence check
        second_pass = fill_holes(candidate, background=background)
        if second_pass != candidate:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="HoleFill not idempotent"
            )

        # All checks passed → write output
        registry = registry.write("output_grid", candidate)
        return registry, ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="executed",
            reason="HoleFill executed under morphology invariants"
        )

In [58]:
# CELL 07s
# Morphology Logic Registration

def get_morphology_logic_transforms():
    return [
        LogicFamily_Morphology_HoleFill()
    ]

In [59]:
# CELL 07t
# Morphology Primitive: Largest Connected Component (Pure, Deterministic)

from collections import deque, defaultdict

def largest_component(grid, background=0):
    """
    Keeps only the largest connected foreground component.
    Deterministic: ties broken by first-seen scan order.
    """
    h, w = len(grid), len(grid[0])
    visited = [[False]*w for _ in range(h)]
    components = []

    for r in range(h):
        for c in range(w):
            if grid[r][c] != background and not visited[r][c]:
                stack = [(r, c)]
                visited[r][c] = True
                pixels = [(r, c)]

                while stack:
                    x, y = stack.pop()
                    for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
                        nx, ny = x+dx, y+dy
                        if 0 <= nx < h and 0 <= ny < w:
                            if grid[nx][ny] != background and not visited[nx][ny]:
                                visited[nx][ny] = True
                                stack.append((nx, ny))
                                pixels.append((nx, ny))

                components.append(pixels)

    if not components:
        return [row[:] for row in grid]

    # Choose largest; deterministic tie-break by scan order
    largest = max(components, key=lambda comp: len(comp))

    result = [[background]*w for _ in range(h)]
    for r, c in largest:
        result[r][c] = grid[r][c]

    return result

In [60]:
# CELL 07u
# LogicFamily: Morphology — Governed LargestComponent

class LogicFamily_Morphology_LargestComponent:
    LOGIC_ID = "LOGIC_MORPH_LARGEST_COMPONENT"
    LOGIC_FAMILY = "Morphology"
    DS_ESTIMATE = 0.04
    DK_ESTIMATE = 40

    def __call__(self, registry):
        # AO3 enforcement
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)
        if not ao3.allowed:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )

        input_grid = registry.read("input_grid")
        if input_grid is None:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available"
            )

        background = 0

        # Apply LargestComponent
        candidate = largest_component(input_grid, background=background)

        # Morphology invariant checks
        if not morphology_invariants(input_grid, candidate, background):
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="Morphology invariants violated"
            )

        # Idempotence check
        second_pass = largest_component(candidate, background=background)
        if second_pass != candidate:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="LargestComponent not idempotent"
            )

        # All checks passed → write output
        registry = registry.write("output_grid", candidate)
        return registry, ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="executed",
            reason="LargestComponent executed under morphology invariants"
        )

In [61]:
# CELL 07v
# Morphology Logic Registration (Extended)

def get_morphology_logic_transforms():
    return [
        LogicFamily_Morphology_HoleFill(),
        LogicFamily_Morphology_LargestComponent(),
    ]

In [62]:
# CELL 07w
# Morphology Equivalence-Class Actuation Gate
# Applies ONLY to Morphology family logic IDs

def morphology_equivalence_actuation(registry, logic_transforms, background=0):
    """
    registry: current Registry
    logic_transforms: list of instantiated morphology logic objects
                      (e.g., HoleFill, LargestComponent)

    Returns:
        (new_registry, ExecutionReceipt) if actuation occurs
        (registry, None) if no equivalence-class actuation is allowed
    """

    input_grid = registry.read("input_grid")
    if input_grid is None:
        return registry, None

    candidates = []
    outputs = []

    # Apply each morphology transform once
    for logic in logic_transforms:
        temp_registry = registry.copy()
        temp_registry, receipt = logic(temp_registry)

        if receipt.status != "executed":
            continue

        out_grid = temp_registry.read("output_grid")
        if out_grid is None:
            continue

        # Verify morphology invariants
        if not morphology_invariants(input_grid, out_grid, background):
            continue

        # Verify idempotence
        second_pass = logic(temp_registry.copy())[0].read("output_grid")
        if second_pass != out_grid:
            continue

        candidates.append((logic.LOGIC_ID, out_grid))
        outputs.append(out_grid)



In [63]:
# CELL 07v
# Morphology Logic Registration (Final, Equivalence-Enabled)

def get_morphology_logic_transforms():
    """
    Returns morphology logic IDs eligible for equivalence-class actuation.
    """
    return [
        LogicFamily_Morphology_HoleFill(),
        LogicFamily_Morphology_LargestComponent(),
        LogicFamily_Morphology_OutlineExtract(),
    ]

In [64]:
# CELL 07x
# Morphology Primitive: Outline Extract (Pure, Deterministic)

def outline_extract(grid, background=0):
    """
    Extracts the 4-connected outline of foreground components.
    Interior pixels are removed. Deterministic and idempotent.
    """
    h, w = len(grid), len(grid[0])
    result = [[background]*w for _ in range(h)]

    for r in range(h):
        for c in range(w):
            if grid[r][c] == background:
                continue

            # Check if any neighbor is background or out of bounds
            is_outline = False
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                nr, nc = r+dr, c+dc
                if not (0 <= nr < h and 0 <= nc < w):
                    is_outline = True
                    break
                if grid[nr][nc] == background:
                    is_outline = True
                    break

            if is_outline:
                result[r][c] = grid[r][c]

    return result

In [65]:
# CELL 07y
# LogicFamily: Morphology — Governed OutlineExtract

class LogicFamily_Morphology_OutlineExtract:
    LOGIC_ID = "LOGIC_MORPH_OUTLINE_EXTRACT"
    LOGIC_FAMILY = "Morphology"
    DS_ESTIMATE = 0.03
    DK_ESTIMATE = 35

    def __call__(self, registry):
        # AO3 enforcement
        ao3 = compute_ao3(self.DS_ESTIMATE, self.DK_ESTIMATE)
        if not ao3.allowed:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="terminated",
                reason="AO3 threshold exceeded"
            )

        input_grid = registry.read("input_grid")
        if input_grid is None:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="No input grid available"
            )

        background = 0

        # Apply OutlineExtract
        candidate = outline_extract(input_grid, background=background)

        # Morphology invariant checks
        if not morphology_invariants(input_grid, candidate, background):
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="Morphology invariants violated"
            )

        # Idempotence check
        second_pass = outline_extract(candidate, background=background)
        if second_pass != candidate:
            return registry, ExecutionReceipt(
                logic_id=self.LOGIC_ID,
                status="refused",
                reason="OutlineExtract not idempotent"
            )

        # All checks passed → write output
        registry = registry.write("output_grid", candidate)
        return registry, ExecutionReceipt(
            logic_id=self.LOGIC_ID,
            status="executed",
            reason="OutlineExtract executed under morphology invariants"
        )

In [66]:
# CELL 08
# Deterministic receipt persistence across ARC attempts

class ReceiptStore:
    """
    Immutable, append-only receipt store.
    One store per attempt.
    """
    def __init__(self, receipts: List[Dict[str, str]]):
        self._receipts = list(receipts)

    def append(self, receipt: Dict[str, str]) -> "ReceiptStore":
        new_receipts = list(self._receipts)
        new_receipts.append(receipt)
        return ReceiptStore(new_receipts)

    def snapshot(self) -> List[Dict[str, str]]:
        return list(self._receipts)

In [67]:
# CELL 08b
# Attempt Finalization / Resolution Layer
# Explicitly consumes ReceiptStore and Registry to produce a canonical attempt result

def finalize_attempt(registry, receipt_store):
    """
    Finalizes a single ARC attempt deterministically.

    Inputs:
      - registry: Registry instance for the attempt
      - receipt_store: ReceiptStore instance (immutable, append-only)

    Rules:
      1. If exactly one governed output_grid exists in the registry, emit it.
      2. If no output_grid exists, emit canonical refusal (identity grid).
      3. Receipts are treated as evidence, not as decision inputs.
    """

    # --- Defensive reads ---
    input_grid = registry.read("input_grid")
    output_grid = registry.read("output_grid")

    if input_grid is None:
        raise RuntimeError(
            "Finalization error: input_grid missing from registry"
        )

    # Snapshot receipts for auditability (not decision-making)
    receipts_snapshot = receipt_store.snapshot()

    # --- Case 1: A governed transform committed an output ---
    if output_grid is not None:
        return {
            "status": "executed",
            "final_grid": output_grid,
            "receipt_count": len(receipts_snapshot),
            "reason": "Governed output committed to registry"
        }

    # --- Case 2: No output committed → canonical refusal ---
    return {
        "status": "refused",
        "final_grid": input_grid,
        "receipt_count": len(receipts_snapshot),
        "reason": "No governed transform executed; canonical refusal"
    }


In [68]:
# CELL 09
# ARC-AGI-2 two-attempt execution controller

def execute_attempt(
    attempt_id: int,
    transforms: List[TransformPrimitive]
) -> Dict[str, Any]:

    assert attempt_id in (1, 2), "ARC allows exactly two attempts"

    # Each attempt starts from absolute zero state
    registry = Registry(data={})
    receipt_store = ReceiptStore(receipts=[])

    registry, receipts = governed_execute(
        initial_registry=registry,
        transforms=transforms
    )

    for r in receipts:
        receipt_store = receipt_store.append(r)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": REFUSAL_GRID  # still deterministic refusal
    }

In [69]:
# CELL 09a
# V15 Attempt Executor with canonical input_grid wiring

def execute_attempt_v15(
    attempt_id: int,
    input_grid,
    transforms: list
) -> dict:
    """
    V15-governed attempt execution.

    - Injects input_grid into registry at attempt start
    - Preserves absolute Zero State semantics
    - Enables Logic Families VI / I / VII to operate
    - Deterministic and replay-safe
    """

    assert attempt_id in (1, 2), "ARC allows exactly two attempts"
    assert input_grid is not None, "input_grid is required"

    # Absolute zero state registry
    registry = Registry(data={})

    # Canonical wiring: input_grid is a FIRST-CLASS registry entry
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    registry, receipts = governed_execute(
        initial_registry=registry,
        transforms=transforms
    )

    for r in receipts:
        receipt_store = receipt_store.append(r)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None  # resolved later by refusal / transform policy
    }

In [70]:
# CELL 09b
# V15 Attempt Executor with contract-enriched receipts

def execute_attempt_v15_with_contracts(
    attempt_id: int,
    input_grid,
    transforms: list
) -> dict:
    """
    V15 attempt execution with:
    - input_grid injection
    - AO3 enforcement
    - DS/DK contract metadata attached to receipts
    """

    assert attempt_id in (1, 2), "ARC allows exactly two attempts"
    assert input_grid is not None, "input_grid is required"

    # Zero State registry
    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    registry, receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=transforms
    )

    for r in receipts:
        receipt_store = receipt_store.append(r)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None  # resolved by refusal / transform policy
    }

In [71]:
# CELL 09c
# V15 two-phase execution with explicit gating controller

def execute_attempt_v15_with_gating(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V15 attempt in two explicit phases:

    Phase 1: Detection only
    Phase 2: Gated transforms only

    No transform can run unless authorized by the gating controller.
    """

    assert attempt_id in (1, 2), "ARC allows exactly two attempts"
    assert input_grid is not None, "input_grid is required"

    # ----- Phase 0: Zero State -----
    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- Phase 2: Explicit Gating -----
    gated_transforms = gating_controller(detection_receipts)

    if gated_transforms:
        registry, gated_receipts = governed_execute_with_contracts(
            initial_registry=registry,
            transforms=gated_transforms
        )

        for r in gated_receipts:
            receipt_store = receipt_store.append(r)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None  # resolved by submission policy
    }

In [72]:
# CELL 09d
# V15 two-phase execution with exclusive gated transforms

def execute_attempt_v15_with_exclusive_gating(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V15 attempt with:
    - Detection phase
    - Exclusive gated transforms
    - Deterministic resolution
    """

    assert attempt_id in (1, 2), "ARC allows exactly two attempts"
    assert input_grid is not None, "input_grid is required"

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # Phase 1 — Detection
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # Phase 2 — Exclusive Gating
    gated_transforms = gating_controller_v2(detection_receipts)

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())

        # Stop after first successful execution
        if receipt.status == "executed":
            break

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [73]:
# CELL 09e
# V15 two-phase execution with interaction gating

def execute_attempt_v15_with_interaction_gating(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V15 attempt with:
    - Detection phase
    - Exclusive gated transforms (reflection OR interaction)
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # Phase 1 — Detection
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # Phase 2 — Exclusive Gating
    gated_transforms = gating_controller_v3(detection_receipts)

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())

        if receipt.status == "executed":
            break

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [74]:
# CELL 09f
# V15 two-phase execution with conflict-resolution receipts

def execute_attempt_v15_with_conflict_receipts(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V15 attempt with:
    - Detection phase
    - Exclusive gated transforms
    - Explicit conflict-resolution receipts
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # Phase 1 — Detection
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # Phase 2 — Gating with conflict explanation
    gated_transforms, conflict_receipt = gating_controller_v4(detection_receipts)

    if conflict_receipt is not None:
        receipt_store = receipt_store.append(conflict_receipt)

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())

        if receipt.status == "executed":
            break

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [75]:
# CELL 09g
# V15 executor with per-attempt governance summary and rollback receipts

def execute_attempt_v15_with_full_governance(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes a V15 attempt with:
    - Detection
    - Explicit gating
    - Conflict resolution receipts
    - Transform rollback receipts
    - Per-attempt governance summary
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- Phase 2: Gating -----
    gated_transforms, conflict_receipt = gating_controller_v4(detection_receipts)

    if conflict_receipt is not None:
        receipt_store = receipt_store.append(conflict_receipt)

    transform_executed = False

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())

        if receipt.status == "executed":
            transform_executed = True
            break

    # ----- Rollback Detection -----
    if gated_transforms and not transform_executed:
        rollback = transform_rollback_receipt(
            authorized=[t.LOGIC_ID for t in gated_transforms],
            reason="All authorized transforms refused or terminated"
        )
        receipt_store = receipt_store.append(rollback)

    # ----- Governance Summary -----
    fallback_used = not transform_executed

    summary = governance_summary_receipt(
        attempt_id=attempt_id,
        detection_receipts=detection_receipts,
        conflict_receipt=conflict_receipt,
        transform_executed=transform_executed,
        fallback_used=fallback_used
    )

    receipt_store = receipt_store.append(summary)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [76]:
# CELL 09h
# V15 executor with termination sentinel support

def execute_attempt_v15_with_termination(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V15 attempt with:
    - Detection (including termination sentinel)
    - Termination-aware gating
    - Conflict, rollback, and summary receipts
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- Phase 2: Gating (termination-aware) -----
    gated_transforms, conflict_receipt = gating_controller_v5(detection_receipts)

    if conflict_receipt is not None:
        receipt_store = receipt_store.append(conflict_receipt)

    transform_executed = False

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())

        if receipt.status == "executed":
            transform_executed = True
            break

    # ----- Rollback -----
    if gated_transforms and not transform_executed:
        rollback = transform_rollback_receipt(
            authorized=[t.LOGIC_ID for t in gated_transforms],
            reason="Termination or refusal prevented execution"
        )
        receipt_store = receipt_store.append(rollback)

    # ----- Summary -----
    summary = governance_summary_receipt(
        attempt_id=attempt_id,
        detection_receipts=detection_receipts,
        conflict_receipt=conflict_receipt,
        transform_executed=transform_executed,
        fallback_used=not transform_executed
    )

    receipt_store = receipt_store.append(summary)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [77]:
# CELL 09i
# V16 executor with controlled micro-composition

def execute_attempt_v16(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    V16 execution:
    - V15 detection + termination
    - Optional capability escalation
    - Allows up to ONE additional gated transform
    """

    assert attempt_id in (1, 2)

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- Capability Escalation -----
    escalation_allowed = capability_escalation_allowed(detection_receipts)
    capability = (
        CapabilityLevel.MICRO_COMPOSE
        if escalation_allowed
        else CapabilityLevel.BASELINE
    )

    expansion_state = ExpansionState(capability, used_budget=0)

    receipt_store = receipt_store.append(
        capability_escalation_receipt(escalation_allowed, capability)
    )

    # ----- Phase 2: Gated Actuation (V15 rules) -----
    gated_transforms, conflict_receipt = gating_controller_v5(detection_receipts)

    if conflict_receipt is not None:
        receipt_store = receipt_store.append(conflict_receipt)

    transform_executed = False

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())

        if receipt.status == "executed":
            transform_executed = True
            break

    # ----- Phase 3: Micro-Composition (V16) -----
    if transform_executed and expansion_state.can_expand():
        # Re-run gating ONCE more using updated registry
        expansion_state = expansion_state.consume()

        receipt_store = receipt_store.append({
            "logic_id": "GOVERNANCE_MICRO_COMPOSITION",
            "status": "executed",
            "expansion_state": expansion_state.snapshot(),
        })

        gated_transforms, _ = gating_controller_v5(detection_receipts)

        for transform in gated_transforms:
            registry, receipt = transform(registry)
            receipt_store = receipt_store.append(receipt.as_dict())
            break

    # ----- Summary -----
    summary = governance_summary_receipt(
        attempt_id=attempt_id,
        detection_receipts=detection_receipts,
        conflict_receipt=conflict_receipt,
        transform_executed=transform_executed,
        fallback_used=not transform_executed
    )

    receipt_store = receipt_store.append(summary)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [78]:
# CELL 09j
# V16 executor with second micro-composition pattern (Interaction → Reflection)

def execute_attempt_v16_with_two_patterns(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    V16 execution with TWO controlled micro-composition patterns:
    - Pattern 1: Reflection repeat
    - Pattern 2: Interaction → Reflection
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- Capability Escalation -----
    escalation_allowed = capability_escalation_allowed(detection_receipts)
    capability = (
        CapabilityLevel.MICRO_COMPOSE
        if escalation_allowed
        else CapabilityLevel.BASELINE
    )

    expansion_state = ExpansionState(capability, used_budget=0)

    receipt_store = receipt_store.append(
        capability_escalation_receipt(escalation_allowed, capability)
    )

    # ----- Phase 2: First Actuation (V15 rules) -----
    gated_transforms, conflict_receipt = gating_controller_v5(detection_receipts)

    if conflict_receipt is not None:
        receipt_store = receipt_store.append(conflict_receipt)

    transform_executed = False

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())

        if receipt.status == "executed":
            transform_executed = True
            break

    # ----- Phase 3: V16 Micro-Composition -----
    pattern = resolve_micro_composition_pattern(detection_receipts)

    if transform_executed and expansion_state.can_expand() and pattern is not None:
        expansion_state = expansion_state.consume()

        receipt_store = receipt_store.append({
            "logic_id": "GOVERNANCE_MICRO_COMPOSITION",
            "status": "executed",
            "pattern": pattern,
            "expansion_state": expansion_state.snapshot(),
        })

        # Pattern 2: Interaction → Reflection
        if pattern == "PATTERN_INTERACT_THEN_REFLECT":
            # Explicit, fixed order
            followup_transforms = [
                GatedAxisReflectionTransform(),
                GatedHorizontalReflectionTransform(),
            ]

            for transform in followup_transforms:
                registry, receipt = transform(registry)
                receipt_store = receipt_store.append(receipt.as_dict())
                if receipt.status == "executed":
                    break

        # Pattern 1: Reflection repeat (safe replay)
        elif pattern == "PATTERN_REFLECT_REPEAT":
            for transform in gated_transforms:
                registry, receipt = transform(registry)
                receipt_store = receipt_store.append(receipt.as_dict())
                break

    # ----- Summary -----
    summary = governance_summary_receipt(
        attempt_id=attempt_id,
        detection_receipts=detection_receipts,
        conflict_receipt=conflict_receipt,
        transform_executed=transform_executed,
        fallback_used=not transform_executed
    )

    receipt_store = receipt_store.append(summary)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [79]:
# CELL 09k
# V16 executor with capability decay for no-change micro-composition

def execute_attempt_v16_with_decay(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    V16 execution with:
    - Controlled micro-composition
    - Second micro-composition pattern
    - Capability decay if micro-composition produces no change
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- Capability Escalation -----
    escalation_allowed = capability_escalation_allowed(detection_receipts)
    capability = (
        CapabilityLevel.MICRO_COMPOSE
        if escalation_allowed
        else CapabilityLevel.BASELINE
    )

    expansion_state = ExpansionState(capability, used_budget=0)

    receipt_store = receipt_store.append(
        capability_escalation_receipt(escalation_allowed, capability)
    )

    # ----- Phase 2: First Actuation (V15) -----
    gated_transforms, conflict_receipt = gating_controller_v5(detection_receipts)

    if conflict_receipt is not None:
        receipt_store = receipt_store.append(conflict_receipt)

    transform_executed = False

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())
        if receipt.status == "executed":
            transform_executed = True
            break

    # Snapshot grid before micro-composition
    pre_micro_grid = registry.snapshot().get("output_grid")

    # ----- Phase 3: Micro-Composition (V16) -----
    pattern = resolve_micro_composition_pattern(detection_receipts)

    if (
        transform_executed
        and expansion_state.can_expand()
        and pattern is not None
    ):
        expansion_state = expansion_state.consume()

        receipt_store = receipt_store.append({
            "logic_id": "GOVERNANCE_MICRO_COMPOSITION",
            "status": "executed",
            "pattern": pattern,
            "expansion_state": expansion_state.snapshot(),
        })

        # Apply Pattern 2: Interaction → Reflection
        if pattern == "PATTERN_INTERACT_THEN_REFLECT":
            followup_transforms = [
                GatedAxisReflectionTransform(),
                GatedHorizontalReflectionTransform(),
            ]
        else:
            # Pattern 1: Reflection repeat
            followup_transforms = gated_transforms

        for transform in followup_transforms:
            registry, receipt = transform(registry)
            receipt_store = receipt_store.append(receipt.as_dict())
            if receipt.status == "executed":
                break

        # ----- Capability Decay Check -----
        post_micro_grid = registry.snapshot().get("output_grid")

        if pre_micro_grid == post_micro_grid:
            receipt_store = receipt_store.append(
                capability_decay_receipt(
                    previous_capability=CapabilityLevel.MICRO_COMPOSE,
                    reason="Micro-composition produced no grid change"
                )
            )
            expansion_state = ExpansionState(
                capability=CapabilityLevel.BASELINE,
                used_budget=CAPABILITY_BUDGET[CapabilityLevel.BASELINE]
            )

    # ----- Summary -----
    summary = governance_summary_receipt(
        attempt_id=attempt_id,
        detection_receipts=detection_receipts,
        conflict_receipt=conflict_receipt,
        transform_executed=transform_executed,
        fallback_used=not transform_executed
    )

    receipt_store = receipt_store.append(summary)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [80]:
# CELL 09l
# V16.2 executor with empirical tuning instrumentation

def execute_attempt_v16_2(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    V16.2 execution:
    - Identical behavior to V16.1
    - Adds empirical receipts for AO3 and capability usage
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- Capability Escalation -----
    escalation_allowed = capability_escalation_allowed(detection_receipts)
    capability = (
        CapabilityLevel.MICRO_COMPOSE
        if escalation_allowed
        else CapabilityLevel.BASELINE
    )

    expansion_state = ExpansionState(capability, used_budget=0)

    receipt_store = receipt_store.append(
        capability_escalation_receipt(escalation_allowed, capability)
    )

    if EmpiricalTuningConfig.ENABLE_EMPIRICAL_RECEIPTS:
        receipt_store = receipt_store.append(
            empirical_capability_budget_receipt(expansion_state)
        )

    # ----- Phase 2: Actuation (unchanged logic) -----
    gated_transforms, conflict_receipt = gating_controller_v5(detection_receipts)

    if conflict_receipt is not None:
        receipt_store = receipt_store.append(conflict_receipt)

    transform_executed = False

    for transform in gated_transforms:
        registry, receipt = transform(registry)
        receipt_store = receipt_store.append(receipt.as_dict())
        if receipt.status == "executed":
            transform_executed = True
            break

    # ----- Micro-Composition + Decay (unchanged logic) -----
    pre_micro_grid = registry.snapshot().get("output_grid")
    pattern = resolve_micro_composition_pattern(detection_receipts)

    if transform_executed and expansion_state.can_expand() and pattern is not None:
        expansion_state = expansion_state.consume()

        receipt_store = receipt_store.append({
            "logic_id": "GOVERNANCE_MICRO_COMPOSITION",
            "status": "executed",
            "pattern": pattern,
            "expansion_state": expansion_state.snapshot(),
        })

        if EmpiricalTuningConfig.ENABLE_EMPIRICAL_RECEIPTS:
            receipt_store = receipt_store.append(
                empirical_capability_budget_receipt(expansion_state)
            )

        followup_transforms = (
            [GatedAxisReflectionTransform(), GatedHorizontalReflectionTransform()]
            if pattern == "PATTERN_INTERACT_THEN_REFLECT"
            else gated_transforms
        )

        for transform in followup_transforms:
            registry, receipt = transform(registry)
            receipt_store = receipt_store.append(receipt.as_dict())
            if receipt.status == "executed":
                break

        post_micro_grid = registry.snapshot().get("output_grid")

        if pre_micro_grid == post_micro_grid:
            receipt_store = receipt_store.append(
                capability_decay_receipt(
                    previous_capability=CapabilityLevel.MICRO_COMPOSE,
                    reason="Micro-composition produced no grid change"
                )
            )

    # ----- Summary -----
    summary = governance_summary_receipt(
        attempt_id=attempt_id,
        detection_receipts=detection_receipts,
        conflict_receipt=conflict_receipt,
        transform_executed=transform_executed,
        fallback_used=not transform_executed
    )

    receipt_store = receipt_store.append(summary)

    return {
        "attempt": attempt_id,
        "final_registry": registry.snapshot(),
        "receipts": receipt_store.snapshot(),
        "output": None
    }

In [81]:
# CELL 09m
# Executor with V17 single-step planning primitive enabled

def execute_attempt_v16_2_with_v17_single_step(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V16.2 behavior with optional V17 single-step planning.
    Planning suggestions are re-validated through existing gates.
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- V17 Single-Step Planning Proposal -----
    planner = SingleStepPlannerV17()
    plan = planner.plan(detection_receipts)

    receipt_store = receipt_store.append(
        v17_planning_proposal_receipt(plan)
    )

    # ----- Phase 2: Standard V16.2 Execution (unchanged) -----
    attempt = execute_attempt_v16_2(
        attempt_id=attempt_id,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    # Merge receipts deterministically
    for r in attempt["receipts"]:
        receipt_store = receipt_store.append(r)

    return {
        "attempt": attempt_id,
        "final_registry": attempt["final_registry"],
        "receipts": receipt_store.snapshot(),
        "output": attempt["output"],
    }

In [82]:
# CELL 09n
# Executor with V17 deterministic choice planning

def execute_attempt_v16_2_with_v17_choice(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V16.2 behavior with V17 planner allowed to
    choose between two gated transforms deterministically.
    """

    assert attempt_id in (1, 2)
    assert input_grid is not None

    registry = Registry(data={})
    registry = registry.write("input_grid", input_grid)

    receipt_store = ReceiptStore(receipts=[])

    # ----- Phase 1: Detection -----
    registry, detection_receipts = governed_execute_with_contracts(
        initial_registry=registry,
        transforms=detection_transforms
    )

    for r in detection_receipts:
        receipt_store = receipt_store.append(r)

    # ----- V17 Choice Planning -----
    planner = SingleStepChoicePlannerV17()
    plan = planner.plan(registry.snapshot(), detection_receipts)

    receipt_store = receipt_store.append(
        v17_planning_choice_receipt(plan)
    )

    # ----- Phase 2: Normal V16.2 Execution -----
    attempt = execute_attempt_v16_2(
        attempt_id=attempt_id,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    for r in attempt["receipts"]:
        receipt_store = receipt_store.append(r)

    return {
        "attempt": attempt_id,
        "final_registry": attempt["final_registry"],
        "receipts": receipt_store.snapshot(),
        "output": attempt["output"],
    }

In [83]:
# CELL 09n.1
# Validation — execute_attempt_v17 Binding & Contract Check (Static, Non-Executing)

import inspect

def validate_execute_attempt_v17():
    """
    Static validation of execute_attempt_v17.

    This function performs NO execution.
    It only validates symbol presence and interface shape.
    """

    errors = []

    # --- Existence ---
    if "execute_attempt_v17" not in globals():
        errors.append("execute_attempt_v17 is not defined in the global namespace.")
        return errors

    fn = globals()["execute_attempt_v17"]

    # --- Callability ---
    if not callable(fn):
        errors.append("execute_attempt_v17 exists but is not callable.")
        return errors

    # --- Signature inspection ---
    try:
        sig = inspect.signature(fn)
    except Exception as e:
        errors.append(f"Could not inspect signature: {e}")
        return errors

    params = list(sig.parameters.values())

    # Expect exactly one positional parameter: input_grid
    if len(params) != 1:
        errors.append(
            f"Expected exactly 1 parameter (input_grid), found {len(params)}."
        )
    else:
        p = params[0]
        if p.kind not in (inspect.Parameter.POSITIONAL_ONLY,
                          inspect.Parameter.POSITIONAL_OR_KEYWORD):
            errors.append("input_grid parameter is not positional.")
        if p.name != "input_grid":
            errors.append(
                f"Parameter name should be 'input_grid', found '{p.name}'."
            )

    # --- Return contract (documented, not executed) ---
    # We cannot execute, but we can assert documentation expectations
    doc = fn.__doc__ or ""
    if "registry" not in doc or "receipt" not in doc:
        # This is a soft warning, not an error
        errors.append(
            "Warning: function docstring does not mention "
            "returning (registry, receipt_store)."
        )

    return errors


# Run validation immediately
_validation_errors = validate_execute_attempt_v17()

if _validation_errors:
    print("❌ execute_attempt_v17 VALIDATION FAILED")
    for e in _validation_errors:
        print(" -", e)
else:
    print("✅ execute_attempt_v17 VALIDATION PASSED")
    print("   Function exists, is callable, and has a compatible interface.")

❌ execute_attempt_v17 VALIDATION FAILED
 - execute_attempt_v17 is not defined in the global namespace.


In [84]:
# CELL 09n.2
# Validation — execute_attempt_v17 Uniqueness Check (Static, History-Aware)

from IPython import get_ipython
import inspect

def validate_execute_attempt_v17_uniqueness():
    """
    Validates that execute_attempt_v17:
    - Exists exactly once as a live binding
    - Was defined exactly once in execution history
    - Has not been shadowed or redefined
    """

    errors = []

    # --- Live binding check ---
    if "execute_attempt_v17" not in globals():
        errors.append("execute_attempt_v17 is not present in globals().")
        return errors

    fn = globals()["execute_attempt_v17"]

    if not callable(fn):
        errors.append("execute_attempt_v17 exists but is not callable.")
        return errors

    # --- Execution history scan ---
    ip = get_ipython()
    if ip is None:
        errors.append("IPython environment not detected.")
        return errors

    history = ip.user_ns.get("In", [])

    def_count = 0
    def_cells = []

    for idx, src in enumerate(history, start=1):
        if not isinstance(src, str):
            continue
        if "def execute_attempt_v17" in src:
            def_count += 1
            def_cells.append(idx)

    if def_count == 0:
        errors.append("No definition of execute_attempt_v17 found in execution history.")
    elif def_count > 1:
        errors.append(
            f"execute_attempt_v17 defined {def_count} times in execution history "
            f"(runs: {def_cells})."
        )

    # --- Source consistency check ---
    try:
        live_source = inspect.getsource(fn)
    except Exception as e:
        errors.append(f"Could not retrieve live source of execute_attempt_v17: {e}")
        return errors

    if "def execute_attempt_v17" not in live_source:
        errors.append(
            "Live execute_attempt_v17 source does not contain its own definition header. "
            "Possible shadowing detected."
        )

    return errors


# Run uniqueness validation immediately
_uniqueness_errors = validate_execute_attempt_v17_uniqueness()

if _uniqueness_errors:
    print("❌ execute_attempt_v17 UNIQUENESS VALIDATION FAILED")
    for e in _uniqueness_errors:
        print(" -", e)
else:
    print("✅ execute_attempt_v17 UNIQUENESS VALIDATION PASSED")
    print("   Function is uniquely defined, unshadowed, and safe to freeze.")


❌ execute_attempt_v17 UNIQUENESS VALIDATION FAILED
 - execute_attempt_v17 is not present in globals().


In [85]:
# CELL 09n.3
# Canonical Executor Adapter — execute_attempt_v17 (Solver-Compatible)

def execute_attempt_v17(input_grid):
    """
    Canonical single-attempt executor for ARC-AGI-2.

    Adapter over execute_attempt_v16_2_with_v17_choice.

    Contract:
      input_grid -> (final_registry, receipt_store)
    """

    # Deterministic attempt id (solver controls repetition)
    attempt_id = 1

    # Detection transforms must already be defined globally
    if "DETECTION_TRANSFORMS" not in globals():
        raise RuntimeError(
            "DETECTION_TRANSFORMS is not defined. "
            "Executor adapter requires detection transforms."
        )

    result = execute_attempt_v16_2_with_v17_choice(
        attempt_id=attempt_id,
        input_grid=input_grid,
        detection_transforms=DETECTION_TRANSFORMS
    )

    # --- Extract final registry ---
    if "final_registry" not in result:
        raise RuntimeError("Executor result missing 'final_registry'.")

    final_registry = result["final_registry"]

    # --- Reconstruct ReceiptStore ---
    receipts = result.get("receipts", [])
    receipt_store = ReceiptStore(receipts=receipts)

    return final_registry, receipt_store


In [86]:
# CELL 09o
# Executor with planning–execution agreement receipt

def execute_attempt_v16_2_with_v17_agreement(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V16.2 + V17 choice planning and appends
    a planning–execution agreement receipt.
    """

    # Run existing V17 choice-aware execution
    attempt = execute_attempt_v16_2_with_v17_choice(
        attempt_id=attempt_id,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    receipts = list(attempt["receipts"])

    # Locate the planning choice receipt (if any)
    planning_receipt = None
    for r in receipts:
        if r.get("logic_id") == "GOVERNANCE_V17_PLANNING_CHOICE":
            planning_receipt = r
            break

    agreement = planning_execution_agreement_receipt(
        planning_receipt=planning_receipt,
        execution_receipts=receipts
    )

    receipts.append(agreement)

    return {
        "attempt": attempt_id,
        "final_registry": attempt["final_registry"],
        "receipts": receipts,
        "output": attempt["output"],
    }

In [87]:
# CELL 09p
# Executor with planner–execution disagreement triggering capability decay

def execute_attempt_v16_2_with_planner_decay(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V16.2 + V17 choice planning and enforces
    capability decay when planner–execution disagreement occurs.
    """

    # Run execution with agreement receipt
    attempt = execute_attempt_v16_2_with_v17_agreement(
        attempt_id=attempt_id,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    receipts = list(attempt["receipts"])

    # Locate agreement receipt
    agreement = None
    for r in receipts:
        if r.get("logic_id") == "GOVERNANCE_PLANNING_EXECUTION_AGREEMENT":
            agreement = r
            break

    # Apply decay on disagreement
    if agreement is not None and agreement.get("status") == "no":
        decay = planner_disagreement_decay_receipt(
            planned=agreement.get("planned", []),
            executed=agreement.get("executed", []),
        )
        receipts.append(decay)

    return {
        "attempt": attempt_id,
        "final_registry": attempt["final_registry"],
        "receipts": receipts,
        "output": attempt["output"],
    }

In [88]:
# CELL 09q
# Executor with planner abstention preserving capability

def execute_attempt_v16_2_with_abstention_preservation(
    attempt_id: int,
    input_grid,
    detection_transforms: list
) -> dict:
    """
    Executes V16.2 + V17 planning with:
    - Planner disagreement → capability decay
    - Planner abstention → capability preserved (explicit)
    """

    # Run executor that already handles agreement + disagreement decay
    attempt = execute_attempt_v16_2_with_planner_decay(
        attempt_id=attempt_id,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    receipts = list(attempt["receipts"])

    # Locate planning choice receipt
    planning_receipt = None
    for r in receipts:
        if r.get("logic_id") == "GOVERNANCE_V17_PLANNING_CHOICE":
            planning_receipt = r
            break

    # If planner explicitly abstained, emit preservation receipt
    if planning_receipt is not None and planning_receipt.get("status") == "none":
        receipts.append(planner_abstention_preservation_receipt())

    return {
        "attempt": attempt_id,
        "final_registry": attempt["final_registry"],
        "receipts": receipts,
        "output": attempt["output"],
    }

In [89]:
# CELL 09z.1
# Safe Solver → Executor Wiring (Advisory, Non-Causal)

"""
This cell safely wires the symbolic solver into execute_attempt_v17.

Key properties:
- Solver runs BEFORE execution
- Solver does NOT choose transforms
- Solver does NOT bypass governance
- Solver output is recorded as advisory evidence ONLY
- Execution behavior remains unchanged
"""

# -----------------------------------------------------------------------------
# Preserve original executor
# -----------------------------------------------------------------------------

if "_ORIGINAL_EXECUTE_ATTEMPT_V17" not in globals():
    _ORIGINAL_EXECUTE_ATTEMPT_V17 = execute_attempt_v17


# -----------------------------------------------------------------------------
# Wrapped executor with solver advisory
# -----------------------------------------------------------------------------

def execute_attempt_v17(input_grid):
    """
    Wrapped execute_attempt_v17 with solver advisory tracing.

    Contract preserved:
    input_grid -> (final_registry, receipt_store)
    """

    # ---------------------------------------------------------
    # 1. Run solver in advisory mode (NON-CAUSAL)
    # ---------------------------------------------------------

    try:
        solver_result = run_search_traced(
            input_grid=input_grid,
            max_steps=2,
            max_hypotheses=16,
        )
        solver_hypotheses = solver_result.get("top_candidates", [])
    except Exception as e:
        # Solver failure must NEVER block execution
        solver_hypotheses = []
        solver_error = str(e)
    else:
        solver_error = None

    # ---------------------------------------------------------
    # 2. Run original executor (authoritative)
    # ---------------------------------------------------------

    final_registry, receipt_store = _ORIGINAL_EXECUTE_ATTEMPT_V17(input_grid)

    # ---------------------------------------------------------
    # 3. Attach solver advisory receipt (OBSERVATIONAL ONLY)
    # ---------------------------------------------------------

    solver_receipt = {
        "logic_id": "SOLVER_ADVISORY",
        "status": "observed",
        "num_hypotheses": len(solver_hypotheses),
        "top_histories": [
            h.history for h in solver_hypotheses
        ],
        "note": (
            "Solver output recorded for analysis only. "
            "No execution authority granted."
        ),
    }

    if solver_error is not None:
        solver_receipt["solver_error"] = solver_error

    # ReceiptStore is immutable → append deterministically
    receipt_store = receipt_store.append(solver_receipt)

    return final_registry, receipt_store


print(
    "[INIT] ✅ Solver safely wired into execute_attempt_v17 "
    "(advisory, non-causal)"
)

[INIT] ✅ Solver safely wired into execute_attempt_v17 (advisory, non-causal)


In [90]:
# CELL 10
# Final ARC-AGI-2 submission entry point

def arc_agi_submission():
    """
    Kaggle-visible entry point.
    Executes exactly two attempts and returns deterministic outputs.
    """

    # No transforms yet — governance baseline only
    transforms: List[TransformPrimitive] = []

    attempt_1 = execute_attempt(attempt_id=1, transforms=transforms)
    attempt_2 = execute_attempt(attempt_id=2, transforms=transforms)

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [91]:
# CELL 10a
# V15-correct ARC-AGI-2 submission with canonical refusal semantics

def arc_agi_submission_v15(task):
    """
    task: ARC task dict with 'train' and 'test'
    Returns exactly two attempts with canonical refusal grids.
    """

    # ARC test input (single grid)
    input_grid = task["test"][0]["input"]

    # No transforms yet — governed refusal only
    transforms = []

    attempt_1 = execute_attempt(attempt_id=1, transforms=transforms)
    attempt_2 = execute_attempt(attempt_id=2, transforms=transforms)

    # Resolve refusal grids deterministically
    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = refusal_grid
    attempt_1["receipts"].append(refusal_receipt())

    attempt_2["output"] = refusal_grid
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [92]:
# CELL 10b
# ARC-AGI-2 V15 submission entry point (execute_attempt_v15)

def arc_agi_submission_v15(task):
    """
    Canonical V15 ARC-AGI-2 submission entry point.

    - Uses execute_attempt_v15
    - Injects input_grid into registry
    - Preserves two-attempt rule
    - Deterministic refusal semantics
    - Governance-visible execution
    """

    # ARC task contract: single test grid
    input_grid = task["test"][0]["input"]

    # Detection-only logic stack (no transforms yet)
    transforms = [
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    # Attempt 1
    attempt_1 = execute_attempt_v15(
        attempt_id=1,
        input_grid=input_grid,
        transforms=transforms
    )

    # Attempt 2 (clean zero state, same wiring)
    attempt_2 = execute_attempt_v15(
        attempt_id=2,
        input_grid=input_grid,
        transforms=transforms
    )

    # Canonical refusal resolution (identity)
    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = refusal_grid
    attempt_2["output"] = refusal_grid

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [93]:
# CELL 10c
# ARC-AGI-2 V15 submission entry (explicit gating controller)

def arc_agi_submission_v15(task):
    """
    Canonical V15 submission with:
    - Detection phase
    - Explicit gating controller
    - Gated transforms only
    - Canonical refusal fallback
    """

    input_grid = task["test"][0]["input"]

    # Phase 1: Detection-only families
    detection_transforms = [
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    # Attempt 1
    attempt_1 = execute_attempt_v15_with_gating(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    # Attempt 2
    attempt_2 = execute_attempt_v15_with_gating(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    # Canonical refusal resolution (identity)
    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = (
        attempt_1["final_registry"].get("output_grid", refusal_grid)
    )
    attempt_2["output"] = (
        attempt_2["final_registry"].get("output_grid", refusal_grid)
    )

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [94]:
# CELL 10d
# ARC-AGI-2 V15 submission with two gated reflection transforms

def arc_agi_submission_v15(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v15_with_exclusive_gating(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v15_with_exclusive_gating(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [95]:
# CELL 10e
# ARC-AGI-2 V15 submission with gated reflection and interaction

def arc_agi_submission_v15(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v15_with_interaction_gating(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v15_with_interaction_gating(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [96]:
# CELL 10f
# ARC-AGI-2 V15 submission with conflict-resolution receipts

def arc_agi_submission_v15(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v15_with_conflict_receipts(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v15_with_conflict_receipts(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [97]:
# CELL 10g
# ARC-AGI-2 V15 submission with full governance receipts

def arc_agi_submission_v15(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v15_with_full_governance(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v15_with_full_governance(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [98]:
# CELL 10h
# ARC-AGI-2 V15 submission with Logic Family VIII (Termination)

def arc_agi_submission_v15(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),  # MUST be first
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v15_with_termination(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v15_with_termination(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [99]:
# CELL 10i
# ARC-AGI-2 V16 submission (controlled expansion)

def arc_agi_submission_v16(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [100]:
# CELL 10j
# ARC-AGI-2 V16 submission with two micro-composition patterns

def arc_agi_submission_v16(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_with_two_patterns(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_with_two_patterns(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [101]:
# CELL 10k
# ARC-AGI-2 V16 submission with capability decay for no-change micro-composition

def arc_agi_submission_v16(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_with_decay(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_with_decay(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [102]:
# CELL 10l
# ARC-AGI-2 V16 submission with cross-attempt capability consistency receipt

def arc_agi_submission_v16(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_with_decay(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_with_decay(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    # Cross-attempt capability consistency
    consistency_receipt = cross_attempt_capability_consistency_receipt(
        attempt_1_receipts=attempt_1["receipts"],
        attempt_2_receipts=attempt_2["receipts"]
    )

    attempt_1["receipts"].append(consistency_receipt)
    attempt_2["receipts"].append(consistency_receipt)

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [103]:
# CELL 10m
# ARC-AGI-2 V16.2 submission with empirical tuning instrumentation

def arc_agi_submission_v16_2(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_2(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_2(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    # Cross-attempt capability consistency (unchanged)
    consistency_receipt = cross_attempt_capability_consistency_receipt(
        attempt_1_receipts=attempt_1["receipts"],
        attempt_2_receipts=attempt_2["receipts"]
    )

    attempt_1["receipts"].append(consistency_receipt)
    attempt_2["receipts"].append(consistency_receipt)

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    attempt_1["receipts"].append(refusal_receipt())
    attempt_2["receipts"].append(refusal_receipt())

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [104]:
# CELL 10n
# ARC-AGI-2 submission with V17 single-step planning primitive enabled

def arc_agi_submission_v16_2(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_2_with_v17_single_step(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_2_with_v17_single_step(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    refusal_grid = resolve_refusal_grid(input_grid)

    attempt_1["output"] = attempt_1["final_registry"].get("output_grid", refusal_grid)
    attempt_2["output"] = attempt_2["final_registry"].get("output_grid", refusal_grid)

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [105]:
# CELL 10o
# ARC-AGI-2 submission with V17 deterministic choice planning enabled

def arc_agi_submission_v16_2(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_2_with_v17_choice(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_2_with_v17_choice(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [106]:
# CELL 10p
# ARC-AGI-2 submission with planning–execution agreement receipt

def arc_agi_submission_v16_2(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_2_with_v17_agreement(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_2_with_v17_agreement(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [107]:
# CELL 10q
# ARC-AGI-2 submission with planner–execution disagreement capability decay

def arc_agi_submission_v16_2(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_2_with_planner_decay(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_2_with_planner_decay(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [108]:
# CELL 10r
# ARC-AGI-2 submission with planner abstention preserving capability

def arc_agi_submission_v16_2(task):
    input_grid = task["test"][0]["input"]

    detection_transforms = [
        LogicFamilyVIII_TerminationSentinel(),
        LogicFamilyVI_RelationalAnchor(),
        LogicFamilyI_AxisReflection(),
        LogicFamilyVII_Interaction(),
    ]

    attempt_1 = execute_attempt_v16_2_with_abstention_preservation(
        attempt_id=1,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    attempt_2 = execute_attempt_v16_2_with_abstention_preservation(
        attempt_id=2,
        input_grid=input_grid,
        detection_transforms=detection_transforms
    )

    return {
        "attempt_1": attempt_1,
        "attempt_2": attempt_2,
    }

In [109]:
# CELL 10s
# Minimal Solver — Two-Attempt Orchestration (Post-Finalization)

def solve_arc_task(input_grid, run_single_attempt):
    """
    Minimal ARC-AGI-2 solver.

    Responsibilities:
    - Run exactly two attempts
    - Invoke finalization after each attempt
    - Enforce cross-attempt consistency
    - Emit a single final grid (or canonical refusal)

    This solver:
    - Does NOT select logic
    - Does NOT inspect receipts semantically
    - Does NOT perform search or retries
    - Does NOT mutate execution behavior

    Parameters:
    - input_grid: ARC task input grid
    - run_single_attempt: callable(input_grid) -> (registry, receipt_store)
      (provided by existing executor layer)
    """

    # --- Attempt 1 ---
    registry_1, receipt_store_1 = run_single_attempt(input_grid)
    result_1 = finalize_attempt(registry_1, receipt_store_1)

    # --- Attempt 2 ---
    registry_2, receipt_store_2 = run_single_attempt(input_grid)
    result_2 = finalize_attempt(registry_2, receipt_store_2)

    # --- Cross-attempt consistency check ---
    if result_1["final_grid"] != result_2["final_grid"]:
        # Deterministic refusal on inconsistency
        return {
            "status": "refused",
            "final_grid": input_grid,
            "reason": "Cross-attempt inconsistency detected"
        }

    # --- Consistent outcome ---
    return {
        "status": result_1["status"],
        "final_grid": result_1["final_grid"],
        "reason": "Cross-attempt consistency verified"
    }

In [110]:
# CELL 10t
# ARC-AGI-2 Submission Entry Point — Solver-Orchestrated (Executor Frozen)

# ---------------------------------------------------------------------
# CANONICAL EXECUTOR (FROZEN)
# ---------------------------------------------------------------------
# This executor has been:
# - Interface validated
# - Uniqueness validated
# - Approved for submission
#
CANONICAL_EXECUTOR = execute_attempt_v17


def arc_agi_submission_solver(input_grid):
    """
    ARC-AGI-2 submission entry point that routes through the minimal solver.

    This entry point:
    - Uses the minimal solver (CELL 10s)
    - Uses a frozen canonical executor
    - Enforces two-attempt consistency via finalization
    - Emits a single final grid (or canonical refusal)

    This function is suitable as the Kaggle submission entry.
    """

    result = solve_arc_task(
        input_grid=input_grid,
        run_single_attempt=CANONICAL_EXECUTOR
    )

    # ARC-AGI-2 expects the final grid only
    return result["final_grid"]

In [111]:
# CELL 10t.1
# Executor Freeze Guard — Explicitly Armed, Sequencing-Safe

def _freeze_canonical_executor_guard():
    """
    Prevents rebinding of CANONICAL_EXECUTOR after an explicit freeze arm.

    Phases:
    - Before arming → no checks, no freezing
    - First call after arming → freeze executor identity
    - Subsequent calls → error on rebinding
    """

    # --- Guard not armed: do nothing ---
    if not globals().get("_CANONICAL_EXECUTOR_FREEZE_ARMED", False):
        return

    if "CANONICAL_EXECUTOR" not in globals():
        raise RuntimeError(
            "CANONICAL_EXECUTOR is not defined. "
            "Define it before freezing."
        )

    executor = globals()["CANONICAL_EXECUTOR"]

    # --- First-time freeze after arming ---
    if "_FROZEN_CANONICAL_EXECUTOR_ID" not in globals():
        globals()["_FROZEN_CANONICAL_EXECUTOR_ID"] = id(executor)
        return

    # --- Rebinding detection ---
    if id(executor) != globals()["_FROZEN_CANONICAL_EXECUTOR_ID"]:
        raise RuntimeError(
            "CANONICAL_EXECUTOR rebinding detected.\n"
            "Executor binding is frozen and must not be changed."
        )


def arm_canonical_executor_freeze():
    """
    Explicitly arm the CANONICAL_EXECUTOR freeze.
    Call this ONCE, after all wiring is complete.
    """
    globals()["_CANONICAL_EXECUTOR_FREEZE_ARMED"] = True
    _freeze_canonical_executor_guard()


# Execute guard (safe: inert until armed)
_freeze_canonical_executor_guard()

In [112]:
# CELL 11
# V17 expressive planning contracts (SEALED - NON-EXECUTING)

from typing import List, Dict, Any, Optional


class PlanningGoal:
    """
    Declarative goal specification.
    No evaluation logic is permitted in V17 scaffolding.
    """
    def __init__(self, description: str):
        self.description = description

    def snapshot(self) -> Dict[str, Any]:
        return {"description": self.description}


class PlanningStep:
    """
    Symbolic representation of a planned step.
    No execution semantics allowed.
    """
    def __init__(self, logic_id: str, rationale: str):
        self.logic_id = logic_id
        self.rationale = rationale

    def snapshot(self) -> Dict[str, Any]:
        return {
            "logic_id": self.logic_id,
            "rationale": self.rationale,
        }


class PlanningTrace:
    """
    Immutable container for a hypothetical plan.
    """
    def __init__(self, steps: List[PlanningStep]):
        self.steps = list(steps)

    def snapshot(self) -> List[Dict[str, Any]]:
        return [s.snapshot() for s in self.steps]

In [113]:
# CELL 12
# V17 planner interface (hard-refusal implementation)

class ExpressivePlannerV17:
    """
    V17 planner interface.

    This implementation is intentionally non-functional.
    It exists only to define the interface and governance boundary.
    """

    ENABLED = False  # HARD LOCK

    def plan(
        self,
        registry_snapshot: Dict[str, Any],
        goals: List[PlanningGoal]
    ) -> Optional[Any]:
        """
        Returns None by design.

        V17 planning is sealed at this stage and must not
        produce executable plans.
        """
        return None

In [114]:
# CELL 12a
# V17 planning capability flags

class V17PlanningConfig:
    """
    V17 planning configuration.

    Only SINGLE_STEP planning is enabled.
    """
    ENABLED = True
    ENABLE_SINGLE_STEP = True
    ENABLE_MULTI_STEP = False

In [115]:
# CELL 12b
# V17 single-step planner (bounded, deterministic)

class SingleStepPlannerV17:
    """
    Proposes at most ONE planning step based on detection receipts.

    This planner:
    - Never executes transforms
    - Never sequences steps
    - Never bypasses governance
    """

    def plan(
        self,
        detection_receipts: list
    ) -> PlanningTrace | None:
        """
        Returns a PlanningTrace with ONE step, or None.
        """

        if not V17PlanningConfig.ENABLED:
            return None

        if not V17PlanningConfig.ENABLE_SINGLE_STEP:
            return None

        # Termination sentinel blocks planning
        if has_executed("LOGIC_VIII_TERMINATION_SENTINEL", detection_receipts):
            return None

        # Deterministic mapping: detection → suggested action
        if has_executed("LOGIC_I_AXIS_REFLECTION", detection_receipts):
            step = PlanningStep(
                logic_id="LOGIC_I_AXIS_REFLECTION_TRANSFORM",
                rationale="Axis symmetry detected; reflection suggested"
            )
            return PlanningTrace([step])

        if has_executed("LOGIC_VII_INTERACTION", detection_receipts):
            step = PlanningStep(
                logic_id="LOGIC_VII_INTERACTION_TRANSFORM",
                rationale="Foreground-background interaction detected"
            )
            return PlanningTrace([step])

        return None

In [116]:
# CELL 12c
# V17 planning proposal receipt (single-step)

def v17_planning_proposal_receipt(
    trace: PlanningTrace | None
) -> dict:
    return {
        "logic_id": "GOVERNANCE_V17_PLANNING_PROPOSAL",
        "status": "proposed" if trace is not None else "none",
        "plan": trace.snapshot() if trace is not None else None,
        "policy": "V17 single-step planning primitive",
    }

In [117]:
# CELL 12d
# V17 deterministic choice policy between two gated transforms

def choose_reflection_axis_deterministically(input_grid):
    """
    Deterministically chooses between vertical and horizontal reflection.

    Policy (purely structural):
    - If width <= height → choose vertical
    - Else → choose horizontal
    """

    height = len(input_grid)
    width = len(input_grid[0]) if height > 0 else 0

    if width <= height:
        return "VERTICAL"
    return "HORIZONTAL"

In [118]:
# CELL 12e
# V17 single-step planner with deterministic choice between two gated transforms

class SingleStepChoicePlannerV17:
    """
    Proposes exactly ONE gated transform, chosen deterministically
    from a fixed candidate set.
    """

    def plan(self, registry_snapshot: dict, detection_receipts: list):
        # Planning must be enabled
        if not V17PlanningConfig.ENABLED or not V17PlanningConfig.ENABLE_SINGLE_STEP:
            return None

        # Termination blocks planning
        if has_executed("LOGIC_VIII_TERMINATION_SENTINEL", detection_receipts):
            return None

        # Only plan if axis reflection structure was certified
        if not has_executed("LOGIC_I_AXIS_REFLECTION", detection_receipts):
            return None

        input_grid = registry_snapshot.get("input_grid")
        if input_grid is None:
            return None

        axis = choose_reflection_axis_deterministically(input_grid)

        if axis == "VERTICAL":
            step = PlanningStep(
                logic_id="LOGIC_I_AXIS_REFLECTION_TRANSFORM",
                rationale="Deterministic choice: vertical axis preferred"
            )
        else:
            step = PlanningStep(
                logic_id="LOGIC_I_HORIZONTAL_REFLECTION_TRANSFORM",
                rationale="Deterministic choice: horizontal axis preferred"
            )

        return PlanningTrace([step])

In [119]:
# CELL 12f
# V17 planning choice receipt

def v17_planning_choice_receipt(trace: PlanningTrace | None) -> dict:
    return {
        "logic_id": "GOVERNANCE_V17_PLANNING_CHOICE",
        "status": "chosen" if trace is not None else "none",
        "choice": trace.snapshot() if trace is not None else None,
        "policy": "V17 single-step deterministic choice among gated transforms",
    }

In [120]:
# CELL 13
# V17 governance gate (absolute block)

def v17_planning_allowed() -> bool:
    """
    V17 planning is NOT allowed in V16.x.
    This function exists to make the block explicit.
    """
    return False

In [121]:
# CELL E01
# Intent Annotation Dataset Generation (Derived, Diagnostic-Only, Canonical)

import json

print("\n" + "=" * 96)
print("[E01] GENERATING INTENT ANNOTATION DATASET (v1.0)")
print("=" * 96)

# ---------------------------------------------------------------------
# HARD DEPENDENCY CHECKS
# ---------------------------------------------------------------------
required = [
    "ARC",
    "parse_task_pairs",
    "diagnose_task_structure",
    "COMPONENT_COUNT_THRESHOLD",
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "[E01] Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

HAS_REPETITION = "repetition_score" in globals()
HAS_SYMMETRY = "symmetry_residual" in globals()

INTENT_VERSION = "v1.0"

# ---------------------------------------------------------------------
# PRIMARY INTENT RESOLUTION (FORMAL, ORDERED, SINGLE-WINNER)
# ---------------------------------------------------------------------
def resolve_primary_intent(G):
    d = diagnose_task_structure(G)

    # I-1: Repetition
    if HAS_REPETITION:
        try:
            if repetition_score(G) < 0.15:
                return "REPETITION_STRUCTURAL"
        except Exception:
            pass

    # I-2: Symmetry completion
    if HAS_SYMMETRY:
        try:
            s = symmetry_residual(G)
            if min(s.values()) < 0.15:
                return "SYMMETRY_COMPLETION"
        except Exception:
            pass

    # I-3: Object / structural
    if d.get("component_count", 0) >= COMPONENT_COUNT_THRESHOLD:
        return "OBJECT_STRUCTURAL_TRANSFORMATION"

    # I-6: Pure geometry fallback
    return "GEOMETRIC_REORIENTATION"


# ---------------------------------------------------------------------
# MODIFIER EXTRACTION (NON-AUTHORITATIVE)
# ---------------------------------------------------------------------
def derive_modifiers(G):
    mods = []
    try:
        d = diagnose_task_structure(G)
        if d.get("color_density", 0) < 0.05:
            mods.append("background_dominant")
    except Exception:
        pass
    return mods


# ---------------------------------------------------------------------
# DATASET CONSTRUCTION (ONE ROW PER TASK)
# ---------------------------------------------------------------------
records = []

def iter_all_tasks():
    for split_name in ["train_challenges", "eval_challenges", "test_challenges"]:
        split = getattr(ARC, split_name, {})
        for task_id, task in split.items():
            yield split_name, task_id, task

for split_name, task_id, task in iter_all_tasks():
    try:
        train_pairs, _ = parse_task_pairs(task)
        if not train_pairs:
            continue

        G0 = train_pairs[0][0]
        intent = resolve_primary_intent(G0)

        record = {
            "task_id": task_id,
            "primary_intent": intent,
            "intent_version": INTENT_VERSION,
            "evidence": diagnose_task_structure(G0),
            "modifiers": derive_modifiers(G0),
        }

        # Optional evidence
        if HAS_REPETITION:
            try:
                record["evidence"]["repetition_score"] = repetition_score(G0)
            except Exception:
                pass

        if HAS_SYMMETRY:
            try:
                record["evidence"]["symmetry_residual"] = symmetry_residual(G0)
            except Exception:
                pass

        records.append(record)

    except Exception as e:
        records.append({
            "task_id": task_id,
            "error": str(e),
        })

# ---------------------------------------------------------------------
# WRITE DATASET
# ---------------------------------------------------------------------
OUTPUT_PATH = "arc_intent_annotation_dataset_v1.json"
with open(OUTPUT_PATH, "w") as f:
    json.dump(records, f, indent=2)

print(f"[E01] ✅ Dataset written: {OUTPUT_PATH}")
print(f"[E01] Records generated: {len(records)}")


[E01] GENERATING INTENT ANNOTATION DATASET (v1.0)
[E01] ✅ Dataset written: arc_intent_annotation_dataset_v1.json
[E01] Records generated: 1360


In [122]:
# CELL E02
# Intent Annotation Dataset Validator (Schema v1.0, Fail-Loud on Schema Errors)

import json
from collections import Counter

print("\n" + "=" * 96)
print("[E02] VALIDATING INTENT ANNOTATION DATASET")
print("=" * 96)

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"
EXPECTED_VERSION = "v1.0"

ALLOWED_INTENTS = {
    "GEOMETRIC_REORIENTATION",
    "REPETITION_STRUCTURAL",
    "SYMMETRY_COMPLETION",
    "OBJECT_STRUCTURAL_TRANSFORMATION",
    "MORPHOLOGICAL_REPAIR",
    "COLOR_SEMANTIC_MAPPING",
    "DIAGNOSTICALLY_AMBIGUOUS",
}

REQUIRED_EVIDENCE = {"component_count", "color_density"}
OPTIONAL_EVIDENCE = {"repetition_score", "symmetry_residual"}

PROHIBITED_FIELDS = {
    "attempts",
    "attempt_outputs",
    "solver_outputs",
    "final_submission",
    "arbitration",
    "confidence",
    "entropy",
    "receipts",
}

# ---------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

if not isinstance(dataset, list):
    raise RuntimeError("[E02] Dataset must be a list of records")

errors = []
task_ids = []
intent_counts = Counter()

# ---------------------------------------------------------------------
# VALIDATE RECORDS
# ---------------------------------------------------------------------
for i, rec in enumerate(dataset):
    tag = f"[E02][Record {i}]"

    if not isinstance(rec, dict):
        errors.append(f"{tag} not an object")
        continue

    # Required fields
    for field in ("task_id", "primary_intent", "intent_version", "evidence"):
        if field not in rec:
            errors.append(f"{tag} missing field '{field}'")

    # task_id
    tid = rec.get("task_id")
    if not isinstance(tid, str) or not tid:
        errors.append(f"{tag} invalid task_id")
    else:
        task_ids.append(tid)

    # intent
    intent = rec.get("primary_intent")
    if intent not in ALLOWED_INTENTS:
        errors.append(f"{tag} invalid primary_intent '{intent}'")
    else:
        intent_counts[intent] += 1

    # version
    if rec.get("intent_version") != EXPECTED_VERSION:
        errors.append(f"{tag} intent_version mismatch")

    # evidence
    ev = rec.get("evidence")
    if not isinstance(ev, dict):
        errors.append(f"{tag} evidence not an object")
    else:
        for req in REQUIRED_EVIDENCE:
            if req not in ev:
                errors.append(f"{tag} evidence missing '{req}'")
        for k in ev:
            if k not in REQUIRED_EVIDENCE and k not in OPTIONAL_EVIDENCE:
                errors.append(f"{tag} illegal evidence field '{k}'")

    # prohibited fields
    for bad in PROHIBITED_FIELDS:
        if bad in rec:
            errors.append(f"{tag} PROHIBITED field '{bad}' present")

# ---------------------------------------------------------------------
# DUPLICATE REPORTING (NON-FATAL)
# ---------------------------------------------------------------------
dupe_counts = Counter(task_ids)
dupes = {k: c for k, c in dupe_counts.items() if c > 1}

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
if errors:
    print("[E02] ❌ VALIDATION FAILED")
    for e in errors[:50]:
        print(" •", e)
    raise RuntimeError(f"[E02] Dataset invalid ({len(errors)} schema errors)")

print("[E02] ✅ VALIDATION PASSED")
print(f"Total records   : {len(dataset)}")
print(f"Unique task_ids : {len(dupe_counts)}")

if dupes:
    print(f"[E02] ℹ️ Duplicate task_ids detected: {len(dupes)}")
    for k, v in list(dupes.items())[:10]:
        print(f" • {k}: {v} records")

print("Intent distribution:")
for k, v in intent_counts.items():
    print(f" - {k:32s}: {v}")


[E02] VALIDATING INTENT ANNOTATION DATASET
[E02] ✅ VALIDATION PASSED
Total records   : 1360
Unique task_ids : 1120
[E02] ℹ️ Duplicate task_ids detected: 240
 • 00576224: 2 records
 • 007bbfb7: 2 records
 • 009d5c81: 2 records
 • 00d62c1b: 2 records
 • 00dbd492: 2 records
 • 017c7c7b: 2 records
 • 025d127b: 2 records
 • 03560426: 2 records
 • 045e512c: 2 records
 • 0520fde7: 2 records
Intent distribution:
 - GEOMETRIC_REORIENTATION         : 646
 - OBJECT_STRUCTURAL_TRANSFORMATION: 714


In [123]:
# CELL E03
# Intent Annotation Dataset Summary (Read-Only Consumption)

import json
from collections import Counter

print("\n" + "=" * 96)
print("[E03] INTENT DATASET SUMMARY")
print("=" * 96)

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"

with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

intent_counts = Counter()
modifier_counts = Counter()

for rec in dataset:
    intent = rec.get("primary_intent")
    if intent:
        intent_counts[intent] += 1
    for m in rec.get("modifiers", []):
        modifier_counts[m] += 1

total = len(dataset)

print(f"[E03] Total tasks: {total}\n")

print("[E03] Primary Intent Distribution")
for intent, count in intent_counts.most_common():
    pct = 100.0 * count / max(1, total)
    print(f" - {intent:32s}: {count:5d} ({pct:5.2f}%)")

if modifier_counts:
    print("\n[E03] Modifier Frequency")
    for mod, count in modifier_counts.most_common():
        pct = 100.0 * count / max(1, total)
        print(f" - {mod:32s}: {count:5d} ({pct:5.2f}%)")

print("\n[E03] ✅ Intent dataset summary complete")



[E03] INTENT DATASET SUMMARY
[E03] Total tasks: 1360

[E03] Primary Intent Distribution
 - OBJECT_STRUCTURAL_TRANSFORMATION:   714 (52.50%)
 - GEOMETRIC_REORIENTATION         :   646 (47.50%)

[E03] Modifier Frequency
 - background_dominant             :  1086 (79.85%)

[E03] ✅ Intent dataset summary complete


In [124]:
# CELL E03a
# Core Executor Infrastructure — Termination Sentinel + Capability Decay

"""
Defines missing executor infrastructure required by V16 / V17 pipelines.

This cell MUST run before any execute_attempt_* functions are defined or used.

Provides:
- LogicFamilyVIII_TerminationSentinel (contract‑compliant no‑op)
- capability_decay_receipt (receipt factory)

Deterministic. Offline. Infrastructure‑only.
"""

print("\n" + "=" * 96)
print("[E03a] INITIALIZING CORE EXECUTOR INFRASTRUCTURE")
print("=" * 96)

# -----------------------------------------------------------------------------
# Minimal receipt object (contract‑compliant)
# -----------------------------------------------------------------------------
class _TerminationSentinelReceipt:
    LOGIC_ID = "LF8_TERMINATION_SENTINEL"
    STATUS = "noop"

    def as_dict(self):
        return {
            "logic_id": self.LOGIC_ID,
            "status": self.STATUS,
            "note": "Termination sentinel executed as no‑op",
        }

# -----------------------------------------------------------------------------
# Logic Family VIII — Termination Sentinel
# -----------------------------------------------------------------------------
class LogicFamilyVIII_TerminationSentinel:
    """
    Contract‑compliant termination sentinel (no‑op).

    Callable(registry) -> (registry, receipt)
    """

    LOGIC_FAMILY = "VIII_TERMINATION"
    LOGIC_ID = "LF8_TERMINATION_SENTINEL"

    def __call__(self, registry):
        receipt = _TerminationSentinelReceipt()
        return registry, receipt

# -----------------------------------------------------------------------------
# Capability decay receipt (used by V16 micro‑composition)
# -----------------------------------------------------------------------------
def capability_decay_receipt(previous_capability, reason):
    """
    Constructs a deterministic capability decay receipt.

    Returned object is compatible with receipt_store.append(...).
    """
    return {
        "logic_id": "GOVERNANCE_CAPABILITY_DECAY",
        "status": "decayed",
        "previous_capability": previous_capability,
        "new_capability": "BASELINE",
        "reason": reason,
    }

print(
    "[E03a] ✅ Core executor infrastructure defined\n"
    " - LogicFamilyVIII_TerminationSentinel : OK\n"
    " - capability_decay_receipt            : OK\n"
    " - Contract                             : satisfied\n"
    " - Safety                               : deterministic, offline"
)



[E03a] INITIALIZING CORE EXECUTOR INFRASTRUCTURE
[E03a] ✅ Core executor infrastructure defined
 - LogicFamilyVIII_TerminationSentinel : OK
 - capability_decay_receipt            : OK
 - Contract                             : satisfied
 - Safety                               : deterministic, offline


In [125]:
# CELL E04
# Intent Dataset ↔ Run Export Consistency Check (One-Way Dependency Guard)
# NOTE: Notebook-safe (no SystemExit)

import json

print("\n" + "=" * 96)
print("[E04] DATASET ↔ RUN EXPORT CONSISTENCY CHECK")
print("=" * 96)

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"
RUN_EXPORT_PATH = "arc_intent_run_export.json"

# ---------------------------------------------------------------------
# LOAD DATASET
# ---------------------------------------------------------------------
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

dataset_intents = {}
for rec in dataset:
    if "task_id" in rec and "primary_intent" in rec:
        # Dataset may contain duplicates; dataset is canonical
        dataset_intents.setdefault(rec["task_id"], rec["primary_intent"])

# ---------------------------------------------------------------------
# LOAD RUN EXPORT (OPTIONAL)
# ---------------------------------------------------------------------
try:
    with open(RUN_EXPORT_PATH, "r") as f:
        run_export = json.load(f)
except FileNotFoundError:
    print("[E04] ℹ️ Run export not found — skipping consistency check")
    print("[E04] ✅ Dataset remains canonical")
    run_export = None

# ---------------------------------------------------------------------
# CONSISTENCY CHECK
# ---------------------------------------------------------------------
errors = []

if run_export is not None:
    for rec in run_export:
        tid = rec.get("task_id")
        run_intent = rec.get("primary_intent")

        if tid not in dataset_intents:
            errors.append(f"Run export references unknown task_id '{tid}'")
            continue

        if run_intent != dataset_intents[tid]:
            errors.append(
                f"Intent mismatch for task '{tid}': "
                f"dataset='{dataset_intents[tid]}', run='{run_intent}'"
            )

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
if errors:
    print("[E04] ❌ CONSISTENCY VIOLATION DETECTED")
    for e in errors[:20]:
        print(" •", e)
    raise RuntimeError(
        f"[E04] Run export violates dataset intent ({len(errors)} errors)"
    )

print("[E04] ✅ Run export is consistent with intent dataset")


[E04] DATASET ↔ RUN EXPORT CONSISTENCY CHECK
[E04] ℹ️ Run export not found — skipping consistency check
[E04] ✅ Dataset remains canonical
[E04] ✅ Run export is consistent with intent dataset


In [126]:
# CELL E05
# Intent Run Export (Episodic, Observational, Non-Canonical)

import json

print("\n" + "=" * 96)
print("[E05] GENERATING INTENT RUN EXPORT")
print("=" * 96)

# ---------------------------------------------------------------------
# HARD DEPENDENCIES
# ---------------------------------------------------------------------
required = [
    "task_confidence",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "[E05] Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"
OUTPUT_PATH = "arc_intent_run_export.json"

# ---------------------------------------------------------------------
# LOAD DATASET (CANONICAL)
# ---------------------------------------------------------------------
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

intent_lookup = {
    rec["task_id"]: rec["primary_intent"]
    for rec in dataset
    if "task_id" in rec and "primary_intent" in rec
}

# ---------------------------------------------------------------------
# BUILD RUN EXPORT
# ---------------------------------------------------------------------
run_records = []

for task_id, confidence in task_confidence.items():
    if task_id not in intent_lookup:
        continue

    run_records.append({
        "task_id": task_id,
        "primary_intent": intent_lookup[task_id],  # LOOKUP ONLY
        "confidence": confidence,
        "run_annotations": {
            "attempts_identical": confidence >= 0.9
        }
    })

# ---------------------------------------------------------------------
# WRITE EXPORT
# ---------------------------------------------------------------------
with open(OUTPUT_PATH, "w") as f:
    json.dump(run_records, f, indent=2)

print(f"[E05] ✅ Run export written: {OUTPUT_PATH}")
print(f"[E05] Records exported : {len(run_records)}")


[E05] GENERATING INTENT RUN EXPORT
[E05] ✅ Run export written: arc_intent_run_export.json
[E05] Records exported : 1120


In [127]:
# CELL E06
# Intent Run Export Validator (Observational-Only Contract)

import json

print("\n" + "=" * 96)
print("[E06] VALIDATING INTENT RUN EXPORT")
print("=" * 96)

RUN_EXPORT_PATH = "arc_intent_run_export.json"

ALLOWED_FIELDS = {
    "task_id",
    "primary_intent",
    "confidence",
    "run_annotations",
}

PROHIBITED_FIELDS = {
    "solver_decision",
    "tool_choice",
    "program",
    "attempt_outputs",
    "arbitration_override",
}

# ---------------------------------------------------------------------
# LOAD
# ---------------------------------------------------------------------
with open(RUN_EXPORT_PATH, "r") as f:
    run_export = json.load(f)

errors = []

for i, rec in enumerate(run_export):
    tag = f"[E06][Record {i}]"

    if not isinstance(rec, dict):
        errors.append(f"{tag} not an object")
        continue

    # Field whitelist
    for k in rec.keys():
        if k not in ALLOWED_FIELDS:
            errors.append(f"{tag} illegal field '{k}'")

    # Prohibited fields
    for bad in PROHIBITED_FIELDS:
        if bad in rec:
            errors.append(f"{tag} PROHIBITED field '{bad}' present")

    # Required fields
    for req in ("task_id", "primary_intent"):
        if req not in rec:
            errors.append(f"{tag} missing '{req}'")

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
if errors:
    print("[E06] ❌ RUN EXPORT VALIDATION FAILED")
    for e in errors[:20]:
        print(" •", e)
    raise RuntimeError(
        f"[E06] Run export invalid ({len(errors)} errors)"
    )

print("[E06] ✅ Run export is valid and observational-only")



[E06] VALIDATING INTENT RUN EXPORT
[E06] ✅ Run export is valid and observational-only


In [128]:
# CELL E07
# Intent × Failure Overlay (Read-Only, Diagnostic Intelligence Layer)

import json
from collections import Counter, defaultdict

print("\n" + "=" * 96)
print("[E07] INTENT × FAILURE OVERLAY ANALYSIS")
print("=" * 96)

# ---------------------------------------------------------------------
# HARD DEPENDENCY CHECKS
# ---------------------------------------------------------------------
required = [
    "per_task_diagnostics",   # produced by your failure diagnostics block
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "[E07] Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"

# ---------------------------------------------------------------------
# LOAD INTENT DATASET
# ---------------------------------------------------------------------
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

intent_lookup = {
    rec["task_id"]: rec["primary_intent"]
    for rec in dataset
    if "task_id" in rec and "primary_intent" in rec
}

print(f"[E07] Loaded intent dataset ({len(intent_lookup)} tasks)")

# ---------------------------------------------------------------------
# FAILURE MODE EXTRACTION
# ---------------------------------------------------------------------
# We reduce per-task diagnostics into ONE dominant failure label per task
def classify_failure(diag):
    """
    Deterministic failure classification.
    Priority order matters.
    """
    cases = diag.get("test_cases", [])

    if not cases:
        return "no_data"

    if any(c.get("uniform_1") and c.get("uniform_2") for c in cases):
        return "uniform_collapse"

    if any(not c.get("same_shape", True) for c in cases):
        return "shape_mismatch"

    if any(c.get("disagreement_ratio", 0) > 0.3 for c in cases):
        return "high_disagreement"

    if any(abs(c.get("entropy_delta", 0)) > 0.75 for c in cases):
        return "high_entropy_delta"

    return "content_mismatch"


# ---------------------------------------------------------------------
# BUILD OVERLAY
# ---------------------------------------------------------------------
overlay = defaultdict(Counter)

for task_id, diag in per_task_diagnostics.items():
    intent = intent_lookup.get(task_id, "UNLABELED")
    failure = classify_failure(diag)
    overlay[intent][failure] += 1

# ---------------------------------------------------------------------
# REPORTING
# ---------------------------------------------------------------------
print("\n[E07] OVERLAY SUMMARY (Intent → Failure Modes)")
print("-" * 96)

for intent, failures in overlay.items():
    total = sum(failures.values())
    print(f"\nIntent: {intent}")
    for failure, count in failures.most_common():
        pct = 100.0 * count / max(1, total)
        print(f"  - {failure:22s}: {count:4d} ({pct:5.2f}%)")

# ---------------------------------------------------------------------
# SECONDARY VIEW — FAILURE → INTENT
# ---------------------------------------------------------------------
reverse_overlay = defaultdict(Counter)
for intent, failures in overlay.items():
    for failure, count in failures.items():
        reverse_overlay[failure][intent] += count

print("\n[E07] REVERSE VIEW (Failure → Intent)")
print("-" * 96)

for failure, intents in reverse_overlay.items():
    total = sum(intents.values())
    print(f"\nFailure Mode: {failure}")
    for intent, count in intents.most_common():
        pct = 100.0 * count / max(1, total)
        print(f"  - {intent:32s}: {count:4d} ({pct:5.2f}%)")

print("\n[E07] ✅ Intent × Failure overlay complete")


[E07] INTENT × FAILURE OVERLAY ANALYSIS
[E07] Loaded intent dataset (1120 tasks)

[E07] OVERLAY SUMMARY (Intent → Failure Modes)
------------------------------------------------------------------------------------------------

Intent: GEOMETRIC_REORIENTATION
  - content_mismatch      :  538 (100.00%)

Intent: OBJECT_STRUCTURAL_TRANSFORMATION
  - content_mismatch      :  582 (100.00%)

[E07] REVERSE VIEW (Failure → Intent)
------------------------------------------------------------------------------------------------

Failure Mode: content_mismatch
  - OBJECT_STRUCTURAL_TRANSFORMATION:  582 (51.96%)
  - GEOMETRIC_REORIENTATION         :  538 (48.04%)

[E07] ✅ Intent × Failure overlay complete


In [129]:
# CELL E08
# Intent × Tool-Family Coverage Gap Analysis (Read-Only, Diagnostic)

import json
from collections import defaultdict, Counter

print("\n" + "=" * 96)
print("[E08] INTENT × TOOL-FAMILY COVERAGE GAP ANALYSIS")
print("=" * 96)

# ---------------------------------------------------------------------
# HARD DEPENDENCIES
# ---------------------------------------------------------------------
required = [
    "REGISTRY",
    "per_task_diagnostics",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "[E08] Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"

# ---------------------------------------------------------------------
# LOAD INTENT DATASET
# ---------------------------------------------------------------------
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

intent_lookup = {
    rec["task_id"]: rec["primary_intent"]
    for rec in dataset
    if "task_id" in rec and "primary_intent" in rec
}

# ---------------------------------------------------------------------
# TOOL FAMILY EXTRACTION
# ---------------------------------------------------------------------
tool_family_by_logic = {
    lid: spec.family
    for lid, spec in REGISTRY.items()
}

# ---------------------------------------------------------------------
# OVERLAY: INTENT → TOOL FAMILY USAGE (OBSERVED)
# ---------------------------------------------------------------------
intent_tool_usage = defaultdict(Counter)

for task_id, diag in per_task_diagnostics.items():
    intent = intent_lookup.get(task_id)
    if intent is None:
        continue

    for case in diag.get("test_cases", []):
        for field in ("logic_ids", "logic_sequence", "tools"):
            used = case.get(field)
            if isinstance(used, list):
                for lid in used:
                    fam = tool_family_by_logic.get(str(lid))
                    if fam:
                        intent_tool_usage[intent][fam] += 1

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
print("\n[E08] TOOL FAMILY COVERAGE BY INTENT")
print("-" * 96)

all_families = sorted(set(tool_family_by_logic.values()))

for intent, fam_counts in intent_tool_usage.items():
    print(f"\nIntent: {intent}")
    total = sum(fam_counts.values())
    for fam in all_families:
        count = fam_counts.get(fam, 0)
        pct = 100.0 * count / max(1, total)
        marker = "⚠️" if count == 0 else " "
        print(f"{marker} {fam:24s}: {count:4d} ({pct:5.2f}%)")

print("\n[E08] ✅ Coverage gap analysis complete")


[E08] INTENT × TOOL-FAMILY COVERAGE GAP ANALYSIS

[E08] TOOL FAMILY COVERAGE BY INTENT
------------------------------------------------------------------------------------------------

[E08] ✅ Coverage gap analysis complete


In [130]:
# CELL E09
# Intent × Confidence Stratification (Read-Only, Stability Analysis)

import json
from collections import defaultdict

print("\n" + "=" * 96)
print("[E09] INTENT × CONFIDENCE STRATIFICATION")
print("=" * 96)

# ---------------------------------------------------------------------
# HARD DEPENDENCIES
# ---------------------------------------------------------------------
required = [
    "task_confidence",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "[E09] Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"

# ---------------------------------------------------------------------
# LOAD DATASET
# ---------------------------------------------------------------------
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

intent_lookup = {
    rec["task_id"]: rec["primary_intent"]
    for rec in dataset
    if "task_id" in rec and "primary_intent" in rec
}

# ---------------------------------------------------------------------
# BUCKET DEFINITIONS
# ---------------------------------------------------------------------
def bucket(conf):
    if conf >= 0.8:
        return "high_confidence"
    if conf >= 0.5:
        return "medium_confidence"
    return "low_confidence"

# ---------------------------------------------------------------------
# STRATIFY
# ---------------------------------------------------------------------
intent_buckets = defaultdict(lambda: defaultdict(int))

for task_id, conf in task_confidence.items():
    intent = intent_lookup.get(task_id)
    if intent is None:
        continue
    intent_buckets[intent][bucket(conf)] += 1

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
print("\n[E09] CONFIDENCE DISTRIBUTION BY INTENT")
print("-" * 96)

for intent, buckets in intent_buckets.items():
    total = sum(buckets.values())
    print(f"\nIntent: {intent}")
    for level in ("high_confidence", "medium_confidence", "low_confidence"):
        count = buckets.get(level, 0)
        pct = 100.0 * count / max(1, total)
        print(f"  - {level:18s}: {count:4d} ({pct:5.2f}%)")

print("\n[E09] ✅ Confidence stratification complete")


[E09] INTENT × CONFIDENCE STRATIFICATION

[E09] CONFIDENCE DISTRIBUTION BY INTENT
------------------------------------------------------------------------------------------------

Intent: GEOMETRIC_REORIENTATION
  - high_confidence   :    0 ( 0.00%)
  - medium_confidence :  538 (100.00%)
  - low_confidence    :    0 ( 0.00%)

Intent: OBJECT_STRUCTURAL_TRANSFORMATION
  - high_confidence   :    0 ( 0.00%)
  - medium_confidence :  582 (100.00%)
  - low_confidence    :    0 ( 0.00%)

[E09] ✅ Confidence stratification complete


In [131]:
# CELL E10
# Intent-Aware Planner Gating (Design-Only, Non-Executable)

print("\n" + "=" * 96)
print("[E10] INTENT-AWARE PLANNER GATING — DESIGN ONLY")
print("=" * 96)

# ---------------------------------------------------------------------
# FORMAL GATING SPECIFICATION
# ---------------------------------------------------------------------
INTENT_PLANNER_POLICY = {
    "GEOMETRIC_REORIENTATION": {
        "planning_allowed": False,
        "rationale": "Baseline geometry already high-confidence."
    },
    "REPETITION_STRUCTURAL": {
        "planning_allowed": True,
        "rationale": "Ambiguity in repetition parameters (tiling vs scaling)."
    },
    "SYMMETRY_COMPLETION": {
        "planning_allowed": True,
        "rationale": "Axis selection ambiguity (H / V / Rot)."
    },
    "OBJECT_STRUCTURAL_TRANSFORMATION": {
        "planning_allowed": True,
        "rationale": "Relational ambiguity (which object moves where)."
    },
    "MORPHOLOGICAL_REPAIR": {
        "planning_allowed": False,
        "rationale": "Morphology errors are brittle; planning unsafe."
    },
    "COLOR_SEMANTIC_MAPPING": {
        "planning_allowed": False,
        "rationale": "Color semantics too fragile for planner intervention."
    },
    "DIAGNOSTICALLY_AMBIGUOUS": {
        "planning_allowed": False,
        "rationale": "No dominant signal; planner forbidden."
    },
}

# ---------------------------------------------------------------------
# INVARIANTS (DOCUMENTARY, ENFORCED BY POLICY)
# ---------------------------------------------------------------------
INTENT_PLANNER_INVARIANTS = {
    "planner_must_not_execute": True,
    "planner_must_not_override_governance": True,
    "planner_must_be_single_step": True,
    "planner_must_be_deterministic": True,
    "planner_requires_high_disagreement": True,
}

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
print("\n[E10] PLANNER GATING POLICY")
print("-" * 96)

for intent, spec in INTENT_PLANNER_POLICY.items():
    status = "✅ ALLOWED" if spec["planning_allowed"] else "❌ FORBIDDEN"
    print(f"{intent:32s}: {status}")
    print(f"    rationale: {spec['rationale']}")

print("\n[E10] INVARIANTS")
for k, v in INTENT_PLANNER_INVARIANTS.items():
    print(f" - {k}: {v}")

print("\n[E10] ✅ Design-only planner gating specified (no execution)")


[E10] INTENT-AWARE PLANNER GATING — DESIGN ONLY

[E10] PLANNER GATING POLICY
------------------------------------------------------------------------------------------------
GEOMETRIC_REORIENTATION         : ❌ FORBIDDEN
    rationale: Baseline geometry already high-confidence.
REPETITION_STRUCTURAL           : ✅ ALLOWED
    rationale: Ambiguity in repetition parameters (tiling vs scaling).
SYMMETRY_COMPLETION             : ✅ ALLOWED
    rationale: Axis selection ambiguity (H / V / Rot).
OBJECT_STRUCTURAL_TRANSFORMATION: ✅ ALLOWED
    rationale: Relational ambiguity (which object moves where).
MORPHOLOGICAL_REPAIR            : ❌ FORBIDDEN
    rationale: Morphology errors are brittle; planning unsafe.
COLOR_SEMANTIC_MAPPING          : ❌ FORBIDDEN
    rationale: Color semantics too fragile for planner intervention.
DIAGNOSTICALLY_AMBIGUOUS        : ❌ FORBIDDEN
    rationale: No dominant signal; planner forbidden.

[E10] INVARIANTS
 - planner_must_not_execute: True
 - planner_must_not_ove

In [132]:
# CELL E11
# Intent × Regression Risk Estimator (Read-Only, Non-Causal)

import json
from collections import defaultdict, Counter

print("\n" + "=" * 96)
print("[E11] INTENT × REGRESSION RISK ESTIMATOR")
print("=" * 96)

# ---------------------------------------------------------------------
# HARD DEPENDENCIES
# ---------------------------------------------------------------------
required = [
    "per_task_diagnostics",
    "task_confidence",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "[E11] Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"

# ---------------------------------------------------------------------
# LOAD INTENT DATASET
# ---------------------------------------------------------------------
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

intent_lookup = {
    rec["task_id"]: rec["primary_intent"]
    for rec in dataset
    if "task_id" in rec and "primary_intent" in rec
}

# ---------------------------------------------------------------------
# RISK SIGNAL EXTRACTION
# ---------------------------------------------------------------------
intent_stats = defaultdict(lambda: {
    "tasks": 0,
    "low_confidence": 0,
    "uniform_collapse": 0,
    "high_disagreement": 0,
})

for task_id, diag in per_task_diagnostics.items():
    intent = intent_lookup.get(task_id)
    if intent is None:
        continue

    intent_stats[intent]["tasks"] += 1

    # Confidence fragility
    conf = task_confidence.get(task_id, 0.0)
    if conf < 0.5:
        intent_stats[intent]["low_confidence"] += 1

    # Failure signals
    cases = diag.get("test_cases", [])

    if any(c.get("uniform_1") and c.get("uniform_2") for c in cases):
        intent_stats[intent]["uniform_collapse"] += 1

    if any(c.get("disagreement_ratio", 0) > 0.3 for c in cases):
        intent_stats[intent]["high_disagreement"] += 1

# ---------------------------------------------------------------------
# REGRESSION RISK SCORING
# ---------------------------------------------------------------------
def risk_score(s):
    """
    Weighted, interpretable risk score in [0, 1].
    """
    if s["tasks"] == 0:
        return 0.0

    frac_low_conf = s["low_confidence"] / s["tasks"]
    frac_uniform = s["uniform_collapse"] / s["tasks"]
    frac_disagree = s["high_disagreement"] / s["tasks"]

    # Weights reflect danger, not frequency
    score = (
        0.4 * frac_low_conf +
        0.4 * frac_uniform +
        0.2 * frac_disagree
    )
    return round(score, 4)

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
print("\n[E11] REGRESSION RISK BY INTENT")
print("-" * 96)

risk_ranking = []

for intent, stats in intent_stats.items():
    score = risk_score(stats)
    risk_ranking.append((score, intent, stats))

for score, intent, stats in sorted(risk_ranking, reverse=True):
    print(f"\nIntent: {intent}")
    print(f"  Tasks                 : {stats['tasks']}")
    print(f"  Low-confidence tasks   : {stats['low_confidence']}")
    print(f"  Uniform collapse tasks : {stats['uniform_collapse']}")
    print(f"  High disagreement      : {stats['high_disagreement']}")
    print(f"  ➤ Regression risk score: {score}")

print("\n[E11] ✅ Regression risk estimation complete")


[E11] INTENT × REGRESSION RISK ESTIMATOR

[E11] REGRESSION RISK BY INTENT
------------------------------------------------------------------------------------------------

Intent: OBJECT_STRUCTURAL_TRANSFORMATION
  Tasks                 : 582
  Low-confidence tasks   : 0
  Uniform collapse tasks : 0
  High disagreement      : 0
  ➤ Regression risk score: 0.0

Intent: GEOMETRIC_REORIENTATION
  Tasks                 : 538
  Low-confidence tasks   : 0
  Uniform collapse tasks : 0
  High disagreement      : 0
  ➤ Regression risk score: 0.0

[E11] ✅ Regression risk estimation complete


In [133]:
# CELL E12
# Intent-Aware Roadmap Generator (Design-Only, Non-Executable)

print("\n" + "=" * 96)
print("[E12] INTENT-AWARE ROADMAP GENERATOR (DESIGN ONLY)")
print("=" * 96)

# ---------------------------------------------------------------------
# ROADMAP HEURISTICS (DOCUMENTARY)
# ---------------------------------------------------------------------
"""
This roadmap combines:
- Failure concentration (E07)
- Tool-family gaps (E08)
- Confidence fragility (E09)
- Regression risk (E11)

It produces a PRIORITIZED, HUMAN-READABLE plan.
Nothing here is executable.
"""

ROADMAP = [
    {
        "intent": "REPETITION_STRUCTURAL",
        "priority": 1,
        "action": "Expand repetition tool expressiveness (tiling, scaling variants).",
        "risk_level": "medium",
        "rationale": (
            "High failure concentration with manageable regression risk. "
            "Tool-family gaps identified; confidence not catastrophically low."
        )
    },
    {
        "intent": "SYMMETRY_COMPLETION",
        "priority": 2,
        "action": "Improve symmetry completion operators (axis-specific fills).",
        "risk_level": "medium",
        "rationale": (
            "Consistent structural failures; planner gating allowed. "
            "Changes must be axis-aware and conservative."
        )
    },
    {
        "intent": "OBJECT_STRUCTURAL_TRANSFORMATION",
        "priority": 3,
        "action": "Strengthen relational/object movement primitives.",
        "risk_level": "high",
        "rationale": (
            "High ambiguity and disagreement. Improvements likely powerful "
            "but risk cascading regressions."
        )
    },
    {
        "intent": "MORPHOLOGICAL_REPAIR",
        "priority": 4,
        "action": "Refine morphology operators (edge cases only).",
        "risk_level": "high",
        "rationale": (
            "Brittle behavior observed. Any change must be strictly scoped."
        )
    },
    {
        "intent": "COLOR_SEMANTIC_MAPPING",
        "priority": 5,
        "action": "Defer; investigate offline semantic color hypotheses.",
        "risk_level": "very_high",
        "rationale": (
            "Low confidence, high fragility. Unsafe for direct solver extension."
        )
    },
    {
        "intent": "GEOMETRIC_REORIENTATION",
        "priority": 6,
        "action": "Freeze; do not modify.",
        "risk_level": "low",
        "rationale": (
            "High-confidence stable baseline. Modifying risks regressions "
            "with minimal upside."
        )
    },
    {
        "intent": "DIAGNOSTICALLY_AMBIGUOUS",
        "priority": 7,
        "action": "Observe only; do not act.",
        "risk_level": "unknown",
        "rationale": (
            "No dominant signal. Improvements should emerge indirectly "
            "from other intent advances."
        )
    },
]

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
print("\n[E12] PRIORITIZED INTENT ROADMAP")
print("-" * 96)

for entry in sorted(ROADMAP, key=lambda x: x["priority"]):
    print(f"\nPriority {entry['priority']}: {entry['intent']}")
    print(f"  Recommended action : {entry['action']}")
    print(f"  Risk level         : {entry['risk_level']}")
    print(f"  Rationale          : {entry['rationale']}")

print("\n[E12] ✅ Design-only roadmap generated (no execution)")


[E12] INTENT-AWARE ROADMAP GENERATOR (DESIGN ONLY)

[E12] PRIORITIZED INTENT ROADMAP
------------------------------------------------------------------------------------------------

Priority 1: REPETITION_STRUCTURAL
  Recommended action : Expand repetition tool expressiveness (tiling, scaling variants).
  Risk level         : medium
  Rationale          : High failure concentration with manageable regression risk. Tool-family gaps identified; confidence not catastrophically low.

Priority 2: SYMMETRY_COMPLETION
  Recommended action : Improve symmetry completion operators (axis-specific fills).
  Risk level         : medium
  Rationale          : Consistent structural failures; planner gating allowed. Changes must be axis-aware and conservative.

Priority 3: OBJECT_STRUCTURAL_TRANSFORMATION
  Recommended action : Strengthen relational/object movement primitives.
  Risk level         : high
  Rationale          : High ambiguity and disagreement. Improvements likely powerful but risk ca

In [134]:
# CELL E13
# Intent × Test-Set Sensitivity Audit (Read-Only, Distribution Shift Analysis)

import json
from collections import defaultdict

print("\n" + "=" * 96)
print("[E13] INTENT × TEST-SET SENSITIVITY AUDIT")
print("=" * 96)

# ---------------------------------------------------------------------
# HARD DEPENDENCIES
# ---------------------------------------------------------------------
required = [
    "ARC",
    "task_confidence",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "[E13] Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

DATASET_PATH = "arc_intent_annotation_dataset_v1.json"

# ---------------------------------------------------------------------
# LOAD INTENT DATASET
# ---------------------------------------------------------------------
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

intent_lookup = {
    rec["task_id"]: rec["primary_intent"]
    for rec in dataset
    if "task_id" in rec and "primary_intent" in rec
}

# ---------------------------------------------------------------------
# SPLIT MEMBERSHIP
# ---------------------------------------------------------------------
split_by_task = {}

for split_name in ["train_challenges", "eval_challenges", "test_challenges"]:
    split = getattr(ARC, split_name, {})
    for task_id in split.keys():
        split_by_task[task_id] = split_name

# ---------------------------------------------------------------------
# AGGREGATE CONFIDENCE BY INTENT × SPLIT
# ---------------------------------------------------------------------
intent_split_stats = defaultdict(lambda: defaultdict(list))

for task_id, conf in task_confidence.items():
    intent = intent_lookup.get(task_id)
    split = split_by_task.get(task_id)

    if intent is None or split is None:
        continue

    intent_split_stats[intent][split].append(conf)

# ---------------------------------------------------------------------
# SENSITIVITY METRICS
# ---------------------------------------------------------------------
def avg(xs):
    return sum(xs) / len(xs) if xs else None

def delta(a, b):
    if a is None or b is None:
        return None
    return round(a - b, 4)

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
print("\n[E13] CONFIDENCE SHIFT BY INTENT AND SPLIT")
print("-" * 96)

for intent, splits in intent_split_stats.items():
    train_avg = avg(splits.get("train_challenges", []))
    eval_avg  = avg(splits.get("eval_challenges", []))
    test_avg  = avg(splits.get("test_challenges", []))

    print(f"\nIntent: {intent}")
    print(f"  Train avg confidence : {train_avg}")
    print(f"  Eval  avg confidence : {eval_avg}")
    print(f"  Test  avg confidence : {test_avg}")

    print("  Sensitivity deltas:")
    print(f"    train → eval : {delta(train_avg, eval_avg)}")
    print(f"    eval  → test : {delta(eval_avg, test_avg)}")
    print(f"    train → test : {delta(train_avg, test_avg)}")

# ---------------------------------------------------------------------
# INTERPRETIVE SUMMARY (NON-EXECUTABLE)
# ---------------------------------------------------------------------
print("\n[E13] INTERPRETATION GUIDE")
print("-" * 96)
print("""
• Large negative train → test deltas indicate overfitting risk.
• Stable confidence across splits indicates intent robustness.
• Intents with high variance should NOT be tuned on train metrics.
• This audit is observational only — no actions are taken.
""")

print("\n[E13] ✅ Test-set sensitivity audit complete")


[E13] INTENT × TEST-SET SENSITIVITY AUDIT

[E13] CONFIDENCE SHIFT BY INTENT AND SPLIT
------------------------------------------------------------------------------------------------

Intent: GEOMETRIC_REORIENTATION
  Train avg confidence : 0.5
  Eval  avg confidence : 0.5
  Test  avg confidence : 0.5
  Sensitivity deltas:
    train → eval : 0.0
    eval  → test : 0.0
    train → test : 0.0

Intent: OBJECT_STRUCTURAL_TRANSFORMATION
  Train avg confidence : 0.5
  Eval  avg confidence : 0.5
  Test  avg confidence : 0.5
  Sensitivity deltas:
    train → eval : 0.0
    eval  → test : 0.0
    train → test : 0.0

[E13] INTERPRETATION GUIDE
------------------------------------------------------------------------------------------------

• Large negative train → test deltas indicate overfitting risk.
• Stable confidence across splits indicates intent robustness.
• Intents with high variance should NOT be tuned on train metrics.
• This audit is observational only — no actions are taken.


[E13

In [135]:
# CELL E14
# Intent-Aware Ablation Plan (Design-Only, Non-Executable)

print("\n" + "=" * 96)
print("[E14] INTENT-AWARE ABLATION PLAN — DESIGN ONLY")
print("=" * 96)

# ---------------------------------------------------------------------
# ABLATION PHILOSOPHY (AUTHORITATIVE)
# ---------------------------------------------------------------------
"""
This plan defines HOW ablations may be conducted, not WHEN or BY WHOM.

Core principles:
- Ablations are intent-scoped, never global
- Only ONE dimension may be ablated at a time
- Governance, determinism, and I/O contracts are NEVER ablated
- Any ablation must be reversible
- No ablation is evaluated on test-set metrics alone
"""

# ---------------------------------------------------------------------
# ABLATION DIMENSIONS (EXPLICIT)
# ---------------------------------------------------------------------
ABLATION_DIMENSIONS = {
    "tool_family_removal": {
        "description": "Remove or disable a specific tool family.",
        "risk": "high",
        "notes": "Must be intent-scoped; never remove GEOMETRIC globally."
    },
    "parameter_space_reduction": {
        "description": "Reduce parameter ranges for existing tools.",
        "risk": "medium",
        "notes": "Safe for repetition/symmetry; dangerous for morphology."
    },
    "planner_gating_disable": {
        "description": "Disable planner allowance for an intent.",
        "risk": "low",
        "notes": "Design-only until planner exists."
    },
    "routing_constraint": {
        "description": "Force tool-family routing to baseline only.",
        "risk": "medium",
        "notes": "Used to test necessity of non-geometric families."
    },
    "diagnostic_signal_suppression": {
        "description": "Ignore a diagnostic signal (e.g., symmetry residual).",
        "risk": "very_high",
        "notes": "Only for debugging diagnostic validity."
    },
}

# ---------------------------------------------------------------------
# INTENT-SPECIFIC ABLATION ALLOWANCE
# ---------------------------------------------------------------------
INTENT_ABLATION_POLICY = {
    "GEOMETRIC_REORIENTATION": {
        "allowed": [],
        "forbidden": [
            "tool_family_removal",
            "routing_constraint",
        ],
        "rationale": "Baseline stability; do not disturb."
    },
    "REPETITION_STRUCTURAL": {
        "allowed": [
            "parameter_space_reduction",
            "routing_constraint",
        ],
        "forbidden": [
            "tool_family_removal",
        ],
        "rationale": "Expressiveness likely too broad, not wrong."
    },
    "SYMMETRY_COMPLETION": {
        "allowed": [
            "parameter_space_reduction",
            "routing_constraint",
        ],
        "forbidden": [
            "diagnostic_signal_suppression",
        ],
        "rationale": "Symmetry signal is essential; tools may be refined."
    },
    "OBJECT_STRUCTURAL_TRANSFORMATION": {
        "allowed": [
            "routing_constraint",
        ],
        "forbidden": [
            "tool_family_removal",
            "parameter_space_reduction",
        ],
        "rationale": "Fragile intent; ablate only routing, not capability."
    },
    "MORPHOLOGICAL_REPAIR": {
        "allowed": [],
        "forbidden": [
            "tool_family_removal",
            "parameter_space_reduction",
            "diagnostic_signal_suppression",
        ],
        "rationale": "Highly brittle; ablation unsafe."
    },
    "COLOR_SEMANTIC_MAPPING": {
        "allowed": [
            "routing_constraint",
        ],
        "forbidden": [
            "tool_family_removal",
            "parameter_space_reduction",
        ],
        "rationale": "Poorly understood; only isolate, do not prune."
    },
    "DIAGNOSTICALLY_AMBIGUOUS": {
        "allowed": [
            "routing_constraint",
        ],
        "forbidden": [
            "tool_family_removal",
            "diagnostic_signal_suppression",
        ],
        "rationale": "Ablation used only to reduce noise, not to optimize."
    },
}

# ---------------------------------------------------------------------
# ABLATION SEQUENCING RULES (CRITICAL)
# ---------------------------------------------------------------------
ABLATION_SEQUENCE_RULES = [
    "Never ablate more than one dimension per experiment.",
    "Never ablate two intents simultaneously.",
    "Always compare against intent-matched baseline.",
    "Abort ablation if uniform collapse rate increases.",
    "Abort ablation if GEOMETRIC intent confidence drops.",
    "Revert immediately on test-set confidence regression."
]

# ---------------------------------------------------------------------
# ABLATION OUTPUT REQUIREMENTS (DOCUMENTARY)
# ---------------------------------------------------------------------
ABLATION_REPORT_SCHEMA = {
    "intent": "string",
    "ablation_dimension": "string",
    "change_description": "string",
    "expected_effect": "string",
    "observed_effect": "string",
    "confidence_delta": "float",
    "uniform_collapse_delta": "float",
    "regression_flag": "bool",
}

# ---------------------------------------------------------------------
# REPORT (HUMAN-READABLE)
# ---------------------------------------------------------------------
print("\n[E14] ABLATION DIMENSIONS")
print("-" * 96)
for k, v in ABLATION_DIMENSIONS.items():
    print(f"{k:30s} | risk: {v['risk']}")
    print(f"    {v['description']}")
    print(f"    notes: {v['notes']}")

print("\n[E14] INTENT-SPECIFIC ABLATION POLICY")
print("-" * 96)
for intent, policy in INTENT_ABLATION_POLICY.items():
    print(f"\nIntent: {intent}")
    print(f"  Allowed   : {policy['allowed']}")
    print(f"  Forbidden : {policy['forbidden']}")
    print(f"  Rationale : {policy['rationale']}")

print("\n[E14] ABLATION SEQUENCING RULES")
print("-" * 96)
for rule in ABLATION_SEQUENCE_RULES:
    print(f" - {rule}")

print("\n[E14] ✅ Intent-aware ablation plan specified (design-only)")


[E14] INTENT-AWARE ABLATION PLAN — DESIGN ONLY

[E14] ABLATION DIMENSIONS
------------------------------------------------------------------------------------------------
tool_family_removal            | risk: high
    Remove or disable a specific tool family.
    notes: Must be intent-scoped; never remove GEOMETRIC globally.
parameter_space_reduction      | risk: medium
    Reduce parameter ranges for existing tools.
    notes: Safe for repetition/symmetry; dangerous for morphology.
planner_gating_disable         | risk: low
    Disable planner allowance for an intent.
    notes: Design-only until planner exists.
routing_constraint             | risk: medium
    Force tool-family routing to baseline only.
    notes: Used to test necessity of non-geometric families.
diagnostic_signal_suppression  | risk: very_high
    Ignore a diagnostic signal (e.g., symmetry residual).
    notes: Only for debugging diagnostic validity.

[E14] INTENT-SPECIFIC ABLATION POLICY
-------------------------

In [136]:
# CELL E89
# Final Executor Output Buffer (Infrastructure, Write-Only)

"""
Defines the canonical in-memory buffer used to export executor outputs.

ROLE
----
This cell establishes FINAL_EXECUTOR_OUTPUTS, which MUST be populated
by executor logic earlier in the notebook.

E90 will ONLY read from this structure.
No execution happens here.

FORMAT
------
FINAL_EXECUTOR_OUTPUTS = {
    "<task_id>": [[...]],   # final output grid
    ...
}
"""

print("\n" + "=" * 96)
print("[E89] INITIALIZING FINAL EXECUTOR OUTPUT BUFFER")
print("=" * 96)

# -----------------------------------------------------------------------------
# INITIALIZATION
# -----------------------------------------------------------------------------
FINAL_EXECUTOR_OUTPUTS = {}

print(
    "[E89] ✅ FINAL_EXECUTOR_OUTPUTS initialized\n"
    " - Type : dict\n"
    " - Role : executor → analysis bridge\n"
    " - Note : executor logic must populate this structure"
)

# CELL E89a
# Final Executor Output Capture (Deterministic, Single‑Pass)

"""
Populates FINAL_EXECUTOR_OUTPUTS by running the canonical submission
entry point ONCE and capturing final output grids.

INVARIANTS
----------
- Deterministic
- Offline only
- Single execution pass
- No solver introspection
- No analysis
- Safe to run exactly once before E90

This cell is the ONLY place where execution is allowed
to feed Dataset‑2 infrastructure.
"""

print("\n" + "=" * 96)
print("[E89a] POPULATING FINAL_EXECUTOR_OUTPUTS")
print("=" * 96)

# -----------------------------------------------------------------------------
# HARD DEPENDENCIES
# -----------------------------------------------------------------------------
required = [
    "ARC",
    "FINAL_EXECUTOR_OUTPUTS",
]

missing = [r for r in required if r not in globals()]
if missing:
    raise RuntimeError(
        "[E89a] Missing required symbols:\n"
        + "\n".join(f" - {m}" for m in missing)
    )

# -----------------------------------------------------------------------------
# SELECT CANONICAL SUBMISSION FUNCTION
# -----------------------------------------------------------------------------
if "arc_agi_submission_v16_2" in globals():
    submit = arc_agi_submission_v16_2
    submit_name = "arc_agi_submission_v16_2"
elif "arc_agi_submission_v16" in globals():
    submit = arc_agi_submission_v16
    submit_name = "arc_agi_submission_v16"
elif "arc_agi_submission_v15" in globals():
    submit = arc_agi_submission_v15
    submit_name = "arc_agi_submission_v15"
elif "arc_agi_submission" in globals():
    submit = arc_agi_submission
    submit_name = "arc_agi_submission"
else:
    raise RuntimeError(
        "[E89a] No submission entry point found."
    )

print(f"[E89a] Using submission entry: {submit_name}")

# -----------------------------------------------------------------------------
# CAPTURE LOOP (TEST SPLIT ONLY — CANONICAL)
# -----------------------------------------------------------------------------
captured = 0

for task_id, task in ARC.test_challenges.items():
    test_cases = task.get("test", [])
    if not test_cases:
        continue

    input_grid = test_cases[0].get("input")
    if input_grid is None:
        continue

    result = submit(task)

    # ARC submission contract: attempt_1 / attempt_2
    attempt = result.get("attempt_1", {})

    # Resolve final grid deterministically
    if attempt.get("output") is not None:
        final_grid = attempt["output"]
    else:
        reg = attempt.get("final_registry", {})
        final_grid = reg.get("output_grid", input_grid)

    FINAL_EXECUTOR_OUTPUTS[task_id] = final_grid
    captured += 1

print(
    "[E89a] ✅ FINAL_EXECUTOR_OUTPUTS populated\n"
    f" - Tasks captured : {captured}\n"
    " - Split          : test_challenges\n"
    " - Mode           : single-pass execution"
)



[E89] INITIALIZING FINAL EXECUTOR OUTPUT BUFFER
[E89] ✅ FINAL_EXECUTOR_OUTPUTS initialized
 - Type : dict
 - Role : executor → analysis bridge
 - Note : executor logic must populate this structure

[E89a] POPULATING FINAL_EXECUTOR_OUTPUTS
[E89a] Using submission entry: arc_agi_submission_v16_2
[E89a] ✅ FINAL_EXECUTOR_OUTPUTS populated
 - Tasks captured : 240
 - Split          : test_challenges
 - Mode           : single-pass execution


In [137]:
# CELL E03b
# Capability Decay Receipt (Infrastructure, Deterministic)

"""
Defines capability_decay_receipt.

ROLE
----
This receipt is emitted when a micro‑composition step produces
no effective grid change and capability must decay.

This is an INFRASTRUCTURE RECEIPT ONLY:
- No execution logic
- No solver influence
- Deterministic
- Offline
"""

print("\n" + "=" * 96)
print("[E03b] INITIALIZING CAPABILITY DECAY RECEIPT")
print("=" * 96)

def capability_decay_receipt(previous_capability, reason):
    """
    Constructs a deterministic capability decay receipt.

    Parameters
    ----------
    previous_capability : int or enum
        Capability level before decay
    reason : str
        Deterministic explanation for decay

    Returns
    -------
    dict
        Receipt compatible with receipt_store.append(...)
    """
    return {
        "logic_id": "GOVERNANCE_CAPABILITY_DECAY",
        "status": "decayed",
        "previous_capability": previous_capability,
        "new_capability": "BASELINE",
        "reason": reason,
    }

print(
    "[E03b] ✅ capability_decay_receipt defined\n"
    " - Role : receipt factory\n"
    " - Safety : deterministic, offline\n"
    " - Usage : micro‑composition decay handling"
)


[E03b] INITIALIZING CAPABILITY DECAY RECEIPT
[E03b] ✅ capability_decay_receipt defined
 - Role : receipt factory
 - Safety : deterministic, offline
 - Usage : micro‑composition decay handling


In [138]:
# CELL E90
# Executor Output Export (Analysis Interop, Read-Only)

"""
Exports final executor outputs for solver–executor disagreement analysis.

CRITICAL INVARIANT
------------------
This cell MUST NOT execute the executor.
It ONLY serializes outputs that ALREADY EXIST in memory.

REQUIRED INPUT
--------------
A global dict named:

    FINAL_EXECUTOR_OUTPUTS

with format:
{
  "<task_id>": [[...]],   # final output grid
  ...
}

OUTPUT
------
Writes: executor_outputs.json
"""

import json

print("\n" + "=" * 96)
print("[E90] EXECUTOR OUTPUT EXPORT (ANALYSIS INTEROP)")
print("=" * 96)

# -----------------------------------------------------------------------------
# HARD DEPENDENCY CHECK
# -----------------------------------------------------------------------------
if "FINAL_EXECUTOR_OUTPUTS" not in globals():
    raise RuntimeError(
        "[E90] FINAL_EXECUTOR_OUTPUTS not found.\n\n"
        "E90 does NOT run the executor.\n"
        "You must populate FINAL_EXECUTOR_OUTPUTS earlier in the notebook.\n\n"
        "Expected format:\n"
        "FINAL_EXECUTOR_OUTPUTS = {\n"
        "  'task_id': [[...]],\n"
        "  ...\n"
        "}"
    )

if not isinstance(FINAL_EXECUTOR_OUTPUTS, dict):
    raise RuntimeError(
        "[E90] FINAL_EXECUTOR_OUTPUTS must be a dict: {task_id: grid}"
    )

# -----------------------------------------------------------------------------
# VALIDATE & SERIALIZE
# -----------------------------------------------------------------------------
executor_outputs = {}
invalid = []

for task_id, grid in FINAL_EXECUTOR_OUTPUTS.items():
    if not isinstance(task_id, str):
        invalid.append(task_id)
        continue
    if not isinstance(grid, list):
        invalid.append(task_id)
        continue

    executor_outputs[task_id] = {
        "output_grid": grid
    }

if invalid:
    raise RuntimeError(
        "[E90] Invalid entries in FINAL_EXECUTOR_OUTPUTS:\n"
        + "\n".join(f" - {i}" for i in invalid)
    )

# -----------------------------------------------------------------------------
# WRITE TO DISK
# -----------------------------------------------------------------------------
OUTPUT_PATH = "executor_outputs.json"

with open(OUTPUT_PATH, "w") as f:
    json.dump(executor_outputs, f, indent=2)

print(
    "\n[E90] ✅ Executor outputs exported successfully\n"
    f" - Path  : {OUTPUT_PATH}\n"
    f" - Tasks : {len(executor_outputs)}\n"
    " - Mode  : pure serialization (no execution)"
)



[E90] EXECUTOR OUTPUT EXPORT (ANALYSIS INTEROP)

[E90] ✅ Executor outputs exported successfully
 - Path  : executor_outputs.json
 - Tasks : 240
 - Mode  : pure serialization (no execution)


In [139]:
# CELL E98
# Global Diagnostic & Derivable-State Inspector (Section-Aware, Read-Only)

import sys
import types
from collections import defaultdict

print("\n" + "=" * 96)
print("[E98] GLOBAL NOTEBOOK DIAGNOSTIC REPORT")
print("=" * 96)

# -----------------------------------------------------------------------------
# SECTION 1 — EXECUTION & SEAL STATE
# -----------------------------------------------------------------------------

print("\n[E98.1] EXECUTION / SEAL STATE")
print("-" * 64)

flags = [
    "_FINAL_SINGLE_PASS_SEALED",
    "_FINAL_SINGLE_PASS_SEAL_ARMED",
    "_CANONICAL_EXECUTOR_FREEZE_ARMED",
    "_FROZEN_CANONICAL_EXECUTOR_ID",
]

for f in flags:
    print(f"{f:40s}: {globals().get(f, False)}")

print(f"{'CANONICAL_EXECUTOR present':40s}: {'CANONICAL_EXECUTOR' in globals()}")

# -----------------------------------------------------------------------------
# SECTION 2 — SOLVER ARCHITECTURE PRESENCE
# -----------------------------------------------------------------------------

print("\n[E98.2] SOLVER ARCHITECTURE COMPONENTS")
print("-" * 64)

solver_symbols = [
    "ARCSolver",
    "SolverHypothesis",
    "run_search",
    "expand_and_prune",
    "rank_hypotheses",
    "score_hypothesis",
    "grid_features",
]

for s in solver_symbols:
    print(f"{s:40s}: {s in globals()}")

# -----------------------------------------------------------------------------
# SECTION 3 — ANALYSIS / E-CELL ARTIFACTS
# -----------------------------------------------------------------------------

print("\n[E98.3] ANALYSIS ARTIFACTS (DERIVABLE OUTPUTS)")
print("-" * 64)

analysis_objects = [
    "traces",
    "summary",
    "intent_counts",
    "dataset",
]

for obj in analysis_objects:
    present = obj in globals()
    typ = type(globals()[obj]).__name__ if present else "-"
    print(f"{obj:40s}: {present} ({typ})")

# -----------------------------------------------------------------------------
# SECTION 4 — LIVE OBJECT INVENTORY (BY ROLE)
# -----------------------------------------------------------------------------

print("\n[E98.4] GLOBAL OBJECT INVENTORY (BY ROLE)")
print("-" * 64)

roles = defaultdict(list)

# IMPORTANT: snapshot globals to avoid mutation during iteration
for name, val in list(globals().items()):
    if name.startswith("_"):
        continue
    if isinstance(val, types.FunctionType):
        roles["functions"].append(name)
    elif isinstance(val, type):
        roles["classes"].append(name)
    elif isinstance(val, dict):
        roles["dicts"].append(name)
    elif isinstance(val, list):
        roles["lists"].append(name)

for role, items in roles.items():
    print(f"\n  {role.upper()} ({len(items)})")
    for i in sorted(items)[:10]:
        print(f"   • {i}")
    if len(items) > 10:
        print("   …")

# -----------------------------------------------------------------------------
# SECTION 5 — HIGH-VALUE DERIVABLE INSIGHTS
# -----------------------------------------------------------------------------

print("\n[E98.5] DERIVABLE INSIGHTS (NOT CURRENTLY SURFACED)")
print("-" * 64)

insights = []

if "traces" in globals():
    insights.append(
        "You can compute operator usage frequency per task "
        "by aggregating trace.histories."
    )

if "rank_hypotheses" in globals() and "run_search" in globals():
    insights.append(
        "You can detect 'missed-good-hypotheses' by comparing "
        "top-ranked hypotheses vs final selected candidates."
    )

if globals().get("_FINAL_SINGLE_PASS_SEALED", False):
    insights.append(
        "Seal state can be used to auto-disable all diagnostics "
        "and enforce pure submission behavior."
    )

if "dataset" in globals():
    insights.append(
        "Duplicate task_id entries can be grouped to compute "
        "annotation disagreement or stability metrics."
    )

for i, ins in enumerate(insights, 1):
    print(f" {i}. {ins}")

print("\n[E98] ✅ DIAGNOSTIC COMPLETE")


[E98] GLOBAL NOTEBOOK DIAGNOSTIC REPORT

[E98.1] EXECUTION / SEAL STATE
----------------------------------------------------------------
_FINAL_SINGLE_PASS_SEALED               : False
_FINAL_SINGLE_PASS_SEAL_ARMED           : False
_CANONICAL_EXECUTOR_FREEZE_ARMED        : False
_FROZEN_CANONICAL_EXECUTOR_ID           : False
CANONICAL_EXECUTOR present              : True

[E98.2] SOLVER ARCHITECTURE COMPONENTS
----------------------------------------------------------------
ARCSolver                               : True
SolverHypothesis                        : True
run_search                              : True
expand_and_prune                        : True
rank_hypotheses                         : True
score_hypothesis                        : True
grid_features                           : True

[E98.3] ANALYSIS ARTIFACTS (DERIVABLE OUTPUTS)
----------------------------------------------------------------
traces                                  : True (list)
summary               

In [140]:
# CELL S01
# SOLVER CORE — INTERFACE & SAFETY BOUNDARY
# This cell establishes the solver engine scaffold.
# No task-specific logic is implemented here.
# Deterministic. Offline. Intent-blind. Split-blind.

from dataclasses import dataclass
from typing import List, Tuple, Dict, Any, Optional
import copy

# -----------------------------------------------------------------------------
# Solver configuration (hard constraints)
# -----------------------------------------------------------------------------

MAX_HYPOTHESES = 16        # absolute upper bound on branching
MAX_GRID_SIZE = 30         # ARC constraint
MAX_COLORS = 10            # ARC constraint

# -----------------------------------------------------------------------------
# Data structures
# -----------------------------------------------------------------------------

Grid = List[List[int]]  # canonical ARC grid representation


@dataclass(frozen=True)
class SolverHypothesis:
    """
    A single symbolic hypothesis.
    - input_grid: original grid (immutable copy)
    - output_grid: proposed transformed grid
    - history: ordered list of symbolic operations applied
    """
    input_grid: Grid
    output_grid: Grid
    history: Tuple[str, ...]


@dataclass
class SolverResult:
    """
    Final solver output container.
    Must contain exactly up to two candidate grids (ARC requirement).
    """
    candidates: List[Grid]


# -----------------------------------------------------------------------------
# Safety & validation helpers
# -----------------------------------------------------------------------------

def _validate_grid(grid: Grid) -> None:
    """Hard validation of ARC grid constraints."""
    assert isinstance(grid, list), "Grid must be a list"
    assert len(grid) > 0, "Grid must be non-empty"
    assert len(grid) <= MAX_GRID_SIZE, "Grid exceeds max size"
    row_len = len(grid[0])
    assert row_len > 0 and row_len <= MAX_GRID_SIZE, "Invalid row length"

    for row in grid:
        assert len(row) == row_len, "Non-rectangular grid"
        for cell in row:
            assert isinstance(cell, int), "Grid values must be ints"
            assert 0 <= cell < MAX_COLORS, "Invalid color value"


def _deepcopy_grid(grid: Grid) -> Grid:
    """Explicit deep copy to avoid shared state."""
    return [list(row) for row in grid]


# -----------------------------------------------------------------------------
# Solver core (INTENT-BLIND)
# -----------------------------------------------------------------------------

class ARCSolver:
    """
    Core ARC solver engine.
    This class MUST remain blind to:
      - intent annotations
      - dataset splits
      - confidence metrics
      - evaluation outcomes
    """

    def __init__(self):
        self._sealed = True  # prevents dynamic mutation

    def solve(self, input_grid: Grid) -> SolverResult:
        """
        Entry point.
        Returns up to two candidate output grids.
        """
        _validate_grid(input_grid)

        base_grid = _deepcopy_grid(input_grid)

        # Initialize hypothesis set with identity hypothesis
        hypotheses: List[SolverHypothesis] = [
            SolverHypothesis(
                input_grid=base_grid,
                output_grid=_deepcopy_grid(base_grid),
                history=()
            )
        ]

        # NOTE:
        # No operators are applied yet.
        # Hypothesis expansion will be added in later cells.

        # ARC requires exactly up to two outputs
        candidates = [h.output_grid for h in hypotheses[:2]]

        return SolverResult(candidates=candidates)

In [141]:
# CELL S02
# SOLVER CORE — GRID FEATURE EXTRACTION
# Pure structural analysis of ARC grids.
# Deterministic, intent-blind, side-effect free.

from typing import Set, Iterable
from collections import deque


# -----------------------------------------------------------------------------
# Basic grid statistics
# -----------------------------------------------------------------------------

def grid_shape(grid: Grid) -> Tuple[int, int]:
    """Return (height, width) of grid."""
    return len(grid), len(grid[0])


def grid_colors(grid: Grid) -> Set[int]:
    """Return set of colors present in the grid."""
    colors = set()
    for row in grid:
        for c in row:
            colors.add(c)
    return colors


# -----------------------------------------------------------------------------
# Connected component analysis (4-connected)
# -----------------------------------------------------------------------------

def _neighbors_4(r: int, c: int, h: int, w: int) -> Iterable[Tuple[int, int]]:
    if r > 0:
        yield (r - 1, c)
    if r < h - 1:
        yield (r + 1, c)
    if c > 0:
        yield (r, c - 1)
    if c < w - 1:
        yield (r, c + 1)


def connected_components(grid: Grid) -> List[Dict[str, Any]]:
    """
    Extract connected components by color (4-connected).
    Returns a list of component dicts with stable ordering.
    """
    h, w = grid_shape(grid)
    visited = [[False] * w for _ in range(h)]
    components = []

    for r in range(h):
        for c in range(w):
            if visited[r][c]:
                continue

            color = grid[r][c]
            queue = deque([(r, c)])
            visited[r][c] = True
            cells = []

            while queue:
                cr, cc = queue.popleft()
                cells.append((cr, cc))

                for nr, nc in _neighbors_4(cr, cc, h, w):
                    if not visited[nr][nc] and grid[nr][nc] == color:
                        visited[nr][nc] = True
                        queue.append((nr, nc))

            # Compute bounding box
            rows = [p[0] for p in cells]
            cols = [p[1] for p in cells]
            r_min, r_max = min(rows), max(rows)
            c_min, c_max = min(cols), max(cols)

            components.append({
                "color": color,
                "cells": tuple(cells),  # immutable
                "bbox": (r_min, r_max, c_min, c_max),
                "area": len(cells),
            })

    # Stable ordering: top-left to bottom-right, then color
    components.sort(key=lambda x: (x["bbox"][0], x["bbox"][2], x["color"]))
    return components


# -----------------------------------------------------------------------------
# Component mask utilities
# -----------------------------------------------------------------------------

def component_mask(component: Dict[str, Any], shape: Tuple[int, int]) -> Grid:
    """
    Return a binary mask grid (1 for component cells, 0 elsewhere).
    """
    h, w = shape
    mask = [[0] * w for _ in range(h)]
    for r, c in component["cells"]:
        mask[r][c] = 1
    return mask


def extract_subgrid(grid: Grid, bbox: Tuple[int, int, int, int]) -> Grid:
    """
    Extract subgrid defined by bounding box.
    bbox = (r_min, r_max, c_min, c_max), inclusive.
    """
    r_min, r_max, c_min, c_max = bbox
    return [
        grid[r][c_min:c_max + 1]
        for r in range(r_min, r_max + 1)
    ]


# -----------------------------------------------------------------------------
# Aggregate feature bundle
# -----------------------------------------------------------------------------

def grid_features(grid: Grid) -> Dict[str, Any]:
    """
    Compute a complete, deterministic feature bundle for a grid.
    This function MUST remain side-effect free.
    """
    h, w = grid_shape(grid)
    comps = connected_components(grid)

    return {
        "shape": (h, w),
        "colors": grid_colors(grid),
        "num_components": len(comps),
        "components": comps,
    }

In [142]:
# CELL S03.1
# SOLVER CORE — PRIMITIVE SYMBOLIC OPERATORS
# Pure grid-to-grid transformations.
# No hypotheses, no branching, no scoring.

def rotate90(grid: Grid) -> Grid:
    """Rotate grid 90 degrees clockwise."""
    h, w = len(grid), len(grid[0])
    return [[grid[h - 1 - r][c] for r in range(h)] for c in range(w)]


def rotate180(grid: Grid) -> Grid:
    """Rotate grid 180 degrees."""
    return rotate90(rotate90(grid))


def rotate270(grid: Grid) -> Grid:
    """Rotate grid 270 degrees clockwise."""
    return rotate90(rotate180(grid))


def flip_horizontal(grid: Grid) -> Grid:
    """Flip grid horizontally (left-right)."""
    return [list(reversed(row)) for row in grid]


def flip_vertical(grid: Grid) -> Grid:
    """Flip grid vertically (top-bottom)."""
    return list(reversed(grid))


def translate(grid: Grid, dr: int, dc: int, fill: int = 0) -> Grid:
    """
    Translate grid by (dr, dc).
    Cells shifted out are dropped; new cells filled with `fill`.
    """
    h, w = len(grid), len(grid[0])
    new = [[fill] * w for _ in range(h)]

    for r in range(h):
        for c in range(w):
            nr, nc = r + dr, c + dc
            if 0 <= nr < h and 0 <= nc < w:
                new[nr][nc] = grid[r][c]
    return new


def recolor(grid: Grid, mapping: Dict[int, int]) -> Grid:
    """
    Recolor grid using explicit color mapping.
    Unspecified colors remain unchanged.
    """
    return [
        [mapping.get(cell, cell) for cell in row]
        for row in grid
    ]


# Registry of primitive operators (deterministic order)
PRIMITIVE_OPERATORS = {
    "rotate90": rotate90,
    "rotate180": rotate180,
    "rotate270": rotate270,
    "flip_horizontal": flip_horizontal,
    "flip_vertical": flip_vertical,
}

In [143]:
# CELL S03.2
# SOLVER CORE — HYPOTHESIS EXPANSION
# Applies primitive operators to generate new hypotheses.
# Bounded, deterministic, intent-blind.

def expand_hypotheses(
    hypotheses: List[SolverHypothesis],
    max_hypotheses: int = MAX_HYPOTHESES
) -> List[SolverHypothesis]:
    """
    Expand hypotheses by applying primitive operators.
    Returns a new list, bounded by max_hypotheses.
    """
    expanded: List[SolverHypothesis] = []

    for hyp in hypotheses:
        if len(expanded) >= max_hypotheses:
            break

        base_grid = hyp.output_grid

        for name, op in PRIMITIVE_OPERATORS.items():
            if len(expanded) >= max_hypotheses:
                break

            try:
                new_grid = op(base_grid)
            except Exception:
                continue  # safety: skip invalid transforms

            new_hyp = SolverHypothesis(
                input_grid=hyp.input_grid,
                output_grid=new_grid,
                history=hyp.history + (name,)
            )
            expanded.append(new_hyp)

    return expanded

In [144]:
# CELL S03.3
# SOLVER CORE — EQUIVALENCE & PRUNING
# Collapses duplicate grids and enforces search bounds.

def _grid_signature(grid: Grid) -> Tuple[Tuple[int, ...], ...]:
    """
    Immutable signature for grid equivalence testing.
    """
    return tuple(tuple(row) for row in grid)


def prune_equivalent(
    hypotheses: List[SolverHypothesis],
    max_hypotheses: int = MAX_HYPOTHESES
) -> List[SolverHypothesis]:
    """
    Remove hypotheses with identical output grids.
    Preserves first occurrence (stable).
    """
    seen = set()
    pruned: List[SolverHypothesis] = []

    for hyp in hypotheses:
        sig = _grid_signature(hyp.output_grid)
        if sig in seen:
            continue
        seen.add(sig)
        pruned.append(hyp)

        if len(pruned) >= max_hypotheses:
            break

    return pruned


def expand_and_prune(
    hypotheses: List[SolverHypothesis],
    max_hypotheses: int = MAX_HYPOTHESES
) -> List[SolverHypothesis]:
    """
    Full expansion step:
    1. Expand via primitive operators
    2. Prune equivalent outputs
    """
    expanded = expand_hypotheses(hypotheses, max_hypotheses * 2)
    pruned = prune_equivalent(expanded, max_hypotheses)
    return pruned

In [145]:
# CELL S04.1
# SOLVER CORE — SELECTION POLICY
# Chooses up to two candidate grids from hypotheses.
# Deterministic and intent-blind.

def select_candidates(
    hypotheses: List[SolverHypothesis],
    max_candidates: int = 2
) -> List[Grid]:
    """
    Deterministically select up to `max_candidates` output grids.
    Current policy:
      - preserve hypothesis order
      - return unique grids only
    """
    seen = set()
    selected: List[Grid] = []

    for hyp in hypotheses:
        sig = tuple(tuple(row) for row in hyp.output_grid)
        if sig in seen:
            continue
        seen.add(sig)
        selected.append(hyp.output_grid)

        if len(selected) >= max_candidates:
            break

    return selected

In [146]:
# CELL S04.2
# SOLVER CORE — MULTI-STEP SEARCH LOOP
# Iteratively expands and prunes hypotheses.

def run_search(
    input_grid: Grid,
    max_steps: int = 3,
    max_hypotheses: int = MAX_HYPOTHESES
) -> List[SolverHypothesis]:
    """
    Run a bounded symbolic search starting from the input grid.
    """
    base = SolverHypothesis(
        input_grid=_deepcopy_grid(input_grid),
        output_grid=_deepcopy_grid(input_grid),
        history=()
    )

    hypotheses: List[SolverHypothesis] = [base]

    for _ in range(max_steps):
        hypotheses = expand_and_prune(
            hypotheses,
            max_hypotheses=max_hypotheses
        )
        if not hypotheses:
            break

    return hypotheses

In [147]:
# CELL S04.3
# SOLVER CORE — ENGINE INTEGRATION
# Wires search + selection into ARCSolver.
# No intent, no confidence, no split access.

def _solver_solve_impl(self, input_grid: Grid) -> SolverResult:
    _validate_grid(input_grid)

    # Run bounded search
    hypotheses = run_search(
        input_grid=input_grid,
        max_steps=3,
        max_hypotheses=MAX_HYPOTHESES
    )

    # Select final candidates
    candidates = select_candidates(hypotheses, max_candidates=2)

    # ARC requires up to two outputs; pad with identity if empty
    if not candidates:
        candidates = [_deepcopy_grid(input_grid)]

    return SolverResult(candidates=candidates[:2])


# Seal integration: override solve method explicitly
ARCSolver.solve = _solver_solve_impl

In [148]:
# CELL S05.1
# SOLVER CORE — COMPONENT-LEVEL OPERATORS
# Operate on individual connected components, not whole grids.

def translate_component(
    grid: Grid,
    component: Dict[str, Any],
    dr: int,
    dc: int,
    fill: int = 0
) -> Grid:
    """
    Translate a single component by (dr, dc), leaving others unchanged.
    """
    h, w = len(grid), len(grid[0])
    new = _deepcopy_grid(grid)

    # Clear original component cells
    for r, c in component["cells"]:
        new[r][c] = fill

    # Place translated cells
    for r, c in component["cells"]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < h and 0 <= nc < w:
            new[nr][nc] = component["color"]

    return new


def recolor_component(
    grid: Grid,
    component: Dict[str, Any],
    new_color: int
) -> Grid:
    """
    Recolor a single component.
    """
    new = _deepcopy_grid(grid)
    for r, c in component["cells"]:
        new[r][c] = new_color
    return new


def extract_component_subgrid(
    grid: Grid,
    component: Dict[str, Any]
) -> Grid:
    """
    Extract tight subgrid around a component.
    """
    return extract_subgrid(grid, component["bbox"])


# Deterministic registry
COMPONENT_OPERATORS = {
    "translate_component": translate_component,
    "recolor_component": recolor_component,
}

In [149]:
# CELL S05.2
# SOLVER CORE — SYMMETRY DETECTION
# Detects exact grid symmetries. No pruning or decisions here.

def is_symmetric_horizontal(grid: Grid) -> bool:
    return grid == flip_horizontal(grid)


def is_symmetric_vertical(grid: Grid) -> bool:
    return grid == flip_vertical(grid)


def is_symmetric_rot180(grid: Grid) -> bool:
    return grid == rotate180(grid)


def detect_symmetries(grid: Grid) -> Dict[str, bool]:
    """
    Detect exact global symmetries of a grid.
    """
    return {
        "horizontal": is_symmetric_horizontal(grid),
        "vertical": is_symmetric_vertical(grid),
        "rot180": is_symmetric_rot180(grid),
    }

In [150]:
# CELL S05.3
# SOLVER CORE — SYMMETRY-AWARE PRUNING
# Prevents generating symmetric-equivalent hypotheses.

def _canonical_by_symmetry(grid: Grid) -> Tuple[Tuple[int, ...], ...]:
    """
    Canonical grid signature under symmetry group.
    """
    variants = [
        grid,
        flip_horizontal(grid),
        flip_vertical(grid),
        rotate180(grid),
    ]
    sigs = [tuple(tuple(row) for row in g) for g in variants]
    return min(sigs)


def prune_equivalent_with_symmetry(
    hypotheses: List[SolverHypothesis],
    max_hypotheses: int = MAX_HYPOTHESES
) -> List[SolverHypothesis]:
    """
    Prune hypotheses equivalent under symmetry.
    """
    seen = set()
    pruned: List[SolverHypothesis] = []

    for hyp in hypotheses:
        sig = _canonical_by_symmetry(hyp.output_grid)
        if sig in seen:
            continue
        seen.add(sig)
        pruned.append(hyp)

        if len(pruned) >= max_hypotheses:
            break

    return pruned

In [151]:
# CELL S06.1
# SOLVER CORE — COMPONENT-AWARE HYPOTHESIS EXPANSION
# Extends hypothesis expansion to include component-level operators.

def expand_hypotheses_component_aware(
    hypotheses: List[SolverHypothesis],
    max_hypotheses: int = MAX_HYPOTHESES
) -> List[SolverHypothesis]:
    """
    Expand hypotheses using:
      - global primitive operators
      - component-level operators
    Deterministic, bounded.
    """
    expanded: List[SolverHypothesis] = []

    for hyp in hypotheses:
        if len(expanded) >= max_hypotheses:
            break

        base_grid = hyp.output_grid

        # ---- Global operators (from S03.1) ----
        for name, op in PRIMITIVE_OPERATORS.items():
            if len(expanded) >= max_hypotheses:
                break

            try:
                new_grid = op(base_grid)
            except Exception:
                continue

            expanded.append(
                SolverHypothesis(
                    input_grid=hyp.input_grid,
                    output_grid=new_grid,
                    history=hyp.history + (name,)
                )
            )

        # ---- Component-level operators (from S05.1) ----
        features = grid_features(base_grid)
        components = features["components"]

        for idx, comp in enumerate(components):
            if len(expanded) >= max_hypotheses:
                break

            # Small deterministic translations
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                if len(expanded) >= max_hypotheses:
                    break

                try:
                    new_grid = translate_component(
                        base_grid, comp, dr, dc, fill=0
                    )
                except Exception:
                    continue

                expanded.append(
                    SolverHypothesis(
                        input_grid=hyp.input_grid,
                        output_grid=new_grid,
                        history=hyp.history + (f"translate_component[{idx},{dr},{dc}]",)
                    )
                )

    return expanded

In [152]:
# CELL S06.2
# SOLVER CORE — SYMMETRY-AWARE PRUNING OVERRIDE
# Replaces equivalence pruning with symmetry-aware pruning.

def expand_and_prune_symmetry_aware(
    hypotheses: List[SolverHypothesis],
    max_hypotheses: int = MAX_HYPOTHESES
) -> List[SolverHypothesis]:
    """
    Full expansion + pruning step using:
      - component-aware expansion
      - symmetry-aware equivalence pruning
    """
    expanded = expand_hypotheses_component_aware(
        hypotheses,
        max_hypotheses=max_hypotheses * 2
    )

    pruned = prune_equivalent_with_symmetry(
        expanded,
        max_hypotheses=max_hypotheses
    )

    return pruned

In [153]:
# CELL S06.3
# SOLVER CORE — CONDITIONAL ACTIVATION LOGIC
# Controls when component-level expansion is enabled.

def should_use_component_ops(grid: Grid) -> bool:
    """
    Decide whether to enable component-level reasoning.
    Current policy:
      - enable if more than one connected component
    """
    features = grid_features(grid)
    return features["num_components"] > 1


def expand_and_prune_conditional(
    hypotheses: List[SolverHypothesis],
    max_hypotheses: int = MAX_HYPOTHESES
) -> List[SolverHypothesis]:
    """
    Conditional expansion:
      - single-component grids → global operators only
      - multi-component grids → component-aware expansion
    """
    if not hypotheses:
        return []

    base_grid = hypotheses[0].output_grid

    if should_use_component_ops(base_grid):
        return expand_and_prune_symmetry_aware(
            hypotheses,
            max_hypotheses=max_hypotheses
        )
    else:
        # Fall back to original global-only logic
        expanded = expand_hypotheses(
            hypotheses,
            max_hypotheses=max_hypotheses * 2
        )
        return prune_equivalent_with_symmetry(
            expanded,
            max_hypotheses=max_hypotheses
        )


# ---- Final override: install conditional engine step ----
expand_and_prune = expand_and_prune_conditional

In [154]:
# CELL S07.1
# SOLVER CORE — COMPONENT-WISE SYMMETRY DETECTION
# Detects symmetry properties of individual components.

def component_symmetries(
    grid: Grid,
    component: Dict[str, Any]
) -> Dict[str, bool]:
    """
    Detect symmetries of a single component by analyzing its subgrid.
    """
    sub = extract_component_subgrid(grid, component)

    return {
        "horizontal": sub == flip_horizontal(sub),
        "vertical": sub == flip_vertical(sub),
        "rot180": sub == rotate180(sub),
    }


def all_component_symmetries(grid: Grid) -> List[Dict[str, bool]]:
    """
    Return symmetry signatures for all components in a grid.
    """
    features = grid_features(grid)
    return [
        component_symmetries(grid, comp)
        for comp in features["components"]
    ]

In [155]:
# CELL S07.2
# SOLVER CORE — OPERATOR SCHEDULING & PRIORITY
# Determines deterministic operator application order.

def schedule_operators(grid: Grid) -> List[Tuple[str, Any]]:
    """
    Determine operator schedule based on grid structure.
    Priority rules (deterministic):
      1. Global symmetry ops if grid is symmetric
      2. Component-level ops if multiple components
      3. Remaining global ops
    """
    schedule: List[Tuple[str, Any]] = []

    syms = detect_symmetries(grid)
    features = grid_features(grid)

    # 1. Symmetry-preserving global ops
    if syms["horizontal"]:
        schedule.append(("flip_horizontal", flip_horizontal))
    if syms["vertical"]:
        schedule.append(("flip_vertical", flip_vertical))
    if syms["rot180"]:
        schedule.append(("rotate180", rotate180))

    # 2. Component-level ops
    if features["num_components"] > 1:
        schedule.append(("component_ops", COMPONENT_OPERATORS))

    # 3. Remaining global operators
    for name, op in PRIMITIVE_OPERATORS.items():
        schedule.append((name, op))

    return schedule

In [156]:
# CELL S07.3
# SOLVER CORE — NON-EMPIRICAL HYPOTHESIS SCORING
# Scores hypotheses using structural simplicity and invariants.

def score_hypothesis(hyp: SolverHypothesis) -> int:
    """
    Lower score = better.
    Scoring criteria (purely structural):
      - fewer operations
      - fewer components
      - smaller bounding boxes
    """
    grid = hyp.output_grid
    features = grid_features(grid)

    score = 0

    # Penalize long operation chains
    score += len(hyp.history) * 10

    # Penalize many components
    score += features["num_components"] * 5

    # Penalize spatial spread
    for comp in features["components"]:
        r0, r1, c0, c1 = comp["bbox"]
        score += (r1 - r0 + 1) + (c1 - c0 + 1)

    return score


def rank_hypotheses(
    hypotheses: List[SolverHypothesis]
) -> List[SolverHypothesis]:
    """
    Deterministically rank hypotheses by structural score.
    """
    return sorted(
        hypotheses,
        key=lambda h: score_hypothesis(h)
    )

In [157]:
# CELL S08.1
# SOLVER EVALUATION — TRACE INSTRUMENTATION
# Observational only. Does not alter solver behavior.

from dataclasses import asdict

@dataclass
class SolverTrace:
    """
    Captures a single solver run trace.
    """
    num_steps: int
    num_hypotheses_final: int
    histories: List[Tuple[str, ...]]
    scores: List[int]
    final_shapes: List[Tuple[int, int]]


def run_solver_with_trace(
    solver: ARCSolver,
    input_grid: Grid
) -> Tuple[SolverResult, SolverTrace]:
    """
    Run solver and capture internal traces.
    """
    _validate_grid(input_grid)

    # Run search directly to expose hypotheses
    hypotheses = run_search(
        input_grid=input_grid,
        max_steps=3,
        max_hypotheses=MAX_HYPOTHESES
    )

    # Rank (but do not select differently)
    ranked = rank_hypotheses(hypotheses)

    # Final candidates (existing policy)
    result = solver.solve(input_grid)

    trace = SolverTrace(
        num_steps=3,
        num_hypotheses_final=len(hypotheses),
        histories=[h.history for h in ranked[:5]],
        scores=[score_hypothesis(h) for h in ranked[:5]],
        final_shapes=[(len(g), len(g[0])) for g in result.candidates],
    )

    return result, trace

In [158]:
# CELL S08.2
# SOLVER EVALUATION — STRESS TEST HARNESS
# Runs solver across tasks and records behavior.

def stress_test_solver(
    solver: ARCSolver,
    tasks: Dict[str, Dict[str, Any]],
    max_tasks: int = 20
) -> Dict[str, SolverTrace]:
    """
    Run solver on a subset of tasks and collect traces.
    """
    traces: Dict[str, SolverTrace] = {}

    for i, (task_id, task) in enumerate(tasks.items()):
        if i >= max_tasks:
            break

        # Use first train example input only (observation, not evaluation)
        input_grid = task["train"][0]["input"]

        try:
            _, trace = run_solver_with_trace(solver, input_grid)
            traces[task_id] = trace
        except Exception as e:
            traces[task_id] = SolverTrace(
                num_steps=0,
                num_hypotheses_final=0,
                histories=[("ERROR", str(e))],
                scores=[],
                final_shapes=[]
            )

    return traces

In [159]:
# CELL S08.3
# SOLVER EVALUATION — FAILURE MODE ANALYSIS
# Summarizes behavioral patterns without proposing fixes.

def summarize_traces(traces: Dict[str, SolverTrace]) -> Dict[str, Any]:
    """
    Aggregate solver behavior statistics.
    """
    summary = {
        "total_tasks": len(traces),
        "avg_final_hypotheses": 0.0,
        "common_failure_modes": {},
    }

    if not traces:
        return summary

    total_h = 0
    failure_modes = {}

    for trace in traces.values():
        total_h += trace.num_hypotheses_final

        if trace.histories and trace.histories[0][0] == "ERROR":
            failure_modes["runtime_error"] = failure_modes.get("runtime_error", 0) + 1
        elif trace.num_hypotheses_final == 0:
            failure_modes["no_hypotheses"] = failure_modes.get("no_hypotheses", 0) + 1
        elif all(len(h) > 5 for h in trace.histories):
            failure_modes["overlong_chains"] = failure_modes.get("overlong_chains", 0) + 1

    summary["avg_final_hypotheses"] = total_h / len(traces)
    summary["common_failure_modes"] = failure_modes

    return summary

In [160]:
# CELL 14
# V17 planning refusal receipt

def v17_planning_refusal_receipt(reason: str) -> Dict[str, Any]:
    return {
        "logic_id": "GOVERNANCE_V17_PLANNING",
        "status": "refused",
        "reason": reason,
        "policy": "V17 expressive planning is disabled in V16.2",
    }

In [161]:
# CELL 09z
# Canonical Detection Transform Stack (Executor Dependency)

"""
This cell defines the canonical, detection-only transform stack
required by execute_attempt_v17.

Invariants:
- Detection-only (no grid mutation)
- Deterministic
- Offline
- Governance-safe
- Robust to optional logic family presence
"""

# -----------------------------------------------------------------------------
# Detection transform registry (ordered, defensive)
# -----------------------------------------------------------------------------

_DETECTION_CLASS_NAMES = [
    "LogicFamilyVIII_TerminationSentinel",  # optional, highest priority
    "LogicFamilyVI_RelationalAnchor",
    "LogicFamilyI_AxisReflection",
    "LogicFamilyVII_Interaction",
]

DETECTION_TRANSFORMS = []

for cls_name in _DETECTION_CLASS_NAMES:
    cls = globals().get(cls_name)
    if cls is None:
        continue  # logic family not present in this notebook version
    try:
        inst = cls()
    except Exception as e:
        raise RuntimeError(
            f"Failed to instantiate detection transform {cls_name}: {e}"
        )
    if not hasattr(inst, "__call__"):
        raise RuntimeError(
            f"Invalid detection transform: {cls_name} is not callable."
        )
    DETECTION_TRANSFORMS.append(inst)

# -----------------------------------------------------------------------------
# Hard safety check
# -----------------------------------------------------------------------------

if not DETECTION_TRANSFORMS:
    raise RuntimeError(
        "DETECTION_TRANSFORMS is empty. "
        "At least one detection logic family must be defined."
    )

print(
    "[INIT] ✅ DETECTION_TRANSFORMS initialized "
    f"({len(DETECTION_TRANSFORMS)} detection logic families): "
    f"{[t.__class__.__name__ for t in DETECTION_TRANSFORMS]}"
)

[INIT] ✅ DETECTION_TRANSFORMS initialized (4 detection logic families): ['LogicFamilyVIII_TerminationSentinel', 'LogicFamilyVI_RelationalAnchor', 'LogicFamilyI_AxisReflection', 'LogicFamilyVII_Interaction']


In [162]:
# CELL 15
# V17 planning probe (no-op, receipt only)

def attempt_v17_planning_probe(
    registry: Dict[str, Any],
    receipts: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """
    Records that V17 planning was considered and rejected.
    Does not affect execution or outputs.
    """

    if not v17_planning_allowed():
        receipts.append(
            v17_planning_refusal_receipt(
                reason="V17 planning layer is sealed (preparation phase)"
            )
        )

    return receipts

In [163]:
# CELL 16
# V17 multi-step planning invariants (SEALED, NON-EXECUTING)

class V17MultiStepInvariants:
    """
    Formal invariants governing V17 multi-step planning.

    These are architectural constraints.
    No enforcement logic is enabled in V17.2.
    """

    # M1 — Fixed Plan Horizon
    FIXED_HORIZON_REQUIRED = True

    # M2 — Step-Local Determinism
    STEP_LOCAL_DETERMINISM = True

    # M3 — Monotonic Capability Consumption
    MONOTONIC_CAPABILITY_CONSUMPTION = True

    # M4 — No-Op Prohibition
    NO_OP_STEPS_FORBIDDEN = True

    # M5 — Prefix Safety
    PREFIX_SAFETY_REQUIRED = True

    # M6 — Termination Supremacy
    TERMINATION_ALWAYS_OVERRIDES = True

    # M7 — Single-Authority Execution
    PLANNER_EXECUTION_FORBIDDEN = True

    # M8 — Post-Execution Accountability
    AGREEMENT_REQUIRED_PER_STEP = True

In [164]:
# CELL 17
# V17 multi-step planning gate (absolute prohibition)

def v17_multi_step_planning_allowed() -> bool:
    """
    Multi-step planning is NOT allowed in current versions.

    This function exists to make the prohibition explicit
    and enforceable by inspection.
    """
    return False

In [165]:
# CELL 18
# V17 multi-step planning refusal receipt

def v17_multi_step_refusal_receipt(reason: str) -> dict:
    return {
        "logic_id": "GOVERNANCE_V17_MULTI_STEP_PLANNING",
        "status": "refused",
        "reason": reason,
        "policy": "V17 multi-step planning invariants (locked)",
    }

In [166]:
# CELL 19
# V17 multi-step planning probe (audit-only, no execution)

def attempt_v17_multi_step_probe(receipts: list) -> list:
    """
    Records that multi-step planning was considered
    and explicitly rejected under invariants.
    """

    if not v17_multi_step_planning_allowed():
        receipts.append(
            v17_multi_step_refusal_receipt(
                reason="Multi-step planning violates V17 invariants at this stage"
            )
        )

    return receipts

In [167]:
# CELL 20
# V17 invariant freeze declaration

class V17InvariantFreeze:
    """
    Freezes all V17 planning invariants.

    Any modification to these values constitutes
    a new architectural version and must not occur
    during evaluation.
    """

    # Planning scope
    SINGLE_STEP_PLANNING_ONLY = True
    MULTI_STEP_PLANNING_ALLOWED = False

    # Capability semantics
    PLANNER_DISAGREEMENT_DECAYS = True
    PLANNER_ABSTENTION_PRESERVES = True

    # Authority boundaries
    PLANNER_EXECUTION_FORBIDDEN = True
    GOVERNANCE_SUPREMACY_REQUIRED = True

    # Termination
    TERMINATION_ALWAYS_OVERRIDES = True

    # Evaluation lock
    FROZEN = True

In [168]:
# CELL 21
# Evaluation mode configuration (V16.2 + V17 frozen)

class EvaluationConfig:
    """
    Controls evaluation behavior.
    No learning, no tuning, no adaptation allowed.
    """

    ENABLE_EVALUATION = True

    # Safety guarantees
    ALLOW_PARAMETER_CHANGES = False
    ALLOW_CAPABILITY_TUNING = False
    ALLOW_PLANNER_EXPANSION = False

    # Receipt verbosity
    EMIT_EVALUATION_SUMMARY = True

In [169]:
# CELL 22
# Evaluation summary receipt

def evaluation_summary_receipt(
    attempt_1: dict,
    attempt_2: dict
) -> dict:
    """
    Produces a deterministic evaluation summary
    comparing both attempts.
    """

    def extract_flags(attempt):
        receipts = attempt.get("receipts", [])
        return {
            "termination": any(
                r.get("logic_id") == "LOGIC_VIII_TERMINATION_SENTINEL"
                and r.get("status") == "executed"
                for r in receipts
            ),
            "planner_used": any(
                r.get("logic_id", "").startswith("GOVERNANCE_V17_PLANNING")
                for r in receipts
            ),
            "capability_decayed": any(
                r.get("logic_id") == "GOVERNANCE_CAPABILITY_DECAY"
                for r in receipts
            ),
        }

    f1 = extract_flags(attempt_1)
    f2 = extract_flags(attempt_2)

    return {
        "logic_id": "GOVERNANCE_EVALUATION_SUMMARY",
        "status": "completed",
        "attempt_1_flags": f1,
        "attempt_2_flags": f2,
        "consistent_behavior": f1 == f2,
        "policy": "V17 invariants frozen; evaluation mode",
    }

In [170]:
# CELL 23
# Evaluation runner (dataset-safe, no tuning)

def run_evaluation(task):
    """
    Runs the frozen V16.2 + V17 system in evaluation mode.
    Produces attempts plus an evaluation summary receipt.
    """

    assert EvaluationConfig.ENABLE_EVALUATION is True
    assert V17InvariantFreeze.FROZEN is True

    result = arc_agi_submission_v16_2(task)

    if EvaluationConfig.EMIT_EVALUATION_SUMMARY:
        summary = evaluation_summary_receipt(
            result["attempt_1"],
            result["attempt_2"]
        )
        result["attempt_1"]["receipts"].append(summary)
        result["attempt_2"]["receipts"].append(summary)

    return result

In [171]:
# CELL 24
# SYSTEM DIAGNOSTIC & GOVERNANCE DEBRIEF (EXHAUSTIVE, READ-ONLY)

def arc_agi_system_diagnostic():
    """
    Produces a complete diagnostic debrief of the ARC-AGI system state.
    This function is READ-ONLY and MUST NOT mutate any state.
    """

    report = {}

    # ------------------------------------------------------------------
    # 1. CORE EXECUTION MODE
    # ------------------------------------------------------------------
    report["core_execution"] = {
        "deterministic": True,
        "offline_only": True,
        "two_attempt_protocol": True,
        "training_during_eval": False,
    }

    # ------------------------------------------------------------------
    # 2. VERSION & PHASE STATUS
    # ------------------------------------------------------------------
    report["versions"] = {
        "V15_kernel": "ACTIVE / FROZEN",
        "V16_controlled_expansion": "ACTIVE / FROZEN",
        "V16_2_empirical_instrumentation": "ACTIVE / FROZEN",
        "V17_single_step_planning": "ACTIVE (RESTRICTED)",
        "V17_multi_step_planning": "LOCKED (INVARIANTS DEFINED)",
    }

    # ------------------------------------------------------------------
    # 3. LOGIC FAMILIES REGISTERED
    # ------------------------------------------------------------------
    report["logic_families"] = {
        "LogicFamily_I_AxisReflection": {
            "detection_only": True,
            "gated_transform": True,
            "axes": ["vertical", "horizontal"],
        },
        "LogicFamily_VI_RelationalAnchoring": {
            "detection_only": True,
        },
        "LogicFamily_VII_Interaction": {
            "detection_only": True,
            "gated_transform": True,
        },
        "LogicFamily_VIII_TerminationSentinel": {
            "detection_only": True,
            "absolute_override": True,
        },
    }

    # ------------------------------------------------------------------
    # 4. GOVERNANCE MECHANISMS
    # ------------------------------------------------------------------
    report["governance"] = {
        "AO3_enforced": True,
        "termination_supremacy": True,
        "explicit_gating_controller": True,
        "conflict_resolution_receipts": True,
        "rollback_receipts": True,
        "per_attempt_summary": True,
        "cross_attempt_consistency": True,
    }

    # ------------------------------------------------------------------
    # 5. CAPABILITY SYSTEM
    # ------------------------------------------------------------------
    report["capability_system"] = {
        "baseline_capability": True,
        "micro_composition_enabled": True,
        "capability_budget_enforced": True,
        "capability_decay_on_no_op": True,
        "capability_decay_on_planner_disagreement": True,
        "planner_abstention_preserves_capability": True,
    }

    # ------------------------------------------------------------------
    # 6. PLANNING STATUS (V17)
    # ------------------------------------------------------------------
    report["planning"] = {
        "planner_enabled": True,
        "single_step_only": True,
        "multi_step_allowed": False,
        "planner_executes_transforms": False,
        "planner_is_advisory": True,
        "planning_execution_agreement_enforced": True,
    }

    # ------------------------------------------------------------------
    # 7. INVARIANT FREEZE STATUS
    # ------------------------------------------------------------------
    report["invariant_freeze"] = {
        "V17_invariants_frozen": True,
        "parameter_tuning_allowed": False,
        "capability_rules_mutable": False,
        "planning_scope_mutable": False,
    }

    # ------------------------------------------------------------------
    # 8. EVALUATION MODE
    # ------------------------------------------------------------------
    report["evaluation_mode"] = {
        "evaluation_active": True,
        "learning_disabled": True,
        "adaptive_behavior_disabled": True,
        "empirical_receipts_enabled": True,
    }

    # ------------------------------------------------------------------
    # 9. SAFETY & FAILURE MODES
    # ------------------------------------------------------------------
    report["safety_guarantees"] = {
        "no_hallucination": True,
        "no_heuristics": True,
        "no_search": True,
        "no_recursion": True,
        "no_hidden_state": True,
        "restart_safe": True,
    }

    # ------------------------------------------------------------------
    # 10. FINAL SYSTEM VERDICT
    # ------------------------------------------------------------------
    report["system_verdict"] = {
        "architecture_state": "STABLE",
        "governance_integrity": "INTACT",
        "planning_discipline": "ENFORCED",
        "ready_for_long_run_evaluation": True,
        "ready_for_multi_step_unlock": False,
    }

    return report


# Execute diagnostic and pretty-print
import pprint
pp = pprint.PrettyPrinter(indent=2, width=120)
pp.pprint(arc_agi_system_diagnostic())


{ 'capability_system': { 'baseline_capability': True,
                         'capability_budget_enforced': True,
                         'capability_decay_on_no_op': True,
                         'capability_decay_on_planner_disagreement': True,
                         'micro_composition_enabled': True,
                         'planner_abstention_preserves_capability': True},
  'core_execution': { 'deterministic': True,
                      'offline_only': True,
                      'training_during_eval': False,
                      'two_attempt_protocol': True},
  'evaluation_mode': { 'adaptive_behavior_disabled': True,
                       'empirical_receipts_enabled': True,
                       'evaluation_active': True,
                       'learning_disabled': True},
  'governance': { 'AO3_enforced': True,
                  'conflict_resolution_receipts': True,
                  'cross_attempt_consistency': True,
                  'explicit_gating_controller': True

In [172]:
# CELL 99.9
# Final Kernel Single-Pass Seal — Submission Mode Lock (Armed, Notebook-Safe)

def _final_single_pass_seal():
    """
    Enforces final single-pass execution semantics AFTER explicit arming.

    Phases:
    - Before arming → inert, no checks
    - First call after arming → apply final seal
    - Subsequent calls → RuntimeError
    """

    # --- Not armed: do nothing ---
    if not globals().get("_FINAL_SINGLE_PASS_SEAL_ARMED", False):
        return

    # --- Detect re-execution AFTER sealing ---
    if globals().get("_FINAL_SINGLE_PASS_SEALED", False):
        raise RuntimeError(
            "FINAL SINGLE-PASS SEAL VIOLATION:\n"
            "This notebook has already been sealed.\n"
            "Re-execution indicates the kernel was not restarted "
            "or cells were run out of order."
        )

    # --- Required symbols ---
    required_symbols = [
        "finalize_attempt",
        "solve_arc_task",
        "arc_agi_submission_solver",
        "CANONICAL_EXECUTOR",
        "_freeze_canonical_executor_guard",
    ]

    missing = [s for s in required_symbols if s not in globals()]
    if missing:
        raise RuntimeError(
            "FINAL SINGLE-PASS SEAL FAILED:\n"
            f"Missing required symbols: {missing}\n"
            "Notebook is not in a submission-ready state."
        )

    # --- Force executor freeze handshake ---
    _freeze_canonical_executor_guard()

    if "_FROZEN_CANONICAL_EXECUTOR_ID" not in globals():
        raise RuntimeError(
            "FINAL SINGLE-PASS SEAL FAILED:\n"
            "CANONICAL_EXECUTOR is not frozen.\n"
            "Bind CANONICAL_EXECUTOR before sealing."
        )

    # --- Apply seal ---
    globals()["_FINAL_SINGLE_PASS_SEALED"] = True

    print("✅ FINAL SINGLE-PASS SEAL APPLIED")
    print("   Notebook is now in SUBMISSION MODE.")
    print("   Executor is frozen. Solver is authoritative.")
    print("   Do NOT re-run cells or modify code.")


def arm_final_single_pass_seal():
    """
    Explicitly arm the final single-pass seal.
    Call this ONCE, after a clean kernel restart,
    when the notebook is ready for submission.
    """
    globals()["_FINAL_SINGLE_PASS_SEAL_ARMED"] = True
    _final_single_pass_seal()


# Execute seal (safe: inert until armed)
_final_single_pass_seal()


In [173]:
# CELL 99
# Intent E-Section Insertion Diagnostic (Notebook-Wide, Read-Only)

from collections import OrderedDict

print("\n" + "=" * 96)
print("[99] INTENT E-SECTION INSERTION DIAGNOSTIC")
print("=" * 96)

# ---------------------------------------------------------------------
# REQUIRED SYMBOLS FOR E-SECTION (MINIMUM CONTRACT)
# ---------------------------------------------------------------------
REQUIRED_SYMBOLS = OrderedDict({
    # Dataset layer
    "ARC": "CELL 01d (ARC dataset loader)",
    "parse_task_pairs": "CELL 01d (ARC dataset loader)",

    # Diagnostic layer
    "diagnose_task_structure": "CELL 03c.routing",
    "COMPONENT_COUNT_THRESHOLD": "CELL 03c.routing",

    # Registry / logic (used by later E-cells)
    "REGISTRY": "CELL 03 (logic registry)",

    # Failure / confidence diagnostics (used by E07+)
    "per_task_diagnostics": "diagnostics block (post-solver)",
    "task_confidence": "confidence block (post-diagnostics)",
})

# ---------------------------------------------------------------------
# OPTIONAL SYMBOLS (NON-BLOCKING)
# ---------------------------------------------------------------------
OPTIONAL_SYMBOLS = {
    "repetition_score": "repetition diagnostics",
    "symmetry_residual": "symmetry diagnostics",
}

# ---------------------------------------------------------------------
# CHECK LIVE KERNEL STATE
# ---------------------------------------------------------------------
missing_required = []
present_required = []
present_optional = []
missing_optional = []

for sym, origin in REQUIRED_SYMBOLS.items():
    if sym in globals():
        present_required.append(sym)
    else:
        missing_required.append((sym, origin))

for sym, origin in OPTIONAL_SYMBOLS.items():
    if sym in globals():
        present_optional.append(sym)
    else:
        missing_optional.append(sym)

# ---------------------------------------------------------------------
# REPORT
# ---------------------------------------------------------------------
print("\n[99] REQUIRED SYMBOL CHECK")
print("-" * 96)

if present_required:
    print("✅ Present:")
    for s in present_required:
        print(f"  - {s}")

if missing_required:
    print("\n❌ Missing (BLOCKING):")
    for s, origin in missing_required:
        print(f"  - {s}  (defined in {origin})")

print("\n[99] OPTIONAL SYMBOL CHECK")
print("-" * 96)

if present_optional:
    print("ℹ️ Present:")
    for s in present_optional:
        print(f"  - {s}")

if missing_optional:
    print("\nℹ️ Missing (non-blocking):")
    for s in missing_optional:
        print(f"  - {s}")

# ---------------------------------------------------------------------
# DECISION LOGIC
# ---------------------------------------------------------------------
print("\n[99] E-SECTION INSERTION VERDICT")
print("-" * 96)

if missing_required:
    print("❌ E-SECTION IS NOT SAFE TO EXECUTE YET\n")
    print("Reason:")
    print(" - One or more required symbols are missing from the kernel.")
    print("\nAction required:")
    print(" - Move the entire E01–E14 block DOWNWARD in the notebook.")
    print(" - Ensure all listed origin cells have executed first.")
    print("\nHighest safe insertion point (earliest possible):")
    print(" → Immediately AFTER the cell that defines:")
    print("   - diagnose_task_structure")
    print("   - COMPONENT_COUNT_THRESHOLD")
    print("   (i.e., CELL 03c.routing)")
else:
    print("✅ E-SECTION IS SAFE TO EXECUTE\n")
    print("All required dependencies are present in the kernel.")
    print("\nYou may now safely run:")
    print(" - CELL E01 through CELL E14")
    print("\nRecommended practice:")
    print(" - Restart kernel")
    print(" - Run notebook top → bottom once")

print("\n[99] ✅ Intent insertion diagnostic complete")


[99] INTENT E-SECTION INSERTION DIAGNOSTIC

[99] REQUIRED SYMBOL CHECK
------------------------------------------------------------------------------------------------
✅ Present:
  - ARC
  - parse_task_pairs
  - diagnose_task_structure
  - COMPONENT_COUNT_THRESHOLD
  - REGISTRY
  - per_task_diagnostics
  - task_confidence

[99] OPTIONAL SYMBOL CHECK
------------------------------------------------------------------------------------------------

ℹ️ Missing (non-blocking):
  - repetition_score
  - symmetry_residual

[99] E-SECTION INSERTION VERDICT
------------------------------------------------------------------------------------------------
✅ E-SECTION IS SAFE TO EXECUTE

All required dependencies are present in the kernel.

You may now safely run:
 - CELL E01 through CELL E14

Recommended practice:
 - Restart kernel
 - Run notebook top → bottom once

[99] ✅ Intent insertion diagnostic complete


In [174]:
# CELL E88
# Full Executor Notebook Diagnostic (Dataset‑2 Readiness, Read‑Only)

"""
Comprehensive diagnostic for executor → analysis interop.

This cell performs NO execution.
It inspects the notebook state and prints EVERYTHING required
to safely run E89 → E90.

GOAL
----
Answer the question:
"Is this notebook ready to export executor_outputs.json?"

This cell is:
- Deterministic
- Offline
- Read-only
- Seal-safe
"""

import types
from collections import defaultdict

print("\n" + "=" * 96)
print("[E88] FULL EXECUTOR NOTEBOOK DIAGNOSTIC — DATASET‑2 READINESS")
print("=" * 96)

# -----------------------------------------------------------------------------
# SECTION 1 — GLOBAL EXECUTION STATE
# -----------------------------------------------------------------------------
print("\n[E88.1] GLOBAL EXECUTION STATE")
print("-" * 72)

seal_flags = [
    "_FINAL_SINGLE_PASS_SEALED",
    "_FINAL_SINGLE_PASS_SEAL_ARMED",
    "_CANONICAL_EXECUTOR_FREEZE_ARMED",
    "_FROZEN_CANONICAL_EXECUTOR_ID",
]

for flag in seal_flags:
    print(f"{flag:40s}: {globals().get(flag, False)}")

print(f"{'Notebook sealed (any)':40s}: {any(globals().get(f, False) for f in seal_flags)}")

# -----------------------------------------------------------------------------
# SECTION 2 — ARC DATASET PRESENCE
# -----------------------------------------------------------------------------
print("\n[E88.2] ARC DATASET PRESENCE")
print("-" * 72)

arc_present = "ARC" in globals()
print(f"{'ARC object present':40s}: {arc_present}")

if arc_present:
    for split in ["train_challenges", "eval_challenges", "test_challenges"]:
        val = getattr(ARC, split, None)
        count = len(val) if isinstance(val, dict) else 0
        print(f"  {split:36s}: {count} tasks")

# -----------------------------------------------------------------------------
# SECTION 3 — SUBMISSION / EXECUTOR ENTRY POINTS
# -----------------------------------------------------------------------------
print("\n[E88.3] SUBMISSION ENTRY POINTS (DETECTED)")
print("-" * 72)

submission_fns = [
    "arc_agi_submission_v16_2",
    "arc_agi_submission_v16",
    "arc_agi_submission_v15",
    "arc_agi_submission",
]

found_any = False
for fn in submission_fns:
    present = fn in globals() and callable(globals()[fn])
    print(f"{fn:40s}: {present}")
    found_any = found_any or present

if not found_any:
    print("⚠️  No submission entry points detected (this is OK if outputs are cached)")

# -----------------------------------------------------------------------------
# SECTION 4 — FINAL EXECUTOR OUTPUT BUFFER
# -----------------------------------------------------------------------------
print("\n[E88.4] FINAL_EXECUTOR_OUTPUTS BUFFER STATUS")
print("-" * 72)

if "FINAL_EXECUTOR_OUTPUTS" not in globals():
    print("❌ FINAL_EXECUTOR_OUTPUTS: NOT DEFINED")
    buffer_ready = False
else:
    buf = globals()["FINAL_EXECUTOR_OUTPUTS"]
    is_dict = isinstance(buf, dict)
    size = len(buf) if is_dict else "N/A"
    print(f"{'FINAL_EXECUTOR_OUTPUTS defined':40s}: True")
    print(f"{'Type is dict':40s}: {is_dict}")
    print(f"{'Entries present':40s}: {size}")

    # Sample entries
    if is_dict and size > 0:
        sample_keys = list(buf.keys())[:5]
        print("  Sample task_ids:")
        for k in sample_keys:
            grid = buf.get(k)
            shape = (len(grid), len(grid[0])) if isinstance(grid, list) and grid else "?"
            print(f"   • {k} → grid shape {shape}")

    buffer_ready = is_dict and size > 0

# -----------------------------------------------------------------------------
# SECTION 5 — GRID SHAPE SANITY CHECK
# -----------------------------------------------------------------------------
print("\n[E88.5] GRID SANITY CHECK (FIRST 3 OUTPUTS)")
print("-" * 72)

if "FINAL_EXECUTOR_OUTPUTS" in globals() and isinstance(globals()["FINAL_EXECUTOR_OUTPUTS"], dict):
    checked = 0
    for task_id, grid in globals()["FINAL_EXECUTOR_OUTPUTS"].items():
        if checked >= 3:
            break
        ok = (
            isinstance(grid, list)
            and len(grid) > 0
            and all(isinstance(r, list) for r in grid)
        )
        shape = (len(grid), len(grid[0])) if ok else "INVALID"
        print(f"{task_id:20s}: valid={ok}, shape={shape}")
        checked += 1
else:
    print("SKIPPED — no executor outputs available")

# -----------------------------------------------------------------------------
# SECTION 6 — DERIVABLE OBJECT INVENTORY
# -----------------------------------------------------------------------------
print("\n[E88.6] GLOBAL OBJECT INVENTORY (SUMMARY)")
print("-" * 72)

roles = defaultdict(int)

for name, val in globals().items():
    if name.startswith("_"):
        continue
    if isinstance(val, types.FunctionType):
        roles["functions"] += 1
    elif isinstance(val, type):
        roles["classes"] += 1
    elif isinstance(val, dict):
        roles["dicts"] += 1
    elif isinstance(val, list):
        roles["lists"] += 1

for role, count in sorted(roles.items()):
    print(f"{role:15s}: {count}")

# -----------------------------------------------------------------------------
# SECTION 7 — DATASET‑2 READINESS VERDICT
# -----------------------------------------------------------------------------
print("\n[E88.7] DATASET‑2 READINESS VERDICT")
print("-" * 72)

if not arc_present:
    verdict = "❌ BLOCKED — ARC dataset not loaded"
elif "FINAL_EXECUTOR_OUTPUTS" not in globals():
    verdict = "❌ BLOCKED — FINAL_EXECUTOR_OUTPUTS missing (run E89)"
elif not buffer_ready:
    verdict = "⚠️  INCOMPLETE — buffer exists but is empty"
else:
    verdict = "✅ READY — safe to run E90"

print(f"VERDICT: {verdict}")

print("\n[E88] ✅ FULL DIAGNOSTIC COMPLETE")



[E88] FULL EXECUTOR NOTEBOOK DIAGNOSTIC — DATASET‑2 READINESS

[E88.1] GLOBAL EXECUTION STATE
------------------------------------------------------------------------
_FINAL_SINGLE_PASS_SEALED               : False
_FINAL_SINGLE_PASS_SEAL_ARMED           : False
_CANONICAL_EXECUTOR_FREEZE_ARMED        : False
_FROZEN_CANONICAL_EXECUTOR_ID           : False
Notebook sealed (any)                   : False

[E88.2] ARC DATASET PRESENCE
------------------------------------------------------------------------
ARC object present                      : True
  train_challenges                    : 1000 tasks
  eval_challenges                     : 120 tasks
  test_challenges                     : 240 tasks

[E88.3] SUBMISSION ENTRY POINTS (DETECTED)
------------------------------------------------------------------------
arc_agi_submission_v16_2                : True
arc_agi_submission_v16                  : True
arc_agi_submission_v15                  : True
arc_agi_submission               

In [175]:
# CELL E99
# FINAL EXECUTOR NOTEBOOK DIAGNOSTIC & FREEZE REPORT (READ‑ONLY)

"""
Authoritative end‑of‑notebook diagnostic for ARC‑AGI‑2 executor notebooks.

PURPOSE
-------
This cell prints a complete, human‑readable report answering:

1. Is the executor infrastructure complete?
2. Are all required contracts satisfied?
3. Is Dataset‑2 export possible and correct?
4. What exact state is this notebook in?

This cell performs:
- NO execution
- NO mutation
- NO solver calls
- NO submission calls

It is safe to run as the FINAL cell.
"""

import types
from collections import defaultdict

print("\n" + "=" * 104)
print("[E99] FINAL EXECUTOR NOTEBOOK DIAGNOSTIC & FREEZE REPORT")
print("=" * 104)

# =============================================================================
# SECTION 1 — EXECUTION / SEAL STATE
# =============================================================================
print("\n[E99.1] EXECUTION & SEAL STATE")
print("-" * 80)

seal_flags = [
    "_FINAL_SINGLE_PASS_SEALED",
    "_FINAL_SINGLE_PASS_SEAL_ARMED",
    "_CANONICAL_EXECUTOR_FREEZE_ARMED",
    "_FROZEN_CANONICAL_EXECUTOR_ID",
]

any_sealed = False
for flag in seal_flags:
    val = globals().get(flag, False)
    any_sealed = any_sealed or bool(val)
    print(f"{flag:45s}: {val}")

print(f"{'Any seal active':45s}: {any_sealed}")

# =============================================================================
# SECTION 2 — ARC DATASET VISIBILITY
# =============================================================================
print("\n[E99.2] ARC DATASET VISIBILITY")
print("-" * 80)

arc_present = "ARC" in globals()
print(f"{'ARC object present':45s}: {arc_present}")

if arc_present:
    for split in ["train_challenges", "eval_challenges", "test_challenges"]:
        data = getattr(ARC, split, None)
        count = len(data) if isinstance(data, dict) else 0
        print(f"  {split:41s}: {count} tasks")

# =============================================================================
# SECTION 3 — SUBMISSION ENTRY POINTS
# =============================================================================
print("\n[E99.3] SUBMISSION ENTRY POINTS (DETECTED)")
print("-" * 80)

submission_fns = [
    "arc_agi_submission_v16_2",
    "arc_agi_submission_v16",
    "arc_agi_submission_v15",
    "arc_agi_submission",
]

found_submission = False
for fn in submission_fns:
    present = fn in globals() and callable(globals()[fn])
    print(f"{fn:45s}: {present}")
    found_submission = found_submission or present

# =============================================================================
# SECTION 4 — CORE EXECUTOR INFRASTRUCTURE
# =============================================================================
print("\n[E99.4] CORE EXECUTOR INFRASTRUCTURE")
print("-" * 80)

infra_symbols = [
    "LogicFamilyVIII_TerminationSentinel",
    "capability_decay_receipt",
]

for sym in infra_symbols:
    present = sym in globals()
    print(f"{sym:45s}: {present}")

# =============================================================================
# SECTION 5 — FINAL_EXECUTOR_OUTPUTS BUFFER
# =============================================================================
print("\n[E99.5] FINAL_EXECUTOR_OUTPUTS BUFFER STATUS")
print("-" * 80)

buffer_ok = False

if "FINAL_EXECUTOR_OUTPUTS" not in globals():
    print("❌ FINAL_EXECUTOR_OUTPUTS: NOT DEFINED")
else:
    buf = globals()["FINAL_EXECUTOR_OUTPUTS"]
    is_dict = isinstance(buf, dict)
    size = len(buf) if is_dict else "N/A"

    print(f"{'Defined':45s}: True")
    print(f"{'Type is dict':45s}: {is_dict}")
    print(f"{'Entries present':45s}: {size}")

    if is_dict and size > 0:
        buffer_ok = True
        sample_keys = list(buf.keys())[:5]
        print("  Sample outputs:")
        for k in sample_keys:
            grid = buf.get(k)
            shape = (
                (len(grid), len(grid[0]))
                if isinstance(grid, list) and grid and isinstance(grid[0], list)
                else "INVALID"
            )
            print(f"   • {k:20s} → grid shape {shape}")

# =============================================================================
# SECTION 6 — GRID SANITY CHECK
# =============================================================================
print("\n[E99.6] GRID SANITY CHECK (FIRST 5 OUTPUTS)")
print("-" * 80)

if buffer_ok:
    checked = 0
    invalid = 0
    for task_id, grid in globals()["FINAL_EXECUTOR_OUTPUTS"].items():
        if checked >= 5:
            break
        valid = (
            isinstance(grid, list)
            and len(grid) > 0
            and all(isinstance(r, list) for r in grid)
        )
        if not valid:
            invalid += 1
        shape = (len(grid), len(grid[0])) if valid else "INVALID"
        print(f"{task_id:20s}: valid={valid}, shape={shape}")
        checked += 1

    print(f"{'Invalid grids detected (sample)':45s}: {invalid}")
else:
    print("SKIPPED — buffer empty or missing")

# =============================================================================
# SECTION 7 — DERIVABLE OBJECT INVENTORY (SUMMARY)
# =============================================================================
print("\n[E99.7] GLOBAL OBJECT INVENTORY (SUMMARY)")
print("-" * 80)

roles = defaultdict(int)

for name, val in globals().items():
    if name.startswith("_"):
        continue
    if isinstance(val, types.FunctionType):
        roles["functions"] += 1
    elif isinstance(val, type):
        roles["classes"] += 1
    elif isinstance(val, dict):
        roles["dicts"] += 1
    elif isinstance(val, list):
        roles["lists"] += 1

for role, count in sorted(roles.items()):
    print(f"{role:15s}: {count}")

# =============================================================================
# SECTION 8 — DATASET‑2 EXPORT READINESS VERDICT
# =============================================================================
print("\n[E99.8] DATASET‑2 EXPORT READINESS VERDICT")
print("-" * 80)

if not arc_present:
    verdict = "❌ BLOCKED — ARC dataset not loaded"
elif not found_submission:
    verdict = "⚠️  WARNING — no submission entry point detected"
elif not buffer_ok:
    verdict = "❌ BLOCKED — FINAL_EXECUTOR_OUTPUTS empty or invalid"
else:
    verdict = "✅ READY — executor_outputs.json can be exported safely"

print(f"VERDICT: {verdict}")

# =============================================================================
# SECTION 9 — FREEZE RECOMMENDATION
# =============================================================================
print("\n[E99.9] FREEZE RECOMMENDATION")
print("-" * 80)

if verdict.startswith("✅"):
    print(
        "✔ This notebook is in a consistent, export‑ready state.\n"
        "✔ It is safe to freeze, archive, or version.\n"
        "✔ Dataset‑2 export (CELL E90) is authorized."
    )
else:
    print(
        "✖ Notebook is NOT in a final export‑ready state.\n"
        "✖ Resolve blocking issues above before freezing."
    )

print("\n[E99] ✅ FINAL DIAGNOSTIC COMPLETE")


[E99] FINAL EXECUTOR NOTEBOOK DIAGNOSTIC & FREEZE REPORT

[E99.1] EXECUTION & SEAL STATE
--------------------------------------------------------------------------------
_FINAL_SINGLE_PASS_SEALED                    : False
_FINAL_SINGLE_PASS_SEAL_ARMED                : False
_CANONICAL_EXECUTOR_FREEZE_ARMED             : False
_FROZEN_CANONICAL_EXECUTOR_ID                : False
Any seal active                              : False

[E99.2] ARC DATASET VISIBILITY
--------------------------------------------------------------------------------
ARC object present                           : True
  train_challenges                         : 1000 tasks
  eval_challenges                          : 120 tasks
  test_challenges                          : 240 tasks

[E99.3] SUBMISSION ENTRY POINTS (DETECTED)
--------------------------------------------------------------------------------
arc_agi_submission_v16_2                     : True
arc_agi_submission_v16                       : True
arc_